Environment Check.

In [ ]:
# Quick SageMaker preflight — checks packages, AWS identity, S3 access, base path, HF streaming
import os
import importlib

print("=== OpenFake SageMaker Preflight Check ===\n")

# 1) Package checks
required = ["boto3", "tqdm", "datasets"]
missing = []

for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"[OK] Package available: {pkg}")
    except Exception as e:
        print(f"[MISSING] {pkg} -> {e}")
        missing.append(pkg)

# 2) Base path check
base_dir = os.environ.get("OPENFAKE_BASE_DIR", "/home/ec2-user/SageMaker")
print(f"\n[INFO] OPENFAKE_BASE_DIR = {base_dir}")
print(f"[INFO] Base path exists: {os.path.exists(base_dir)}")

# 3) AWS identity + S3 bucket check
aws_ok = True
bucket_name = "deepfake-d-100k-dataset-tw26"

try:
    import boto3

    sts = boto3.client("sts")
    ident = sts.get_caller_identity()
    print(f"\n[OK] AWS identity detected")
    print(f"     Account: {ident.get('Account')}")
    print(f"     ARN: {ident.get('Arn')}")

    s3 = boto3.client("s3")
    s3.head_bucket(Bucket=bucket_name)
    print(f"[OK] S3 bucket accessible: {bucket_name}")

except Exception as e:
    aws_ok = False
    print(f"\n[FAIL] AWS/S3 check failed: {e}")

# 4) Hugging Face streaming probe
hf_ok = True
try:
    from datasets import load_dataset

    ds = load_dataset("ComplexDataLab/OpenFake", split="train", streaming=True)
    first = next(iter(ds))
    print(f"\n[OK] Hugging Face streaming works")
    print(f"     Sample keys: {list(first.keys())}")

except Exception as e:
    hf_ok = False
    print(f"\n[FAIL] Hugging Face streaming failed: {e}")

# 5) Summary
print("\n=== Summary ===")
if missing:
    print(f"[ACTION] Install missing packages: {missing}")
else:
    print("[OK] No missing core packages")

if not os.path.exists(base_dir):
    print("[ACTION] Base directory does not exist. Set OPENFAKE_BASE_DIR to a valid writable path.")

if not aws_ok:
    print("[ACTION] Fix AWS credentials / IAM role / S3 bucket permissions before running downloader.")

if not hf_ok:
    print("[ACTION] Fix Hugging Face connectivity or package issues before running downloader.")

if (not missing) and os.path.exists(base_dir) and aws_ok and hf_ok:
    print("\nGREEN: Safe to run the downloader.")
else:
    print("\nNOT READY: Resolve the issues above first.")

Datasets Installer.

In [ ]:
%pip install datasets

Download Script for OpenFake - 100K RAW.

HF TOKEN Setup.

In [ ]:
# OpenFake AWS Production Downloader.

# Architecture:
#   - Single Hugging Face streaming session downloads both classes in one pass
#   - Real and fake saved separately to local SSD
#   - After download: real folder zipped → uploaded → verified → deleted
#                     fake folder zipped → uploaded → verified → deleted
#   - Separate manifests per class uploaded to S3
#   - base_dir deleted only after both uploads are fully verified

import os
os.environ['HF_DATASETS_DISABLE_PROGRESS_BARS'] = '1'

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is not set in this kernel session.")

from io import BytesIO
from PIL import Image as PILImage, UnidentifiedImageError
import shutil
import random
import json
import boto3
from datetime import datetime, timezone
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(42)

# CONFIGURATION  —  change these as needed, everything else is derived from them.

TARGET_PER_CLASS   = 50000          # 50K real + 50K fake = 100K raw total
MAX_ITERATIONS     = 600000         # safety ceiling for the HF stream loop
CHECKPOINT_EVERY   = 1000           # write progress checkpoint every N images saved
DISK_MARGIN_FACTOR = 2.2            # require 2.2× the folder size free before zipping

BASE_DIR  = os.environ.get('OPENFAKE_BASE_DIR', '/home/ec2-user/SageMaker')

S3_BUCKET = 'deepfake-d-100k-dataset-tw26'
S3_PREFIX = 'datasets/OpenFake'

# DERIVED PATHS  —  do not hardcode elsewhere

TEMP_RAW_DIR   = os.path.join(BASE_DIR, 'temp_raw')
REAL_DIR       = os.path.join(TEMP_RAW_DIR, 'real')
FAKE_DIR       = os.path.join(TEMP_RAW_DIR, 'fake')
CHECKPOINT_FILE = os.path.join(BASE_DIR, 'openfake_checkpoint.json')

S3_KEYS = {
    'real_zip'       : f'{S3_PREFIX}/openfake_real_raw.zip',
    'fake_zip'       : f'{S3_PREFIX}/openfake_fake_raw.zip',
    'real_manifest'  : f'{S3_PREFIX}/openfake_real_manifest.txt',
    'fake_manifest'  : f'{S3_PREFIX}/openfake_fake_manifest.txt',
}

LOCAL_ZIPS = {
    'real' : os.path.join(BASE_DIR, 'openfake_real_raw'),   # .zip appended by make_archive
    'fake' : os.path.join(BASE_DIR, 'openfake_fake_raw'),
}

MANIFEST_PATHS = {
    'real' : os.path.join(BASE_DIR, 'openfake_real_manifest.txt'),
    'fake' : os.path.join(BASE_DIR, 'openfake_fake_manifest.txt'),
}

# S3 CLIENT

s3 = boto3.client('s3')

# HELPER — CHECKPOINT

def write_checkpoint(iteration, real_count, fake_count, save_errors):
    data = {
        'timestamp'   : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
        'iteration'   : iteration,
        'real_count'  : real_count,
        'fake_count'  : fake_count,
        'save_errors' : save_errors,
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(data, f, indent=2)


def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

# HELPER — DISK SPACE CHECK

def get_folder_size(folder):
    total = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            try:
                total += os.path.getsize(fp)
            except OSError:
                pass
    return total


def check_disk_space_for_zip(folder):
    folder_size   = get_folder_size(folder)
    required      = int(folder_size * DISK_MARGIN_FACTOR)
    _, _, free    = shutil.disk_usage(BASE_DIR)

    folder_gb   = folder_size / (1024 ** 3)
    required_gb = required    / (1024 ** 3)
    free_gb     = free        / (1024 ** 3)

    print(f"  Folder size  : {folder_gb:.2f} GB")
    print(f"  Required free: {required_gb:.2f} GB  (folder × {DISK_MARGIN_FACTOR})")
    print(f"  Actual free  : {free_gb:.2f} GB")

    if free < required:
        raise RuntimeError(
            f"Insufficient disk space before zipping.\n"
            f"  Folder  : {folder_gb:.2f} GB\n"
            f"  Required: {required_gb:.2f} GB\n"
            f"  Free    : {free_gb:.2f} GB\n"
            f"Aborting to prevent archive corruption."
        )

# HELPER — ZIP, UPLOAD, VERIFY, CLEANUP

def create_zip(folder, zip_base_path):
    """Zip a single folder. Returns the .zip path."""
    print(f"  Archiving {os.path.basename(folder)}/  →  {os.path.basename(zip_base_path)}.zip")
    shutil.make_archive(zip_base_path, 'zip', os.path.dirname(folder), os.path.basename(folder))
    zip_path = zip_base_path + '.zip'

    if not os.path.exists(zip_path) or os.path.getsize(zip_path) == 0:
        raise RuntimeError(f"Zip creation failed or produced empty archive: {zip_path}")

    size_gb = os.path.getsize(zip_path) / (1024 ** 3)
    print(f"  Archive ready: {size_gb:.2f} GB")
    return zip_path


def upload_to_s3(local_path, s3_key):
    """Upload a file to S3."""
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  →  s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    """Confirm S3 object exists and its size matches local file."""
    local_size = os.path.getsize(local_path)
    try:
        response  = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
        s3_size   = response['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")

    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size} bytes\n"
            f"  S3    : {s3_size} bytes\n"
            f"Upload may be incomplete. Local files preserved."
        )
    print(f"  Verified: S3 object size matches local ({s3_size / (1024**3):.2f} GB)")


def build_manifest(label, real_count, fake_count, save_errors,
                   error_samples, iteration_counter, zip_path, s3_key):
    count     = real_count if label == 'real' else fake_count
    size_gb   = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

    error_block = ''
    if error_samples:
        error_block = '\nSampled error messages (first 10):\n'
        for i, msg in enumerate(error_samples, 1):
            error_block += f'  [{i:02d}] {msg}\n'

    return (
        f"OpenFake Production Download — {label.upper()} Class Manifest\n"
        f"{'=' * 60}\n"
        f"  Timestamp         : {timestamp}\n"
        f"  Class             : {label}\n"
        f"  Target per class  : {TARGET_PER_CLASS}\n"
        f"  Actual count      : {count}\n"
        f"  Save errors       : {save_errors}\n"
        f"  Iterations used   : {iteration_counter}\n"
        f"  Archive size      : {size_gb:.2f} GB\n"
        f"  S3 destination    : s3://{S3_BUCKET}/{s3_key}\n"
        f"{'=' * 60}\n"
        f"{error_block}"
    )


def process_class(label, folder, zip_base, manifest_path,
                  zip_s3_key, manifest_s3_key,
                  real_count, fake_count, save_errors,
                  error_samples, iteration_counter):
    """
    Full post-download pipeline for one class:
    disk check → zip → upload zip → verify → upload manifest → verify → cleanup
    """
    print(f"\n{'─' * 60}")
    print(f"Processing class: {label.upper()}")
    print(f"{'─' * 60}")

    # 1. Disk space check
    print("\n[1/5] Checking disk space...")
    check_disk_space_for_zip(folder)

    # 2. Zip
    print("\n[2/5] Creating archive...")
    zip_path = create_zip(folder, zip_base)

    # 3. Upload zip + verify
    print("\n[3/5] Uploading archive to S3...")
    try:
        upload_to_s3(zip_path, zip_s3_key)
        verify_s3_upload(zip_path, zip_s3_key)
    except Exception as e:
        print(f"\nUpload/verification failed — local files preserved.\n  {e}")
        raise

    # 4. Write + upload manifest
    print("\n[4/5] Writing and uploading manifest...")
    manifest_text = build_manifest(
        label, real_count, fake_count, save_errors,
        error_samples, iteration_counter, zip_path, zip_s3_key
    )
    with open(manifest_path, 'w') as f:
        f.write(manifest_text)
    print(manifest_text)

    try:
        upload_to_s3(manifest_path, manifest_s3_key)
        verify_s3_upload(manifest_path, manifest_s3_key)
    except Exception as e:
        print(f"\nManifest upload failed — local files preserved.\n  {e}")
        raise

    # 5. Cleanup this class only — only reached if both uploads verified
    print("\n[5/5] Cleaning up local artifacts for this class...")
    shutil.rmtree(folder, ignore_errors=True)
    os.remove(zip_path)
    os.remove(manifest_path)
    print(f"  Removed: {folder}")
    print(f"  Removed: {zip_path}")
    print(f"  Removed: {manifest_path}")

# SETUP

print("\n" + "═" * 60)
print("  OpenFake AWS Production Downloader")
print(f"  Target : {TARGET_PER_CLASS:,} real  +  {TARGET_PER_CLASS:,} fake  =  {TARGET_PER_CLASS * 2:,} total raw images")
print(f"  Bucket : s3://{S3_BUCKET}/{S3_PREFIX}/")
print("═" * 60 + "\n")

shutil.rmtree(TEMP_RAW_DIR, ignore_errors=True)
os.makedirs(REAL_DIR, exist_ok=True)
os.makedirs(FAKE_DIR, exist_ok=True)

# DATASET CONNECTION

print("[Setup] Connecting to ComplexDataLab/OpenFake stream...")
try:
    openfake = load_dataset(
        "ComplexDataLab/OpenFake",
        split='train',
        streaming=True,
        token=hf_token
    )
    openfake = openfake.decode(False)   # disable automatic image decoding
    print("[Setup] Connection successful.\n")
except Exception as e:
    raise RuntimeError(f"Dataset loading failed: {e}")

# DOWNLOAD LOOP

iteration_counter = 0
save_errors       = 0
error_samples     = []    # first 10 exception messages for manifest postmortem
real_count        = 0
fake_count        = 0
last_checkpoint   = 0     # tracks images saved since last checkpoint write

pbar_real = tqdm(total=TARGET_PER_CLASS, desc="  Real", unit="img", mininterval=1.0)
pbar_fake = tqdm(total=TARGET_PER_CLASS, desc="  Fake", unit="img", mininterval=1.0)

print("[Download] Starting single-pass stream...\n")

for item in openfake:
    iteration_counter += 1

    if iteration_counter > MAX_ITERATIONS:
        print("\n[Download] Iteration ceiling reached — stream stopped.")
        break

    image = None

    try:
        raw_label = item['label']
        raw_image = item.get('image', None)

        if raw_image is None:
            continue

        # -battle tested from sandbox-
        if isinstance(raw_label, int):
            label = 'real' if raw_label == 0 else 'fake'
        else:
            raw_label_str = str(raw_label).lower().strip()
            if 'real' in raw_label_str:
                label = 'real'
            elif any(x in raw_label_str for x in ['fake', 'sd', 'flux', 'mj', 'midjourney']):
                label = 'fake'
            else:
                continue

        # manual image decode so corrupt bytes can be skipped safely
        if isinstance(raw_image, dict):
            img_bytes = raw_image.get("bytes")
            img_path  = raw_image.get("path")

            if img_bytes is not None:
                image = PILImage.open(BytesIO(img_bytes))
            elif img_path:
                image = PILImage.open(img_path)
            else:
                continue
        else:
            image = raw_image

        image.load()

        if image.mode != 'RGB':
            image = image.convert('RGB')

        if label == 'real' and real_count < TARGET_PER_CLASS:
            filename = f"raw_openfake_real_{real_count:05d}.jpg"
            image.save(os.path.join(REAL_DIR, filename), format='JPEG', quality=95)
            real_count += 1
            pbar_real.update(1)

        elif label == 'fake' and fake_count < TARGET_PER_CLASS:
            filename = f"raw_openfake_fake_{fake_count:05d}.jpg"
            image.save(os.path.join(FAKE_DIR, filename), format='JPEG', quality=95)
            fake_count += 1
            pbar_fake.update(1)

    except (UnidentifiedImageError, OSError, ValueError) as e:
        save_errors += 1
        if len(error_samples) < 10:
            error_samples.append(f"iter={iteration_counter} | {type(e).__name__}: {str(e)[:120]}")
        continue

    except Exception as e:
        save_errors += 1
        if len(error_samples) < 10:
            error_samples.append(f"iter={iteration_counter} | {type(e).__name__}: {str(e)[:120]}")
        continue

    finally:
        try:
            if image is not None:
                image.close()
        except Exception:
            pass

    # -Periodic checkpoint-
    total_saved = real_count + fake_count
    if total_saved - last_checkpoint >= CHECKPOINT_EVERY:
        write_checkpoint(iteration_counter, real_count, fake_count, save_errors)
        last_checkpoint = total_saved

    if real_count == TARGET_PER_CLASS and fake_count == TARGET_PER_CLASS:
        break

pbar_real.close()
pbar_fake.close()

print(f"\n[Download] Complete.")
print(f"  Real      : {real_count:,} / {TARGET_PER_CLASS:,}")
print(f"  Fake      : {fake_count:,} / {TARGET_PER_CLASS:,}")
print(f"  Iterations: {iteration_counter:,}")
print(f"  Errors    : {save_errors}")

# FAIL HARD — do not proceed if targets not met

if real_count < TARGET_PER_CLASS or fake_count < TARGET_PER_CLASS:
    raise RuntimeError(
        f"\nTarget not reached — aborting. No zipping or uploading will occur.\n"
        f"  Real : {real_count:,} / {TARGET_PER_CLASS:,}\n"
        f"  Fake : {fake_count:,} / {TARGET_PER_CLASS:,}"
    )

clear_checkpoint()

# POST-DOWNLOAD PIPELINE — real class first, then fake

for label in ['real', 'fake']:
    process_class(
        label          = label,
        folder         = REAL_DIR         if label == 'real' else FAKE_DIR,
        zip_base       = LOCAL_ZIPS[label],
        manifest_path  = MANIFEST_PATHS[label],
        zip_s3_key     = S3_KEYS[f'{label}_zip'],
        manifest_s3_key= S3_KEYS[f'{label}_manifest'],
        real_count     = real_count,
        fake_count     = fake_count,
        save_errors    = save_errors,
        error_samples  = error_samples,
        iteration_counter = iteration_counter,
    )
clear_checkpoint()

# FINAL CLEANUP — base temp_raw only after both classes fully verified

shutil.rmtree(TEMP_RAW_DIR, ignore_errors=True)
print(f"\n[Cleanup] Removed base staging dir: {TEMP_RAW_DIR}")

print("\n" + "═" * 60)
print("  OpenFake production download — ALL DONE")
print(f"  Real zip : s3://{S3_BUCKET}/{S3_KEYS['real_zip']}")
print(f"  Fake zip : s3://{S3_BUCKET}/{S3_KEYS['fake_zip']}")
print("═" * 60 + "\n")

Cleanup Cell if the Download gets Interrupted.

In [ ]:
import os, shutil

base_dir = os.environ.get("OPENFAKE_BASE_DIR", "/home/ec2-user/SageMaker")
temp_raw_dir = os.path.join(base_dir, "temp_raw")

if os.path.exists(temp_raw_dir):
    shutil.rmtree(temp_raw_dir, ignore_errors=True)
    print(f"Removed: {temp_raw_dir}")
else:
    print("No temp_raw directory found.")

# optional: also remove any leftover local zips/manifests/checkpoints if your script uses them
for name in [
    "openfake_real_raw.zip",
    "openfake_fake_raw.zip",
    "openfake_checkpoint.json",
    "openfake_real_manifest.txt",
    "openfake_fake_manifest.txt",
]:
    path = os.path.join(base_dir, name)
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed: {path}")

RetinaFace Extractor on OpenFake Raw 100K Images. 

Installers.

In [ ]:
%pip install -U ImageHash pybktree retina-face

In [ ]:
%pip install -U tf-keras

Preflight Check.

In [ ]:
import cv2
import numpy as np
import imagehash
import pybktree
import boto3
from retinaface import RetinaFace
from PIL import Image
from tqdm.auto import tqdm

print("All critical imports OK")

Script.

In [ ]:
# RetinaFace Bouncer — AWS SageMaker Production.
# Architecture:
#   - Pull real and fake raw zips separately from S3
#   - Extract to local SSD
#   - Run RetinaFace face extraction + deduplication in one pass
#   - Zip processed faces - upload to S3
#   - Upload CSV log and manifest to S3
#   - Clean up local workspace only after all uploads verified

import os
import cv2
import numpy as np
import shutil
import glob
import csv
import imagehash
import random
import pybktree
import boto3
import warnings
from datetime import datetime, timezone
from retinaface import RetinaFace
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

random.seed(42)
np.random.seed(42)

# CONFIGURATION  —  change these, everything else derives from them

BASE_DIR  = os.environ.get('BOUNCER_BASE_DIR', '/home/ec2-user/SageMaker')

S3_BUCKET = 'deepfake-d-100k-dataset-tw26'

# Input zips (produced by the downloader)
S3_INPUT_KEYS = {
    'real' : 'datasets/OpenFake/openfake_real_raw.zip',
    'fake' : 'datasets/OpenFake/openfake_fake_raw.zip',
}

# Output destinations
S3_OUTPUT_PREFIX = 'datasets/OpenFake/processed'
S3_OUTPUT_KEYS = {
    'faces_zip' : f'{S3_OUTPUT_PREFIX}/openfake_processed_faces.zip',
    'csv_log'   : f'{S3_OUTPUT_PREFIX}/openfake_face_extraction_log.csv',
    'manifest'  : f'{S3_OUTPUT_PREFIX}/openfake_bouncer_manifest.txt',
}

DISK_MARGIN_FACTOR = 2.2    # require 2.2× folder size free before zipping

# Quota system (70 / 15 / 15 split)
# OpenFake = 40% of 100K total = 40K processed faces (20K real + 20K fake)
target_quotas = {
    "train_real" : 14000, "val_real" : 3000, "test_real" : 3000,
    "train_fake" : 14000, "val_fake" : 3000, "test_fake" : 3000,
}

# DERIVED PATHS

TEMP_WORKSPACE  = os.path.join(BASE_DIR, 'temp_workspace')
LOCAL_INPUT     = os.path.join(TEMP_WORKSPACE, 'input_frames')
LOCAL_OUTPUT    = os.path.join(TEMP_WORKSPACE, 'processed_faces')
LOCAL_ZIP_OUT   = os.path.join(TEMP_WORKSPACE, 'openfake_processed_faces')   # .zip appended
CSV_LOG_PATH    = os.path.join(TEMP_WORKSPACE, 'openfake_face_extraction_log.csv')
MANIFEST_PATH   = os.path.join(TEMP_WORKSPACE, 'openfake_bouncer_manifest.txt')

# S3 CLIENT

s3 = boto3.client('s3')

# HELPERS — DISK

def get_folder_size(folder):
    total = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, f))
            except OSError:
                pass
    return total


def check_disk_space_for_zip(folder):
    folder_size = get_folder_size(folder)
    required    = int(folder_size * DISK_MARGIN_FACTOR)
    _, _, free  = shutil.disk_usage(BASE_DIR)

    folder_gb   = folder_size / (1024 ** 3)
    required_gb = required    / (1024 ** 3)
    free_gb     = free        / (1024 ** 3)

    print(f"  Folder size  : {folder_gb:.2f} GB")
    print(f"  Required free: {required_gb:.2f} GB  (folder × {DISK_MARGIN_FACTOR})")
    print(f"  Actual free  : {free_gb:.2f} GB")

    if free < required:
        raise RuntimeError(
            f"Insufficient disk space before zipping.\n"
            f"  Folder  : {folder_gb:.2f} GB\n"
            f"  Required: {required_gb:.2f} GB\n"
            f"  Free    : {free_gb:.2f} GB\n"
            f"Aborting to prevent archive corruption."
        )

# HELPERS — S3

def download_from_s3(s3_key, local_path):
    size_info = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    print(f"  Downloading s3://{S3_BUCKET}/{s3_key}  ({size_info / (1024**3):.2f} GB)...")
    s3.download_file(S3_BUCKET, s3_key, local_path)
    if not os.path.exists(local_path) or os.path.getsize(local_path) == 0:
        raise RuntimeError(f"Download failed or produced empty file: {local_path}")
    print(f"  Downloaded → {local_path}")


def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  →  s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size} bytes\n"
            f"  S3    : {s3_size} bytes\n"
            f"Upload may be incomplete. Local files preserved."
        )
    print(f"  Verified: S3 size matches local ({s3_size / (1024**3):.2f} GB)")

# HELPERS — QUOTA ROUTER (unchanged from sandbox)

accepted_counts = {key: 0 for key in target_quotas}
total_rejected  = 0


def get_target_bucket(category):
    """Dynamically routes accepted faces to fill Train → Val → Test sequentially."""
    for split in ["train", "val", "test"]:
        bucket = f"{split}_{category}"
        if accepted_counts[bucket] < target_quotas[bucket]:
            return bucket, split
    return None, None

# MASTER CROP ENGINE (unchanged from sandbox — battle tested)

def hash_distance(hash1, hash2):
    return hash1 - hash2

global_seen_tree = pybktree.BKTree(hash_distance)


def master_crop_engine(img_path, save_path,
                       padding=25, min_face_size=30,
                       min_face_ratio=0.005, blur_threshold=50,
                       confidence_threshold=0.90):
    try:
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            return False, "Corrupted Image", 0.0, 0.0, "", "[]"

        height, width = img_cv.shape[:2]
        faces = RetinaFace.detect_faces(img_cv)

        if not isinstance(faces, dict) or len(faces) == 0:
            return False, "No Face Detected", 0.0, 0.0, "", "[]"

        largest_area = 0
        best_face    = None
        for face in faces.values():
            box = face.get('facial_area', None)
            if box is None:
                continue
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > largest_area:
                largest_area = area
                best_face    = face

        if best_face is None:
            return False, "No Valid Face Box", 0.0, 0.0, "", "[]"

        box        = best_face['facial_area']
        str_box    = f"[{box[0]}, {box[1]}, {box[2]}, {box[3]}]"
        confidence = best_face['score']

        if confidence < confidence_threshold:
            return False, "Low Confidence", confidence, 0.0, "", str_box

        face_w, face_h = box[2] - box[0], box[3] - box[1]
        if face_w < min_face_size or face_h < min_face_size:
            return False, "Resolution Too Small", confidence, 0.0, "", str_box
        if (face_w * face_h) / (width * height) < min_face_ratio:
            return False, "Face Ratio Too Small", confidence, 0.0, "", str_box

        x_min = max(0, int(box[0]) - padding)
        y_min = max(0, int(box[1]) - padding)
        x_max = min(width,  int(box[2]) + padding)
        y_max = min(height, int(box[3]) + padding)
        cropped_cv = img_cv[y_min:y_max, x_min:x_max]

        gray_crop = cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2GRAY)
        blur_val  = cv2.Laplacian(gray_crop, cv2.CV_64F).var()

        if blur_val < blur_threshold:
            return False, "Motion Blur", confidence, blur_val, "", str_box

        cropped_pil    = Image.fromarray(cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2RGB))
        standardized   = cropped_pil.resize((260, 260), Image.BICUBIC)
        new_hash       = imagehash.phash(standardized)

        # Global cross-split leakage check
        matches = global_seen_tree.find(new_hash, 2)
        if matches:
            return False, "Global Duplicate Face (Leakage Prevented)", confidence, blur_val, str(new_hash), str_box

        global_seen_tree.add(new_hash)
        standardized.save(save_path, format='JPEG', quality=95)
        return True, "Accepted", confidence, blur_val, str(new_hash), str_box

    except Exception as e:
        return False, f"Engine Error: {str(e)}", 0.0, 0.0, "", "[]"

# SETUP

print("\n" + "═" * 60)
print("  RetinaFace Bouncer — AWS SageMaker Production")
print(f"  Dataset  : OpenFake")
print(f"  Target   : {sum(target_quotas.values()):,} processed faces  "
      f"({sum(v for k,v in target_quotas.items() if 'real' in k):,} real  +  "
      f"{sum(v for k,v in target_quotas.items() if 'fake' in k):,} fake)")
print(f"  Bucket   : s3://{S3_BUCKET}")
print("═" * 60 + "\n")

shutil.rmtree(TEMP_WORKSPACE, ignore_errors=True)
os.makedirs(LOCAL_INPUT,  exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

# CSV log header
with open(CSV_LOG_PATH, mode='w', newline='') as f:
    csv.writer(f).writerow([
        "Image_Name", "Assigned_Split", "Category",
        "Status", "Reason", "Confidence",
        "Blur_Score", "pHash", "Bounding_Box"
    ])

# PULL INPUT ZIPS FROM S3

print("[Setup] Pulling raw image zips from S3...")

for label, s3_key in S3_INPUT_KEYS.items():
    local_zip = os.path.join(TEMP_WORKSPACE, f'openfake_{label}_raw.zip')
    download_from_s3(s3_key, local_zip)

    print(f"  Unpacking {label} zip...")
    shutil.unpack_archive(local_zip, LOCAL_INPUT)
    os.remove(local_zip)
    print(f"  {label} unpacked and zip removed.\n")

# Post-unpack structure validation
for required_dir in [os.path.join(LOCAL_INPUT, 'real'), os.path.join(LOCAL_INPUT, 'fake')]:
    if not os.path.isdir(required_dir):
        raise RuntimeError(
            f"Expected directory not found after unpack: {required_dir}\n"
            f"Check that the zip archives contain a top-level 'real/' and 'fake/' folder."
        )
print("[Setup] Input structure validated — real/ and fake/ directories confirmed.\n")

# FACE EXTRACTION LOOP

print("[Bouncer] Firing up RetinaFace Deduplication Engine...")

image_files = glob.glob(os.path.join(LOCAL_INPUT, "**", "*.jpg"), recursive=True)

if not image_files:
    raise ValueError("CRITICAL: No images found after unpacking. Check zip structure.")

print(f"[Bouncer] Found {len(image_files):,} raw images. Shuffling to prevent split bias...\n")
random.shuffle(image_files)

pbar = tqdm(
    total=sum(target_quotas.values()),
    desc="Securing OpenFake Quota",
    unit="face"
)

run_start = datetime.now(timezone.utc)

with open(CSV_LOG_PATH, mode='a', newline='') as log_file:
    csv_writer = csv.writer(log_file)

    for img_path in image_files:
        cat_name = os.path.basename(os.path.dirname(img_path)).lower().strip()

        if cat_name not in ["real", "fake"]:
            continue

        bucket_key, assigned_split = get_target_bucket(cat_name)
        if not bucket_key:
            continue

        out_folder = os.path.join(LOCAL_OUTPUT, assigned_split, cat_name)
        os.makedirs(out_folder, exist_ok=True)

        save_path = os.path.join(out_folder, os.path.basename(img_path))

        # Collision protection — do not silently overwrite an existing output file
        if os.path.exists(save_path):
            total_rejected += 1
            csv_writer.writerow([
                os.path.basename(img_path), assigned_split, cat_name,
                "Rejected", "Output Filename Collision",
                0.0, 0.0, "", "[]"
            ])
            continue

        passed, msg, conf, blur, phash_val, bbox = master_crop_engine(img_path, save_path)

        if passed:
            accepted_counts[bucket_key] += 1
            csv_writer.writerow([
                os.path.basename(img_path), assigned_split, cat_name,
                "Accepted", "None",
                round(conf, 4), round(blur, 2), phash_val, bbox
            ])
            pbar.update(1)
        else:
            total_rejected += 1
            log_split = assigned_split if "Duplicate" not in msg else "Unassigned"
            csv_writer.writerow([
                os.path.basename(img_path), log_split, cat_name,
                "Rejected", msg,
                round(conf, 4), round(blur, 2), phash_val, bbox
            ])

        if all(accepted_counts[k] >= target_quotas[k] for k in target_quotas):
            print("\n[Bouncer] All target quotas achieved. Shutting down.")
            break

pbar.close()

total_secured = sum(accepted_counts.values())
run_end       = datetime.now(timezone.utc)
duration_min  = (run_end - run_start).total_seconds() / 60

print(f"\n[Bouncer] Extraction complete.")
print(f"  Secured : {total_secured:,}")
print(f"  Rejected: {total_rejected:,}")
print(f"  Duration: {duration_min:.1f} minutes\n")

# Quota breakdown
print("  Split breakdown:")
for key, count in accepted_counts.items():
    quota = target_quotas[key]
    print(f"    {key:<15} {count:>6,} / {quota:,}")

# FAIL HARD IF QUOTAS NOT MET

shortfalls = {k: target_quotas[k] - accepted_counts[k]
              for k in target_quotas if accepted_counts[k] < target_quotas[k]}

if shortfalls:
    raise RuntimeError(
        f"\nQuota not reached — aborting. No upload will occur.\n"
        + "\n".join(f"  {k}: {accepted_counts[k]:,} / {target_quotas[k]:,}  (short by {v:,})"
                    for k, v in shortfalls.items())
    )

# ZIP OUTPUT

print("\n[Upload] Checking disk space before archiving...")
check_disk_space_for_zip(LOCAL_OUTPUT)

print("[Upload] Archiving processed faces...")
shutil.make_archive(LOCAL_ZIP_OUT, 'zip', LOCAL_OUTPUT)

zip_path = LOCAL_ZIP_OUT + '.zip'
if not os.path.exists(zip_path) or os.path.getsize(zip_path) == 0:
    raise RuntimeError("Zip creation failed or produced empty archive.")

zip_size_gb = os.path.getsize(zip_path) / (1024 ** 3)
print(f"  Archive ready: {zip_size_gb:.2f} GB")

# MANIFEST

manifest_text = (
    f"RetinaFace Bouncer — OpenFake Production Manifest\n"
    f"{'=' * 60}\n"
    f"  Timestamp         : {run_end.strftime('%Y-%m-%d %H:%M:%S UTC')}\n"
    f"  Duration          : {duration_min:.1f} minutes\n"
    f"  Total secured     : {total_secured:,}\n"
    f"  Total rejected    : {total_rejected:,}\n"
    f"  Pass rate         : {total_secured / max(total_secured + total_rejected, 1) * 100:.1f}%\n"
    f"  Archive size      : {zip_size_gb:.2f} GB\n"
    f"  S3 output         : s3://{S3_BUCKET}/{S3_OUTPUT_KEYS['faces_zip']}\n"
    f"{'=' * 60}\n"
    f"  Split breakdown:\n"
    + "".join(f"    {k:<15} {accepted_counts[k]:>6,} / {target_quotas[k]:,}\n"
              for k in target_quotas)
)

with open(MANIFEST_PATH, 'w') as f:
    f.write(manifest_text)
print("\n" + manifest_text)

# UPLOAD ALL OUTPUTS TO S3

upload_success = False
try:
    print("[Upload] Uploading processed faces zip...")
    upload_to_s3(zip_path, S3_OUTPUT_KEYS['faces_zip'])
    verify_s3_upload(zip_path, S3_OUTPUT_KEYS['faces_zip'])

    print("\n[Upload] Uploading CSV log...")
    upload_to_s3(CSV_LOG_PATH, S3_OUTPUT_KEYS['csv_log'])
    verify_s3_upload(CSV_LOG_PATH, S3_OUTPUT_KEYS['csv_log'])

    print("\n[Upload] Uploading manifest...")
    upload_to_s3(MANIFEST_PATH, S3_OUTPUT_KEYS['manifest'])
    verify_s3_upload(MANIFEST_PATH, S3_OUTPUT_KEYS['manifest'])

    upload_success = True

except Exception as e:
    print(f"\nUpload failed — local files preserved. Do not shut down instance.\n  {e}")
    raise

# CLEANUP — only after all uploads verified

if upload_success:
    shutil.rmtree(TEMP_WORKSPACE, ignore_errors=True)
    print(f"\n[Cleanup] Workspace removed: {TEMP_WORKSPACE}")

print("\n" + "═" * 60)
print("  RetinaFace Bouncer — ALL DONE")
print(f"  Faces  : s3://{S3_BUCKET}/{S3_OUTPUT_KEYS['faces_zip']}")
print(f"  Log    : s3://{S3_BUCKET}/{S3_OUTPUT_KEYS['csv_log']}")
print(f"  Report : s3://{S3_BUCKET}/{S3_OUTPUT_KEYS['manifest']}")
print("═" * 60 + "\n")

Download Script for FF++.

TUM API Setup.

In [ ]:
# FF++ Video Downloader — AWS SageMaker Production

# Architecture:
#   - Pulls TUM download script using URL from environment variable
#   - Downloads real and fake videos per category to local SSD
#   - Zips real and fake separately → uploads to S3 → verifies → cleans up
#   - Separate manifests per class uploaded to S3
#   - base temp dir deleted only after both uploads are fully verified

import os
import subprocess
import shutil
import glob
import boto3
import urllib.request
from datetime import datetime, timezone
from tqdm.auto import tqdm

# CONFIGURATION.

BASE_DIR = os.environ.get('FFPP_BASE_DIR', '/home/ec2-user/SageMaker')

S3_BUCKET = 'deepfake-d-100k-dataset-tw26'
S3_PREFIX = 'datasets/FFPlus'

DISK_MARGIN_FACTOR = 2.2    # require 2.2× folder size free before zipping

# ── Download targets ──────────────────────────────────────────────────────────
download_targets = {
    'original'       : 1000,   # real videos
    'Deepfakes'      : 1000,   # fake subcategory
    'Face2Face'      : 1000,   # fake subcategory
    'FaceSwap'       : 1000,   # fake subcategory
    'NeuralTextures' : 1000,   # fake subcategory
}

DOWNLOAD_SERVER  = 'EU2'
COMPRESSION      = 'c23'

# DERIVED PATHS

TEMP_BASE           = os.path.join(BASE_DIR, 'ffpp_temp')
REAL_VIDEO_DIR      = os.path.join(TEMP_BASE, 'real')
FAKE_VIDEO_DIR      = os.path.join(TEMP_BASE, 'fake')
DOWNLOAD_STAGE_BASE = os.path.join(TEMP_BASE, 'download_stage')   # staging only — never zipped
DOWNLOAD_SCRIPT     = os.path.join(BASE_DIR,  'ffpp_download.py')

LOCAL_ZIPS = {
    'real' : os.path.join(BASE_DIR, 'ffpp_real_videos'),   # .zip appended by make_archive
    'fake' : os.path.join(BASE_DIR, 'ffpp_fake_videos'),
}

MANIFEST_PATHS = {
    'real' : os.path.join(BASE_DIR, 'ffpp_real_manifest.txt'),
    'fake' : os.path.join(BASE_DIR, 'ffpp_fake_manifest.txt'),
}

S3_KEYS = {
    'real_zip'      : f'{S3_PREFIX}/ffpp_real_videos.zip',
    'fake_zip'      : f'{S3_PREFIX}/ffpp_fake_videos.zip',
    'real_manifest' : f'{S3_PREFIX}/ffpp_real_manifest.txt',
    'fake_manifest' : f'{S3_PREFIX}/ffpp_fake_manifest.txt',
}

# S3 CLIENT

s3 = boto3.client('s3')

# HELPERS — DISK

def get_folder_size(folder):
    total = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, f))
            except OSError:
                pass
    return total


def check_disk_space_for_zip(folder):
    folder_size = get_folder_size(folder)
    required    = int(folder_size * DISK_MARGIN_FACTOR)
    _, _, free  = shutil.disk_usage(BASE_DIR)

    folder_gb   = folder_size / (1024 ** 3)
    required_gb = required    / (1024 ** 3)
    free_gb     = free        / (1024 ** 3)

    print(f"  Folder size  : {folder_gb:.2f} GB")
    print(f"  Required free: {required_gb:.2f} GB  (folder × {DISK_MARGIN_FACTOR})")
    print(f"  Actual free  : {free_gb:.2f} GB")

    if free < required:
        raise RuntimeError(
            f"Insufficient disk space before zipping.\n"
            f"  Folder  : {folder_gb:.2f} GB\n"
            f"  Required: {required_gb:.2f} GB\n"
            f"  Free    : {free_gb:.2f} GB\n"
            f"Aborting to prevent archive corruption."
        )

# HELPERS — S3

def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  →  s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size} bytes\n"
            f"  S3    : {s3_size} bytes\n"
            f"Upload may be incomplete. Local files preserved."
        )
    print(f"  Verified: S3 size matches local ({s3_size / (1024**3):.2f} GB)")

# HELPERS — ZIP + UPLOAD PIPELINE

def create_zip(folder, zip_base_path):
    print(f"  Archiving {os.path.basename(folder)}/  →  {os.path.basename(zip_base_path)}.zip")
    shutil.make_archive(zip_base_path, 'zip', os.path.dirname(folder), os.path.basename(folder))
    zip_path = zip_base_path + '.zip'
    if not os.path.exists(zip_path) or os.path.getsize(zip_path) == 0:
        raise RuntimeError(f"Zip creation failed or produced empty archive: {zip_path}")
    size_gb = os.path.getsize(zip_path) / (1024 ** 3)
    print(f"  Archive ready: {size_gb:.2f} GB")
    return zip_path


def build_manifest(label, video_counts, total_videos, zip_path, s3_key):
    size_gb   = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

    if label == 'real':
        breakdown = f"  original  : {video_counts.get('original', 0):>6,} videos\n"
    else:
        breakdown = "".join(
            f"  {cat:<16}: {video_counts.get(cat, 0):>6,} videos\n"
            for cat in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
        )

    return (
        f"FF++ Video Downloader — {label.upper()} Class Manifest\n"
        f"{'=' * 60}\n"
        f"  Timestamp         : {timestamp}\n"
        f"  Class             : {label}\n"
        f"  Total videos      : {total_videos:,}\n"
        f"  Archive size      : {size_gb:.2f} GB\n"
        f"  S3 destination    : s3://{S3_BUCKET}/{s3_key}\n"
        f"{'=' * 60}\n"
        f"  Category breakdown:\n"
        f"{breakdown}"
    )


def process_class_upload(label, folder, zip_base, manifest_path,
                         zip_s3_key, manifest_s3_key,
                         video_counts, total_videos):
    """Disk check → zip → upload → verify → manifest → upload → verify → cleanup."""
    print(f"\n{'─' * 60}")
    print(f"Processing class: {label.upper()}")
    print(f"{'─' * 60}")

    print("\n[1/5] Checking disk space...")
    check_disk_space_for_zip(folder)

    print("\n[2/5] Creating archive...")
    zip_path = create_zip(folder, zip_base)

    print("\n[3/5] Uploading archive to S3...")
    try:
        upload_to_s3(zip_path, zip_s3_key)
        verify_s3_upload(zip_path, zip_s3_key)
    except Exception as e:
        print(f"\nUpload/verification failed — local files preserved.\n  {e}")
        raise

    print("\n[4/5] Writing and uploading manifest...")
    manifest_text = build_manifest(label, video_counts, total_videos, zip_path, zip_s3_key)
    with open(manifest_path, 'w') as f:
        f.write(manifest_text)
    print(manifest_text)

    try:
        upload_to_s3(manifest_path, manifest_s3_key)
        verify_s3_upload(manifest_path, manifest_s3_key)
    except Exception as e:
        print(f"\nManifest upload failed — local files preserved.\n  {e}")
        raise

    print("\n[5/5] Cleaning up local artifacts for this class...")
    shutil.rmtree(folder, ignore_errors=True)
    os.remove(zip_path)
    os.remove(manifest_path)
    print(f"  Removed: {folder}")
    print(f"  Removed: {zip_path}")

# SETUP

print("\n" + "═" * 60)
print("  FF++ Video Downloader — AWS SageMaker Production")
total_videos_planned = sum(download_targets.values())
print(f"  Target   : {total_videos_planned:,} videos  "
      f"(1,000 real  +  4,000 fake across 4 subcategories)")
print(f"  Bucket   : s3://{S3_BUCKET}/{S3_PREFIX}/")
print("═" * 60 + "\n")

# Pull TUM URL from environment
TUM_URL = os.environ.get('TUM_LINK')
if not TUM_URL:
    raise RuntimeError(
        "TUM_LINK environment variable is not set.\n"
        "Run the API setup cell first."
    )

# Workspace setup
shutil.rmtree(TEMP_BASE, ignore_errors=True)
os.makedirs(REAL_VIDEO_DIR,      exist_ok=True)
os.makedirs(FAKE_VIDEO_DIR,      exist_ok=True)
os.makedirs(DOWNLOAD_STAGE_BASE, exist_ok=True)

# Fetch TUM download script
print("[Setup] Fetching TUM download script...")
urllib.request.urlretrieve(TUM_URL, DOWNLOAD_SCRIPT)
if not os.path.exists(DOWNLOAD_SCRIPT) or os.path.getsize(DOWNLOAD_SCRIPT) == 0:
    raise RuntimeError("Failed to fetch TUM download script.")
print(f"  Saved → {DOWNLOAD_SCRIPT}\n")

# DOWNLOAD LOOP

video_counts    = {}   # tracks actual videos secured per category
download_errors = []   # categories that fell short of target

print("[Download] Starting FF++ video download...\n")

for category, num_videos in download_targets.items():

    is_real    = (category == 'original')
    sub_folder = 'real' if is_real else f'fake/{category.lower()}'
    temp_path  = os.path.join(DOWNLOAD_STAGE_BASE, sub_folder)

    shutil.rmtree(temp_path, ignore_errors=True)
    os.makedirs(temp_path, exist_ok=True)

    print(f"[Download] Pulling {num_videos:,} videos — {category}...")
    cmd = (
        f'echo "" | python3 {DOWNLOAD_SCRIPT} {temp_path} '
        f'-d {category} -c {COMPRESSION} -n {num_videos} --server {DOWNLOAD_SERVER}'
    )
    subprocess.run(cmd, shell=True)

    # Recursive search — bypasses TUM's nested folder structure
    local_vids = glob.glob(os.path.join(temp_path, "**", "*.mp4"), recursive=True)
    secured    = len(local_vids)
    video_counts[category] = secured

    print(f"  Secured: {secured:,} / {num_videos:,} videos for {category}")

    if secured < num_videos:
        download_errors.append(
            f"{category}: {secured:,} / {num_videos:,}  (short by {num_videos - secured:,})"
        )

    # Move videos into the correct class folder
    dest_base = REAL_VIDEO_DIR if is_real else os.path.join(FAKE_VIDEO_DIR, category.lower())
    os.makedirs(dest_base, exist_ok=True)

    for vid_path in tqdm(local_vids, desc=f"  Moving {category}", unit="vid"):
        shutil.move(vid_path, os.path.join(dest_base, os.path.basename(vid_path)))

    shutil.rmtree(temp_path, ignore_errors=True)
    print()

# Download summary
total_real_secured = video_counts.get('original', 0)
total_fake_secured = sum(video_counts.get(c, 0)
                         for c in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures'])
total_secured      = total_real_secured + total_fake_secured

print(f"[Download] Complete.")
print(f"  Real  : {total_real_secured:,} videos")
print(f"  Fake  : {total_fake_secured:,} videos")
print(f"  Total : {total_secured:,} videos")

if download_errors:
    print(f"\n  Shortfalls detected:")
    for e in download_errors:
        print(f"    {e}")

# Fail hard if either class is empty
if total_real_secured == 0 or total_fake_secured == 0:
    raise RuntimeError(
        f"Critical failure — one or both classes have zero videos.\n"
        f"  Real : {total_real_secured:,}\n"
        f"  Fake : {total_fake_secured:,}\n"
        f"Aborting — no zip or upload will occur."
    )

# Strict per-category quota enforcement
quota_failures = []
for category, target in download_targets.items():
    secured = video_counts.get(category, 0)
    if secured < target:
        quota_failures.append(
            f"  {category}: secured {secured:,} / {target:,}  (short by {target - secured:,})"
        )

if quota_failures:
    failure_lines = "\n".join(quota_failures)
    raise RuntimeError(
        f"Per-category quota check failed — the following categories are under target:\n"
        f"{failure_lines}\n"
        f"Aborting — no zip or upload will occur."
    )

# Verify final class folders survived the download loop
if not os.path.isdir(REAL_VIDEO_DIR):
    raise RuntimeError(
        f"REAL_VIDEO_DIR missing before zip pipeline — expected: {REAL_VIDEO_DIR}\n"
        f"Aborting — no zip or upload will occur."
    )
if not os.path.isdir(FAKE_VIDEO_DIR):
    raise RuntimeError(
        f"FAKE_VIDEO_DIR missing before zip pipeline — expected: {FAKE_VIDEO_DIR}\n"
        f"Aborting — no zip or upload will occur."
    )

# POST-DOWNLOAD PIPELINE — Real - Fake.

for label in ['real', 'fake']:
    folder      = REAL_VIDEO_DIR if label == 'real' else FAKE_VIDEO_DIR
    total_label = total_real_secured if label == 'real' else total_fake_secured

    process_class_upload(
        label          = label,
        folder         = folder,
        zip_base       = LOCAL_ZIPS[label],
        manifest_path  = MANIFEST_PATHS[label],
        zip_s3_key     = S3_KEYS[f'{label}_zip'],
        manifest_s3_key= S3_KEYS[f'{label}_manifest'],
        video_counts   = video_counts,
        total_videos   = total_label,
    )

# FINAL CLEANUP

shutil.rmtree(TEMP_BASE, ignore_errors=True)
if os.path.exists(DOWNLOAD_SCRIPT):
    os.remove(DOWNLOAD_SCRIPT)
    print(f"\n[Cleanup] TUM download script wiped: {DOWNLOAD_SCRIPT}")

print("\n" + "═" * 60)
print("  FF++ Video Downloader — ALL DONE")
print(f"  Real zip : s3://{S3_BUCKET}/{S3_KEYS['real_zip']}")
print(f"  Fake zip : s3://{S3_BUCKET}/{S3_KEYS['fake_zip']}")
print("═" * 60 + "\n")

3 Cell Modular Frame Extractor.

In [ ]:
import os, shutil

BASE_DIR = '/home/ec2-user/SageMaker/extractor_temp'

if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR, ignore_errors=True)
    print(f"Removed: {BASE_DIR}")
else:
    print("Nothing to clean.")

In [ ]:
# Cell - 1.
# Delivery to Local EBS.
# Downloads ffpp_real_videos.zip and ffpp_fake_videos.zip from S3.
# Unpacks them to local SageMaker staging, and validates the unpacked structure.

import os
import shutil
import zipfile
import boto3
from datetime import datetime, timezone

# Configuration

S3_BUCKET  = 'deepfake-d-100k-dataset-tw26'
S3_PREFIX  = 'datasets/FFPlus'
BASE_DIR   = '/home/ec2-user/SageMaker/extractor_temp'

# Derived paths

RAW_ZIPS_DIR    = os.path.join(BASE_DIR, 'raw_zips')
RAW_VIDEOS_DIR  = os.path.join(BASE_DIR, 'raw_videos')
REAL_VIDEOS_DIR = os.path.join(RAW_VIDEOS_DIR, 'real')
FAKE_VIDEOS_DIR = os.path.join(RAW_VIDEOS_DIR, 'fake')

S3_OBJECTS = {
    'real' : f'{S3_PREFIX}/ffpp_real_videos.zip',
    'fake' : f'{S3_PREFIX}/ffpp_fake_videos.zip',
}

LOCAL_ZIPS = {
    'real' : os.path.join(RAW_ZIPS_DIR, 'ffpp_real_videos.zip'),
    'fake' : os.path.join(RAW_ZIPS_DIR, 'ffpp_fake_videos.zip'),
}

EXPECTED_FAKE_CATEGORIES = ['deepfakes', 'face2face', 'faceswap', 'neuraltextures']

# S3 client

s3 = boto3.client('s3')

# Helpers

import shutil

def make_dirs():
    # Clean old local staging from previous runs
    for d in [RAW_ZIPS_DIR, RAW_VIDEOS_DIR]:
        if os.path.exists(d):
            shutil.rmtree(d, ignore_errors=True)

    # Recreate clean directory structure
    for d in [RAW_ZIPS_DIR, RAW_VIDEOS_DIR, REAL_VIDEOS_DIR, FAKE_VIDEOS_DIR]:
        os.makedirs(d, exist_ok=True)

    print(f"  Clean staging directories ready under: {BASE_DIR}")


def download_zip(label):
    s3_key     = S3_OBJECTS[label]
    local_path = LOCAL_ZIPS[label]
    size_obj   = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    size_gb    = size_obj / (1024 ** 3)
    print(f"  Downloading {os.path.basename(s3_key)}  ({size_gb:.2f} GB)  ...")
    s3.download_file(S3_BUCKET, s3_key, local_path)
    local_size = os.path.getsize(local_path)
    if local_size != size_obj:
        raise RuntimeError(
            f"Download size mismatch for {label}.\n"
            f"  Expected : {size_obj} bytes\n"
            f"  Got      : {local_size} bytes"
        )
    print(f"  {label.upper()} zip saved → {local_path}  ({local_size / (1024**3):.2f} GB)")


def unpack_zip(label, extract_to):
    local_path = LOCAL_ZIPS[label]
    print(f"  Unpacking {os.path.basename(local_path)} → {extract_to} ...")
    with zipfile.ZipFile(local_path, 'r') as zf:
        zf.extractall(extract_to)
    print(f"  {label.upper()} zip unpacked.")


def validate_structure():
    """Confirm real dir and expected fake category subdirs exist after unpack."""
    errors = []

    if not os.path.isdir(REAL_VIDEOS_DIR):
        errors.append(f"Real video directory missing: {REAL_VIDEOS_DIR}")

    if not os.path.isdir(FAKE_VIDEOS_DIR):
        errors.append(f"Fake video directory missing: {FAKE_VIDEOS_DIR}")
    else:
        present = [d.lower() for d in os.listdir(FAKE_VIDEOS_DIR)
                   if os.path.isdir(os.path.join(FAKE_VIDEOS_DIR, d))]
        for cat in EXPECTED_FAKE_CATEGORIES:
            if cat not in present:
                errors.append(
                    f"Expected fake subcategory '{cat}' not found under {FAKE_VIDEOS_DIR}.\n"
                    f"  Found: {present}"
                )

    if errors:
        raise RuntimeError(
            "Unpacked structure validation failed:\n" +
            "\n".join(f"  - {e}" for e in errors)
        )

    print("  Structure validation PASSED.")
    print(f"    Real dir  : {REAL_VIDEOS_DIR}")
    print(f"    Fake dir  : {FAKE_VIDEOS_DIR}")
    for cat in EXPECTED_FAKE_CATEGORIES:
        cat_path = os.path.join(FAKE_VIDEOS_DIR, cat)
        mp4s = sum(
            len([f for f in files if f.endswith('.mp4')])
            for _, _, files in os.walk(cat_path)
        )
        print(f"      {cat:<20}: {mp4s:,} .mp4 files found")

# Main

print("\n" + "═" * 60)
print("  CELL 1 — THE DELIVERY TRUCK")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("═" * 60 + "\n")

print("[1/4] Creating local directories...")
make_dirs()

print("\n[2/4] Downloading from S3...")
for label in ['real', 'fake']:
    download_zip(label)

print("\n[3/4] Unpacking archives...")
# Unpack one level higher to prevent the "fake/fake/" Russian doll trap
unpack_zip('real', RAW_VIDEOS_DIR)
unpack_zip('fake', RAW_VIDEOS_DIR)

print("\n[4/4] Validating unpacked structure...")
validate_structure()

print("\n" + "─" * 60)
print("  CELL 1 COMPLETE")
print(f"  Zips downloaded to   : {RAW_ZIPS_DIR}")
print(f"  Videos extracted to  : {RAW_VIDEOS_DIR}")
print("─" * 60 + "\n")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — THE SORTING HAT
# Scans unpacked videos, parses identities, builds the undirected identity graph,
# computes connected components, allocates components into 4 isolated master
# buckets, applies strict both-ID filtering, prints a full preflight audit,
# and saves extraction_map.json for Cell 3.
# Does NOT extract frames.
# ══════════════════════════════════════════════════════════════════════════════

import os
import re
import json
import glob
import heapq
from collections import defaultdict
from datetime import datetime, timezone

# ── Configuration — must match Cell 1 ────────────────────────────────────────

BASE_DIR        = '/home/ec2-user/SageMaker/extractor_temp'
RAW_VIDEOS_DIR  = os.path.join(BASE_DIR, 'raw_videos')
REAL_VIDEOS_DIR = os.path.join(RAW_VIDEOS_DIR, 'real')
FAKE_VIDEOS_DIR = os.path.join(RAW_VIDEOS_DIR, 'fake')

EXTRACTION_MAP_PATH = os.path.join(BASE_DIR, 'extraction_map.json')

# Manipulation categories and their assigned master bucket
CATEGORY_TO_BUCKET = {
    'deepfakes'      : 'A',
    'face2face'      : 'B',
    'faceswap'       : 'C',
    'neuraltextures' : 'D',
}
BUCKETS = ['A', 'B', 'C', 'D']

# ── Step A helpers — discover video files ────────────────────────────────────

def discover_real_videos():
    if not os.path.isdir(REAL_VIDEOS_DIR):
        raise RuntimeError(f"Real video directory missing: {REAL_VIDEOS_DIR}")
    paths = glob.glob(os.path.join(REAL_VIDEOS_DIR, '**', '*.mp4'), recursive=True)
    if not paths:
        raise RuntimeError(f"No .mp4 files found under {REAL_VIDEOS_DIR}")
    return sorted(paths)


def discover_fake_videos():
    if not os.path.isdir(FAKE_VIDEOS_DIR):
        raise RuntimeError(f"Fake video directory missing: {FAKE_VIDEOS_DIR}")

    category_paths = {}
    for cat in CATEGORY_TO_BUCKET:
        cat_dir = os.path.join(FAKE_VIDEOS_DIR, cat)
        if not os.path.isdir(cat_dir):
            raise RuntimeError(
                f"Expected fake category directory missing: {cat_dir}\n"
                f"  Available: {os.listdir(FAKE_VIDEOS_DIR)}"
            )
        paths = glob.glob(os.path.join(cat_dir, '**', '*.mp4'), recursive=True)
        if not paths:
            raise RuntimeError(f"No .mp4 files found under {cat_dir}")
        category_paths[cat] = sorted(paths)

    return category_paths

# ── Step B helpers — parse identities ────────────────────────────────────────

_REAL_PATTERN = re.compile(r'^(\d+)\.mp4$', re.IGNORECASE)
_FAKE_PATTERN = re.compile(r'^(\d+)_(\d+)\.mp4$', re.IGNORECASE)


def parse_real_id(filepath):
    """Return integer ID from a real filename like 000.mp4, or None on failure."""
    basename = os.path.basename(filepath)
    m = _REAL_PATTERN.match(basename)
    if not m:
        return None
    return int(m.group(1))


def parse_fake_ids(filepath):
    """Return (int, int) tuple of both IDs from a fake filename like 000_003.mp4, or None."""
    basename = os.path.basename(filepath)
    m = _FAKE_PATTERN.match(basename)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))

# ── Step C — build undirected identity graph ─────────────────────────────────

def build_identity_graph(fake_category_paths):
    """
    Nodes                 : all unique integer identity IDs seen in fake filenames.
    raw_pair_observations : count of every successfully parsed fake filename (includes duplicates across categories).
    unique_edges          : deduplicated set of undirected identity pairs — (min, max) normalized.
    Returns: (adjacency dict, raw_pair_observations, unique_edges, parse_failures dict, all_nodes set)
    """
    adjacency             = defaultdict(set)
    raw_pair_observations = 0
    unique_edges          = set()
    parse_failures        = defaultdict(list)

    for cat, paths in fake_category_paths.items():
        for fp in paths:
            ids = parse_fake_ids(fp)
            if ids is None:
                parse_failures[cat].append(fp)
                continue
            id_a, id_b = ids
            adjacency[id_a].add(id_b)
            adjacency[id_b].add(id_a)
            raw_pair_observations += 1
            unique_edges.add(tuple(sorted((id_a, id_b))))

    all_nodes = set(adjacency.keys())

    return dict(adjacency), raw_pair_observations, unique_edges, parse_failures, all_nodes

# ── Step D — compute connected components ────────────────────────────────────

def compute_connected_components(adjacency, all_nodes):
    """
    BFS over adjacency dict.
    Returns list of frozensets, each frozenset = one connected component.
    """
    visited    = set()
    components = []

    for start in sorted(all_nodes):
        if start in visited:
            continue
        component = set()
        queue     = [start]
        while queue:
            node = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            component.add(node)
            for neighbour in adjacency.get(node, []):
                if neighbour not in visited:
                    queue.append(neighbour)
        components.append(frozenset(component))

    return components

# ── Step E — allocate components into 4 master buckets ───────────────────────

def allocate_components_to_buckets(components):
    """
    Greedy balanced allocation: sort components largest-first, assign each to
    the bucket with the fewest identities so far.
    Ties broken by bucket name order (A < B < C < D) for determinism.
    Returns: dict bucket_name -> set of identity IDs
    """
    heap = [(0, b) for b in BUCKETS]
    heapq.heapify(heap)

    bucket_identities = {b: set() for b in BUCKETS}

    sorted_components = sorted(components, key=lambda c: (-len(c), min(c)))

    for comp in sorted_components:
        size, bucket = heapq.heappop(heap)
        bucket_identities[bucket].update(comp)
        heapq.heappush(heap, (size + len(comp), bucket))

    return bucket_identities

# ── Step F — map videos into buckets with strict both-ID filter ───────────────

def map_real_videos_to_buckets(real_paths, bucket_identities):
    """
    Each original video NNN.mp4 goes to the bucket whose identity set contains NNN.
    Returns: dict bucket_name -> list of file paths, plus list of unmatched paths.
    """
    bucket_reals = {b: [] for b in BUCKETS}
    unmatched    = []

    for fp in real_paths:
        vid_id = parse_real_id(fp)
        if vid_id is None:
            unmatched.append(fp)
            continue
        placed = False
        for b in BUCKETS:
            if vid_id in bucket_identities[b]:
                bucket_reals[b].append(fp)
                placed = True
                break
        if not placed:
            unmatched.append(fp)

    return bucket_reals, unmatched


def map_fake_videos_strict(fake_category_paths, bucket_identities):
    """
    For each manipulation category, check BOTH IDs against the category's assigned bucket.
    A video is eligible only if BOTH IDs are in the assigned bucket's identity set.
    Returns:
      eligible : dict category -> list of eligible file paths
      skipped  : dict category -> list of skipped file paths
    """
    eligible = {}
    skipped  = {}

    for cat, bucket in CATEGORY_TO_BUCKET.items():
        id_set       = bucket_identities[bucket]
        cat_eligible = []
        cat_skipped  = []

        for fp in fake_category_paths.get(cat, []):
            ids = parse_fake_ids(fp)
            if ids is None:
                cat_skipped.append(fp)
                continue
            id_a, id_b = ids
            if id_a in id_set and id_b in id_set:
                cat_eligible.append(fp)
            else:
                cat_skipped.append(fp)

        eligible[cat] = cat_eligible
        skipped[cat]  = cat_skipped

    return eligible, skipped

# ── Step G — preflight audit ──────────────────────────────────────────────────

def print_audit(
    real_paths, fake_category_paths,
    all_nodes, raw_pair_observations, unique_edges, components,
    bucket_identities,
    bucket_reals, unmatched_reals,
    eligible_fakes, skipped_fakes,
    parse_failures,
):
    sep  = "═" * 60
    line = "─" * 60

    print(f"\n{sep}")
    print("  CELL 2 — PREFLIGHT AUDIT REPORT")
    print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(sep)

    print(f"\n{line}")
    print("  [1] DISCOVERED VIDEOS")
    print(line)
    print(f"  Original (real) videos : {len(real_paths):>6,}")
    total_fake = sum(len(v) for v in fake_category_paths.values())
    print(f"  Total fake videos      : {total_fake:>6,}")
    for cat in CATEGORY_TO_BUCKET:
        print(f"    {cat:<20}: {len(fake_category_paths.get(cat, [])):>6,}")

    print(f"\n{line}")
    print("  [2] IDENTITY GRAPH")
    print(line)
    print(f"  Nodes (unique IDs)          : {len(all_nodes):>6,}")
    print(f"  Raw fake pair observations  : {raw_pair_observations:>6,}  (parsed fake files across all categories)")
    print(f"  Unique graph edges          : {len(unique_edges):>6,}  (deduplicated undirected identity pairs)")
    print(f"  Connected components        : {len(components):>6,}")

    comp_sizes = sorted([len(c) for c in components], reverse=True)
    if comp_sizes:
        print(f"  Largest component   : {comp_sizes[0]:>6,} identities")
        print(f"  Smallest component  : {comp_sizes[-1]:>6,} identities")
        print(f"  Median component    : {comp_sizes[len(comp_sizes)//2]:>6,} identities")
    else:
        print(f"  Largest component   :      0 identities (no components)")
        print(f"  Smallest component  :      0 identities")
        print(f"  Median component    :      0 identities")

    size_dist = defaultdict(int)
    for s in comp_sizes:
        size_dist[s] += 1
    print(f"  Component size distribution (size: count):")
    for size in sorted(size_dist.keys(), reverse=True)[:10]:
        print(f"    size {size:>4}: {size_dist[size]:>4} components")
    if len(size_dist) > 10:
        print(f"    ... ({len(size_dist) - 10} more size buckets)")

    print(f"\n{line}")
    print("  [3] MASTER BUCKET IDENTITY ALLOCATION")
    print(line)
    for b in BUCKETS:
        print(f"  Bucket {b} : {len(bucket_identities[b]):>6,} unique identities")

    print(f"\n{line}")
    print("  [4] REAL VIDEO MAPPING")
    print(line)
    for b in BUCKETS:
        print(f"  Bucket {b} real videos : {len(bucket_reals[b]):>6,}")
    if unmatched_reals:
        print(f"\n  WARNING: {len(unmatched_reals):,} real videos could not be placed in any bucket.")

    print(f"\n{line}")
    print("  [5] FAKE VIDEO ELIGIBILITY (strict both-ID filter)")
    print(line)
    for cat, bucket in CATEGORY_TO_BUCKET.items():
        n_elig = len(eligible_fakes.get(cat, []))
        n_skip = len(skipped_fakes.get(cat, []))
        n_tot  = len(fake_category_paths.get(cat, []))
        pct    = (n_elig / n_tot * 100) if n_tot else 0
        print(f"  {cat:<20} (Bucket {bucket}) : "
              f"{n_elig:>6,} eligible  /  {n_skip:>6,} skipped  "
              f"({pct:.1f}% pass rate)")

    total_failures = sum(len(v) for v in parse_failures.values())
    if total_failures:
        print(f"\n{line}")
        print("  [6] PARSE FAILURES (filenames that did not match expected pattern)")
        print(line)
        for cat, paths in parse_failures.items():
            if paths:
                print(f"  {cat}: {len(paths)} failures")
                for p in paths[:5]:
                    print(f"    {os.path.basename(p)}")
                if len(paths) > 5:
                    print(f"    ... ({len(paths) - 5} more)")

    LOW_THRESHOLD = 200
    print(f"\n{line}")
    print("  [7] WARNINGS")
    print(line)
    warned = False
    for cat in CATEGORY_TO_BUCKET:
        n = len(eligible_fakes.get(cat, []))
        if n < LOW_THRESHOLD:
            print(f"  WARNING: {cat} has only {n:,} eligible videos — below threshold of {LOW_THRESHOLD:,}.")
            warned = True
    for b in BUCKETS:
        n = len(bucket_reals[b])
        if n < LOW_THRESHOLD:
            print(f"  WARNING: Bucket {b} real videos only {n:,} — below threshold of {LOW_THRESHOLD:,}.")
            warned = True
    if not warned:
        print("  No warnings.")

    print(f"\n{sep}")
    print("  END OF AUDIT REPORT")
    print(sep + "\n")

# ── Step H — fail-hard checks ─────────────────────────────────────────────────

def fail_hard_checks(real_paths, fake_category_paths, all_nodes, eligible_fakes, bucket_reals):
    if not real_paths:
        raise RuntimeError("No original/real videos discovered. Aborting.")

    total_fake = sum(len(v) for v in fake_category_paths.values())
    if not total_fake:
        raise RuntimeError("No fake videos discovered. Aborting.")

    if not all_nodes:
        raise RuntimeError("Identity graph has zero nodes. Aborting.")

    for cat in CATEGORY_TO_BUCKET:
        if not fake_category_paths.get(cat):
            raise RuntimeError(
                f"Manipulation category '{cat}' has zero discovered videos. Aborting."
            )
        if not eligible_fakes.get(cat):
            raise RuntimeError(
                f"Manipulation category '{cat}' has zero eligible videos after "
                f"strict both-ID filtering. Aborting.\n"
                f"Check bucket allocation and identity graph."
            )

    for b in BUCKETS:
        if not bucket_reals.get(b):
            raise RuntimeError(
                f"Bucket {b} has zero eligible real videos after mapping. Aborting.\n"
                f"Check connected component allocation and identity graph."
            )

# ── Step I — save extraction map ─────────────────────────────────────────────

def save_extraction_map(
    bucket_identities,
    bucket_reals,
    eligible_fakes,
    components,
    real_paths,
    fake_category_paths,
    raw_pair_observations,
    unique_edges,
    skipped_fakes,
    unmatched_reals,
    parse_failures,
):
    component_membership = {
        str(i): sorted(list(comp))
        for i, comp in enumerate(
            sorted(components, key=lambda c: (-len(c), min(c)))
        )
    }

    extraction_map = {
        'generated_at'       : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
        'bucket_identities'  : {
            b: sorted(list(ids)) for b, ids in bucket_identities.items()
        },
        'real_video_paths'   : {
            b: sorted(bucket_reals[b]) for b in BUCKETS
        },
        'fake_video_paths'   : {
            cat: sorted(eligible_fakes.get(cat, []))
            for cat in CATEGORY_TO_BUCKET
        },
        'component_membership' : component_membership,
        'audit_stats'        : {
            'total_real_discovered'    : len(real_paths),
            'total_fake_discovered'    : {cat: len(v) for cat, v in fake_category_paths.items()},
            'total_real_eligible'      : {b: len(bucket_reals[b]) for b in BUCKETS},
            'total_fake_eligible'      : {cat: len(eligible_fakes.get(cat, [])) for cat in CATEGORY_TO_BUCKET},
            'total_fake_skipped'       : {cat: len(skipped_fakes.get(cat, [])) for cat in CATEGORY_TO_BUCKET},
            'total_real_unmatched'     : len(unmatched_reals),
            'graph_nodes'              : len(set().union(*[set(c) for c in components])) if components else 0,
            'raw_pair_observations'    : raw_pair_observations,
            'unique_graph_edges'       : len(unique_edges),
            'num_connected_components' : len(components),
            'largest_component_size'   : max(len(c) for c in components) if components else 0,
            'bucket_identity_counts'   : {b: len(ids) for b, ids in bucket_identities.items()},
            'parse_failures'           : {cat: len(v) for cat, v in parse_failures.items()},
        },
    }

    with open(EXTRACTION_MAP_PATH, 'w') as f:
        json.dump(extraction_map, f, indent=2)

    print(f"  Extraction map saved → {EXTRACTION_MAP_PATH}")
    size_kb = os.path.getsize(EXTRACTION_MAP_PATH) / 1024
    print(f"  File size: {size_kb:.1f} KB")

# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 60)
print("  CELL 2 — THE SORTING HAT")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("═" * 60 + "\n")

print("[A] Discovering video files...")
real_paths          = discover_real_videos()
fake_category_paths = discover_fake_videos()
print(f"  Real videos discovered : {len(real_paths):,}")
for cat, paths in fake_category_paths.items():
    print(f"  {cat:<20}: {len(paths):,} fake videos")

print("\n[B/C] Building identity graph from fake pairings...")
adjacency, raw_pair_observations, unique_edges, parse_failures, all_nodes = build_identity_graph(fake_category_paths)
print(f"  Graph nodes                : {len(all_nodes):,}")
print(f"  Raw fake pair observations : {raw_pair_observations:,}")
print(f"  Unique graph edges         : {len(unique_edges):,}")

print("\n[D] Computing connected components...")
components = compute_connected_components(adjacency, all_nodes)
print(f"  Connected components : {len(components):,}")
print(f"  Largest component   : {max((len(c) for c in components), default=0):,} identities")

print("\n[E] Allocating components into 4 master buckets...")
bucket_identities = allocate_components_to_buckets(components)
for b in BUCKETS:
    print(f"  Bucket {b} : {len(bucket_identities[b]):,} identities")

print("\n[F] Mapping videos into buckets with strict both-ID filtering...")
bucket_reals, unmatched_reals = map_real_videos_to_buckets(real_paths, bucket_identities)
eligible_fakes, skipped_fakes = map_fake_videos_strict(fake_category_paths, bucket_identities)
for b in BUCKETS:
    print(f"  Bucket {b} real videos : {len(bucket_reals[b]):,}")
for cat, bucket in CATEGORY_TO_BUCKET.items():
    print(f"  {cat:<20} (Bucket {bucket}) eligible : {len(eligible_fakes.get(cat, [])):,}")

print("\n[G] Running preflight audit...")
print_audit(
    real_paths, fake_category_paths,
    all_nodes, raw_pair_observations, unique_edges, components,
    bucket_identities,
    bucket_reals, unmatched_reals,
    eligible_fakes, skipped_fakes,
    parse_failures,
)

print("[H] Fail-hard validation checks...")
fail_hard_checks(real_paths, fake_category_paths, all_nodes, eligible_fakes, bucket_reals)
print("  All fail-hard checks passed.\n")

print("[I] Saving extraction map for Cell 3...")
save_extraction_map(
    bucket_identities,
    bucket_reals,
    eligible_fakes,
    components,
    real_paths,
    fake_category_paths,
    raw_pair_observations,
    unique_edges,
    skipped_fakes,
    unmatched_reals,
    parse_failures,
)

print("\n" + "─" * 60)
print("  CELL 2 COMPLETE — ready for Cell 3 (frame extraction)")
print(f"  Extraction map : {EXTRACTION_MAP_PATH}")
print("─" * 60 + "\n")

In [ ]:
# CELL 3 — FF++ FRAME EXTRACTION ENGINE
# Final stage of the 3-cell modular pipeline:
#   Cell 1 = Download and unpack raw videos from S3.
#   Cell 2 = Build identity graph, allocate buckets, audit, save extraction_map.json.
#   Cell 3 = Load the map, extract frames, archive, upload to S3, clean up.
#
# Cell 3 is an EXECUTION ENGINE. It does NOT recompute graph logic, does NOT
# reassign buckets, and does NOT rebuild connected components. It trusts
# extraction_map.json produced by Cell 2 as the single source of truth.
#
# What it does:
#   1. Loads extraction_map.json
#   2. Validates that mapped video paths exist locally
#   3. Extracts 20 evenly-spaced raw frames per eligible video (OpenCV)
#   4. Writes frames into a provenance-preserving directory tree
#   5. Produces a human-readable manifest and a per-video CSV log
#   6. Archives the extracted frames into a single .zip
#   7. Uploads the archive + manifest + log to S3
#   8. Verifies every upload via head_object size comparison
#   9. Cleans local extracted frames and archive only after verified success

import os
import sys
import json
import csv
import cv2
import shutil
import zipfile
import boto3
import time
import numpy as np
from datetime import datetime, timezone

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR = "/home/ec2-user/SageMaker/extractor_temp"
RAW_VIDEOS_DIR = os.path.join(BASE_DIR, "raw_videos")
EXTRACTION_MAP_PATH = os.path.join(BASE_DIR, "extraction_map.json")

EXTRACTED_DIR = os.path.join(BASE_DIR, "extracted_frames")
ARCHIVE_PATH = os.path.join(BASE_DIR, "ffpp_extracted_frames.zip")
MANIFEST_PATH = os.path.join(BASE_DIR, "ffpp_extracted_frames_manifest.txt")
LOG_CSV_PATH = os.path.join(BASE_DIR, "ffpp_frame_extraction_log.csv")

S3_BUCKET = "deepfake-d-100k-dataset-tw26"
S3_PREFIX = "datasets/FFPlus/processed"

TARGET_FRAMES_PER_VIDEO = 20  # buffered raw frame count before RetinaFace filtering

# Mapping from extraction_map.json keys -> local output subdirectories
REAL_BUCKET_MAP = {
    "bucket_a": "real/bucket_a",
    "bucket_b": "real/bucket_b",
    "bucket_c": "real/bucket_c",
    "bucket_d": "real/bucket_d",
}

FAKE_MANIP_MAP = {
    "deepfakes": "fake/deepfakes",
    "face2face": "fake/face2face",
    "faceswap": "fake/faceswap",
    "neuraltextures": "fake/neuraltextures",
}

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: COMPUTE EVENLY-SPACED FRAME INDICES
# ─────────────────────────────────────────────────────────────────────────────

def compute_frame_indices(total_frames, target_count):
    """
    Return a list of unique, deterministic, evenly-spaced frame indices
    spread across the full video duration.

    If total_frames <= target_count, returns all available indices.
    """
    if total_frames <= 0:
        return []
    if total_frames <= target_count:
        return list(range(total_frames))
    # Evenly spaced across [0, total_frames - 1]
    indices = np.linspace(0, total_frames - 1, num=target_count, dtype=int)
    # Deduplicate while preserving order (unlikely with linspace but defensive)
    seen = set()
    unique = []
    for idx in indices:
        if idx not in seen:
            seen.add(idx)
            unique.append(int(idx))
    return unique


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: EXTRACT FRAMES FROM A SINGLE VIDEO
# ─────────────────────────────────────────────────────────────────────────────

def extract_frames_from_video(video_path, output_dir, target_count, video_counter):
    """
    Opens a single video, extracts evenly-spaced frames, saves as JPG.

    Returns a dict with extraction stats for the CSV log.
    """
    result = {
        "video_path": video_path,
        "output_dir": output_dir,
        "total_frames_metadata": 0,
        "target_frames": target_count,
        "frames_saved": 0,
        "status": "failed",
        "failure_reason": "",
    }

    video_stem = os.path.splitext(os.path.basename(video_path))[0]

    if not os.path.isfile(video_path):
        result["failure_reason"] = "file_not_found"
        return result

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        result["failure_reason"] = "cv2_cannot_open"
        return result

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    result["total_frames_metadata"] = total_frames

    if total_frames <= 0:
        cap.release()
        result["failure_reason"] = "zero_or_negative_frame_count"
        return result

    indices = compute_frame_indices(total_frames, target_count)

    if not indices:
        cap.release()
        result["failure_reason"] = "no_valid_indices"
        return result

    os.makedirs(output_dir, exist_ok=True)
    saved = 0

    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        # Filename: {video_counter:04d}_{video_stem}_frame_{frame_idx:06d}.jpg
        fname = f"{video_counter:04d}_{video_stem}_frame_{frame_idx:06d}.jpg"
        out_path = os.path.join(output_dir, fname)
        try:
            cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
        except Exception as e:
            # Log but don't crash
            print(f"  [WARN] Failed to write frame {frame_idx} for {video_stem}: {e}")

    cap.release()
    result["frames_saved"] = saved

    if saved == 0:
        result["status"] = "failed"
        result["failure_reason"] = "all_frame_reads_failed"
    elif saved < len(indices):
        result["status"] = "partial"
        result["failure_reason"] = f"only_{saved}_of_{len(indices)}_frames_read"
    else:
        result["status"] = "success"

    return result


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: CREATE ZIP ARCHIVE
# ─────────────────────────────────────────────────────────────────────────────

def create_zip_archive(source_dir, archive_path):
    """
    Create a zip archive of the entire source directory tree.
    Returns the archive size in bytes, or raises on failure.
    """
    print(f"\n{'='*70}")
    print("CREATING ZIP ARCHIVE")
    print(f"{'='*70}")
    print(f"  Source : {source_dir}")
    print(f"  Target : {archive_path}")

    file_count = 0
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(source_dir):
            for f in files:
                abs_path = os.path.join(root, f)
                arc_name = os.path.relpath(abs_path, os.path.dirname(source_dir))
                zf.write(abs_path, arc_name)
                file_count += 1

    archive_size = os.path.getsize(archive_path)
    print(f"  Files archived : {file_count}")
    print(f"  Archive size   : {archive_size / (1024**2):.2f} MB")
    return archive_size


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: S3 UPLOAD WITH VERIFICATION
# ─────────────────────────────────────────────────────────────────────────────

def upload_and_verify(s3_client, local_path, bucket, s3_key):
    """
    Upload a local file to S3, then verify via head_object size comparison.
    Raises RuntimeError if verification fails.
    """
    local_size = os.path.getsize(local_path)
    print(f"  Uploading: {os.path.basename(local_path)} ({local_size / (1024**2):.2f} MB)")
    print(f"    -> s3://{bucket}/{s3_key}")

    s3_client.upload_file(local_path, bucket, s3_key)

    # Verify
    head = s3_client.head_object(Bucket=bucket, Key=s3_key)
    remote_size = head['ContentLength']

    if remote_size != local_size:
        raise RuntimeError(
            f"UPLOAD VERIFICATION FAILED for {s3_key}: "
            f"local={local_size} bytes, remote={remote_size} bytes"
        )
    print(f"    Verified: {remote_size} bytes match.")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN EXTRACTION PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

start_time = time.time()
timestamp_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print(f"{'='*70}")
print("CELL 3 — FF++ FRAME EXTRACTION ENGINE")
print(f"{'='*70}")
print(f"Timestamp       : {timestamp_str}")
print(f"Base dir        : {BASE_DIR}")
print(f"Extraction map  : {EXTRACTION_MAP_PATH}")
print(f"Frames/video    : {TARGET_FRAMES_PER_VIDEO}")
print()

# ─── STEP 1: LOAD EXTRACTION MAP ─────────────────────────────────────────────

if not os.path.isfile(EXTRACTION_MAP_PATH):
    raise FileNotFoundError(
        f"FATAL: extraction_map.json not found at {EXTRACTION_MAP_PATH}. "
        "Cell 2 must run first."
    )

with open(EXTRACTION_MAP_PATH, 'r') as f:
    extraction_map = json.load(f)

print(f"Loaded extraction_map.json ({os.path.getsize(EXTRACTION_MAP_PATH)} bytes)")

# ─── STEP 2: VALIDATE REQUIRED SECTIONS ──────────────────────────────────────

# We expect these top-level keys (adjust if Cell 2 uses different names)
# Attempt to locate real bucket paths and fake manipulation paths flexibly.

def get_nested(d, *keys):
    """Safely traverse nested dict keys."""
    current = d
    for k in keys:
        if isinstance(current, dict) and k in current:
            current = current[k]
        else:
            return None
    return current

# Try common structures Cell 2 might use
real_bucket_paths = {}
fake_manip_paths = {}

# Strategy: look for keys containing the bucket/manipulation video paths
# Common patterns: extraction_map["real_videos"]["bucket_a"], or
# extraction_map["eligible_real"]["bucket_a"], or
# extraction_map["buckets"]["bucket_a"]["real_videos"], etc.

# Real bucket key aliases: Cell 2 may use "A"/"B"/"C"/"D" instead of "bucket_a" etc.
REAL_BUCKET_ALIASES = {
    "bucket_a": ["bucket_a", "A", "a"],
    "bucket_b": ["bucket_b", "B", "b"],
    "bucket_c": ["bucket_c", "C", "c"],
    "bucket_d": ["bucket_d", "D", "d"],
}

for bkey in REAL_BUCKET_MAP:
    aliases = REAL_BUCKET_ALIASES[bkey]
    candidates = []
    for alias in aliases:
        candidates.extend([
            get_nested(extraction_map, "real_video_paths", alias),
            get_nested(extraction_map, "bucket_reals", alias),
            get_nested(extraction_map, "real_videos", alias),
            get_nested(extraction_map, "eligible_real", alias),
            get_nested(extraction_map, f"real_{alias}"),
            get_nested(extraction_map, "buckets", alias, "real_videos"),
            get_nested(extraction_map, "buckets", alias, "real"),
            get_nested(extraction_map, "buckets", alias, "eligible_real"),
        ])
    for c in candidates:
        if c is not None and isinstance(c, list):
            real_bucket_paths[bkey] = c
            break

for mkey in FAKE_MANIP_MAP:
    candidates = [
        get_nested(extraction_map, "fake_video_paths", mkey),
        get_nested(extraction_map, "eligible_fakes", mkey),
        get_nested(extraction_map, "fake_videos", mkey),
        get_nested(extraction_map, "eligible_fake", mkey),
        get_nested(extraction_map, f"fake_{mkey}"),
        get_nested(extraction_map, "manipulations", mkey, "eligible"),
        get_nested(extraction_map, "manipulations", mkey, "videos"),
        get_nested(extraction_map, "manipulations", mkey),
    ]
    for c in candidates:
        if c is not None and isinstance(c, list):
            fake_manip_paths[mkey] = c
            break

# If the flexible search didn't find everything, dump keys to help debug
if len(real_bucket_paths) < 4 or len(fake_manip_paths) < 4:
    print("\n[DEBUG] Top-level keys in extraction_map.json:")
    for k in extraction_map:
        v = extraction_map[k]
        vtype = type(v).__name__
        vlen = len(v) if isinstance(v, (list, dict)) else "n/a"
        print(f"  '{k}' -> {vtype} (len={vlen})")
        if isinstance(v, dict):
            for sk in v:
                sv = v[sk]
                svtype = type(sv).__name__
                svlen = len(sv) if isinstance(sv, (list, dict)) else "n/a"
                print(f"    '{sk}' -> {svtype} (len={svlen})")

    missing_real = [b for b in REAL_BUCKET_MAP if b not in real_bucket_paths]
    missing_fake = [m for m in FAKE_MANIP_MAP if m not in fake_manip_paths]

    raise ValueError(
        f"FATAL: Could not locate all required sections in extraction_map.json.\n"
        f"  Missing real buckets     : {missing_real}\n"
        f"  Missing fake manips      : {missing_fake}\n"
        f"  Adjust the key lookup patterns in Cell 3 to match Cell 2's output format."
    )

print("\nExtraction map sections located:")
for bkey, paths in real_bucket_paths.items():
    print(f"  Real {bkey:16s} : {len(paths)} videos")
for mkey, paths in fake_manip_paths.items():
    print(f"  Fake {mkey:16s} : {len(paths)} videos")

total_eligible = sum(len(v) for v in real_bucket_paths.values()) + \
                 sum(len(v) for v in fake_manip_paths.values())

if total_eligible == 0:
    raise ValueError("FATAL: No eligible videos found in extraction_map.json.")

print(f"\nTotal eligible videos: {total_eligible}")

# ─── STEP 3: VALIDATE LOCAL VIDEO PATHS EXIST ────────────────────────────────

print(f"\n{'='*70}")
print("VALIDATING LOCAL VIDEO PATHS")
print(f"{'='*70}")

missing_count = 0
present_count = 0

def validate_paths(path_list, label):
    global missing_count, present_count
    local_missing = 0
    for p in path_list:
        if os.path.isfile(p):
            present_count += 1
        else:
            missing_count += 1
            local_missing += 1
            if local_missing <= 3:
                print(f"  [MISSING] {label}: {p}")
    if local_missing > 3:
        print(f"  [MISSING] {label}: ... and {local_missing - 3} more")
    return local_missing

for bkey, paths in real_bucket_paths.items():
    validate_paths(paths, f"real/{bkey}")
for mkey, paths in fake_manip_paths.items():
    validate_paths(paths, f"fake/{mkey}")

print(f"\n  Videos present : {present_count}")
print(f"  Videos missing : {missing_count}")

if present_count == 0:
    raise FileNotFoundError(
        "FATAL: None of the mapped video paths exist locally. "
        "Did Cell 1 run? Is RAW_VIDEOS_DIR correct?"
    )

# ─── STEP 4: PREPARE OUTPUT DIRECTORY ────────────────────────────────────────

if os.path.exists(EXTRACTED_DIR):
    print(f"\nRemoving previous extracted_frames directory: {EXTRACTED_DIR}")
    shutil.rmtree(EXTRACTED_DIR)

os.makedirs(EXTRACTED_DIR, exist_ok=True)
print(f"Created output directory: {EXTRACTED_DIR}")

# ─── STEP 5: EXTRACT FRAMES ──────────────────────────────────────────────────

print(f"\n{'='*70}")
print("FRAME EXTRACTION")
print(f"{'='*70}")

csv_log_rows = []
video_counter = 0

# Counters for manifest
stats = {
    "real_attempted": {},
    "real_frames": {},
    "fake_attempted": {},
    "fake_frames": {},
    "videos_opened": 0,
    "videos_failed": 0,
    "total_frames": 0,
    "warnings": [],
}

def process_video_group(path_list, class_type, category_key, output_subdir):
    """
    Extract frames from all videos in a single category group.
    Updates stats and csv_log_rows in-place.
    """
    global video_counter

    out_dir = os.path.join(EXTRACTED_DIR, output_subdir)
    attempted = 0
    group_frames = 0

    for vpath in path_list:
        attempted += 1
        video_counter += 1

        result = extract_frames_from_video(
            video_path=vpath,
            output_dir=out_dir,
            target_count=TARGET_FRAMES_PER_VIDEO,
            video_counter=video_counter,
        )

        result["class_type"] = class_type
        result["category"] = category_key
        csv_log_rows.append(result)

        if result["status"] == "failed":
            stats["videos_failed"] += 1
            if result["failure_reason"] != "file_not_found":
                print(f"  [FAIL] {os.path.basename(vpath)}: {result['failure_reason']}")
        else:
            stats["videos_opened"] += 1
            if result["status"] == "partial":
                stats["warnings"].append(
                    f"Partial read: {os.path.basename(vpath)} "
                    f"({result['frames_saved']}/{TARGET_FRAMES_PER_VIDEO})"
                )

        group_frames += result["frames_saved"]
        stats["total_frames"] += result["frames_saved"]

        # Progress print every 50 videos
        if video_counter % 50 == 0:
            print(f"  ... processed {video_counter} videos so far ...")

    return attempted, group_frames


# ── Real buckets ──
print("\n--- Real Videos ---")
for bkey, subdir in REAL_BUCKET_MAP.items():
    paths = real_bucket_paths[bkey]
    print(f"\n  Processing real/{bkey} ({len(paths)} videos) ...")
    attempted, frames = process_video_group(paths, "real", bkey, subdir)
    stats["real_attempted"][bkey] = attempted
    stats["real_frames"][bkey] = frames
    print(f"    Attempted: {attempted} | Frames extracted: {frames}")

# ── Fake manipulations ──
print("\n--- Fake Videos ---")
for mkey, subdir in FAKE_MANIP_MAP.items():
    paths = fake_manip_paths[mkey]
    print(f"\n  Processing fake/{mkey} ({len(paths)} videos) ...")
    attempted, frames = process_video_group(paths, "fake", mkey, subdir)
    stats["fake_attempted"][mkey] = attempted
    stats["fake_frames"][mkey] = frames
    print(f"    Attempted: {attempted} | Frames extracted: {frames}")

print(f"\n{'='*70}")
print("EXTRACTION COMPLETE")
print(f"{'='*70}")
print(f"  Videos processed  : {video_counter}")
print(f"  Videos opened OK  : {stats['videos_opened']}")
print(f"  Videos failed     : {stats['videos_failed']}")
print(f"  Total frames      : {stats['total_frames']}")

if stats['total_frames'] == 0:
    raise RuntimeError("FATAL: Zero frames extracted. Something is critically wrong.")

# ─── STEP 6: WRITE CSV LOG ───────────────────────────────────────────────────

print(f"\nWriting CSV log: {LOG_CSV_PATH}")

csv_fields = [
    "video_path", "class_type", "category", "output_dir",
    "total_frames_metadata", "target_frames", "frames_saved",
    "status", "failure_reason",
]

with open(LOG_CSV_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields, extrasaction='ignore')
    writer.writeheader()
    for row in csv_log_rows:
        writer.writerow(row)

print(f"  Rows written: {len(csv_log_rows)}")

# ─── STEP 7: CREATE ZIP ARCHIVE ──────────────────────────────────────────────

archive_size = create_zip_archive(EXTRACTED_DIR, ARCHIVE_PATH)

# ─── STEP 8: WRITE MANIFEST ──────────────────────────────────────────────────

print(f"\nWriting manifest: {MANIFEST_PATH}")

manifest_lines = []
manifest_lines.append("=" * 70)
manifest_lines.append("FF++ EXTRACTED FRAMES MANIFEST")
manifest_lines.append("=" * 70)
manifest_lines.append(f"Timestamp               : {timestamp_str}")
manifest_lines.append(f"Base directory           : {BASE_DIR}")
manifest_lines.append(f"Extraction map path      : {EXTRACTION_MAP_PATH}")
manifest_lines.append(f"Frames per video target  : {TARGET_FRAMES_PER_VIDEO}")
manifest_lines.append("")

manifest_lines.append("--- Real Videos by Bucket ---")
for bkey in REAL_BUCKET_MAP:
    attempted = stats["real_attempted"].get(bkey, 0)
    frames = stats["real_frames"].get(bkey, 0)
    manifest_lines.append(f"  {bkey:16s} : {attempted:4d} videos attempted, {frames:6d} frames extracted")

manifest_lines.append("")
manifest_lines.append("--- Fake Videos by Manipulation ---")
for mkey in FAKE_MANIP_MAP:
    attempted = stats["fake_attempted"].get(mkey, 0)
    frames = stats["fake_frames"].get(mkey, 0)
    manifest_lines.append(f"  {mkey:16s} : {attempted:4d} videos attempted, {frames:6d} frames extracted")

manifest_lines.append("")
manifest_lines.append("--- Summary ---")
manifest_lines.append(f"  Total videos processed   : {video_counter}")
manifest_lines.append(f"  Total videos opened OK   : {stats['videos_opened']}")
manifest_lines.append(f"  Total videos failed      : {stats['videos_failed']}")
manifest_lines.append(f"  Total frames extracted    : {stats['total_frames']}")
manifest_lines.append(f"  Archive size             : {archive_size / (1024**2):.2f} MB")

manifest_lines.append("")
manifest_lines.append("--- S3 Destinations ---")
manifest_lines.append(f"  Bucket  : {S3_BUCKET}")
manifest_lines.append(f"  Archive : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_extracted_frames.zip")
manifest_lines.append(f"  Manifest: s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_extracted_frames_manifest.txt")
manifest_lines.append(f"  CSV Log : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_frame_extraction_log.csv")

if stats["warnings"]:
    manifest_lines.append("")
    manifest_lines.append(f"--- Warnings ({len(stats['warnings'])}) ---")
    for w in stats["warnings"][:50]:
        manifest_lines.append(f"  {w}")
    if len(stats["warnings"]) > 50:
        manifest_lines.append(f"  ... and {len(stats['warnings']) - 50} more warnings")

manifest_lines.append("")
elapsed = time.time() - start_time
manifest_lines.append(f"Extraction elapsed time: {elapsed:.1f} seconds")
manifest_lines.append("=" * 70)

with open(MANIFEST_PATH, 'w') as f:
    f.write("\n".join(manifest_lines))

print(f"  Manifest written ({len(manifest_lines)} lines)")

# ─── STEP 9: UPLOAD TO S3 ────────────────────────────────────────────────────

print(f"\n{'='*70}")
print("S3 UPLOAD")
print(f"{'='*70}")

s3 = boto3.client("s3")

uploads = [
    (ARCHIVE_PATH, f"{S3_PREFIX}/ffpp_extracted_frames.zip"),
    (MANIFEST_PATH, f"{S3_PREFIX}/ffpp_extracted_frames_manifest.txt"),
    (LOG_CSV_PATH, f"{S3_PREFIX}/ffpp_frame_extraction_log.csv"),
]

for local_path, s3_key in uploads:
    upload_and_verify(s3, local_path, S3_BUCKET, s3_key)

print("\nAll uploads verified successfully.")

# ─── STEP 10: LOCAL CLEANUP ──────────────────────────────────────────────────

print(f"\n{'='*70}")
print("LOCAL CLEANUP")
print(f"{'='*70}")

# Clean extracted frames directory (the archive is already on S3)
if os.path.exists(EXTRACTED_DIR):
    shutil.rmtree(EXTRACTED_DIR)
    print(f"  Removed: {EXTRACTED_DIR}")

# Clean local archive (already uploaded and verified)
if os.path.isfile(ARCHIVE_PATH):
    os.remove(ARCHIVE_PATH)
    print(f"  Removed: {ARCHIVE_PATH}")

# Keep manifest and CSV log locally for quick reference
print(f"  Kept locally: {MANIFEST_PATH}")
print(f"  Kept locally: {LOG_CSV_PATH}")

# NOTE: Raw videos from Cell 1 are NOT deleted here.
# They remain in {RAW_VIDEOS_DIR} in case you need to re-run extraction.
# Delete them manually or in a separate cleanup cell if disk space is needed.
print(f"  Raw videos preserved: {RAW_VIDEOS_DIR}")

# ─── DONE ─────────────────────────────────────────────────────────────────────

total_elapsed = time.time() - start_time
print(f"\n{'='*70}")
print("CELL 3 COMPLETE")
print(f"{'='*70}")
print(f"  Total time     : {total_elapsed:.1f} seconds ({total_elapsed/60:.1f} min)")
print(f"  Frames on S3   : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_extracted_frames.zip")
print(f"  Manifest on S3 : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_extracted_frames_manifest.txt")
print(f"  Log on S3      : s3://{S3_BUCKET}/{S3_PREFIX}/ffpp_frame_extraction_log.csv")
print(f"  Total frames   : {stats['total_frames']}")
print(f"  Failed videos  : {stats['videos_failed']}")
print()

Manual Upload from EBS to S3 Due to AWS Credential Loss.

In [ ]:
import os
import boto3

S3_BUCKET = "deepfake-d-100k-dataset-tw26"
S3_PREFIX = "datasets/FFPlus/processed"
BASE_DIR  = "/home/ec2-user/SageMaker/extractor_temp"

ARCHIVE_PATH  = os.path.join(BASE_DIR, "ffpp_extracted_frames.zip")
MANIFEST_PATH = os.path.join(BASE_DIR, "ffpp_extracted_frames_manifest.txt")
LOG_CSV_PATH  = os.path.join(BASE_DIR, "ffpp_frame_extraction_log.csv")

files_to_upload = [
    (ARCHIVE_PATH,  f"{S3_PREFIX}/ffpp_extracted_frames.zip"),
    (MANIFEST_PATH, f"{S3_PREFIX}/ffpp_extracted_frames_manifest.txt"),
    (LOG_CSV_PATH,  f"{S3_PREFIX}/ffpp_frame_extraction_log.csv"),
]

def human_mb(n):
    return f"{n / (1024**2):.2f} MB"

s3 = boto3.client("s3")

print("=== S3 UPLOAD ===\n")

for local_path, s3_key in files_to_upload:
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Missing local file: {local_path}")

    local_size = os.path.getsize(local_path)
    if local_size == 0:
        raise RuntimeError(f"Local file is empty: {local_path}")

    print(f"Uploading: {os.path.basename(local_path)} ({human_mb(local_size)})")
    print(f"  -> s3://{S3_BUCKET}/{s3_key}")

    s3.upload_file(local_path, S3_BUCKET, s3_key)

    head = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
    remote_size = head["ContentLength"]

    if remote_size != local_size:
        raise RuntimeError(
            f"Size mismatch after upload for {s3_key}\n"
            f"  Local  : {local_size:,} bytes\n"
            f"  Remote : {remote_size:,} bytes"
        )

    print("  Verified.\n")

print("ALL FILES UPLOADED AND VERIFIED.")

RetinaFace Face Extraction on Processed FF++ Data.

- Installers.

In [ ]:
!pip install retina-face imagehash pybktree opencv-python-headless

In [ ]:
%pip install -U tf-keras

In [ ]:
# FF++ RetinaFace Face Extractor — AWS SageMaker Production.

# Architecture:
#   1.  Wipe and recreate local staging directories (Fix 1)
#   2.  Download ffpp_extracted_frames.zip from S3
#   3.  Unpack via native subprocess unzip (Fix 4)
#   4.  Validate unpacked folder structure (Fix 3)
#   5.  Scan all frame paths per category
#   6.  Build FF++ identity graph from fake stems → connected components (Fix 6)
#   7.  Assign whole connected components to train/val/test splits (Fix 6/7)
#       - component membership enforces zero leakage of target/donor identities
#       - real stems resolved to splits via their component membership
#   8.  Run RetinaFace / blur / pHash dedup / resize (260×260) per frame
#       - per quota-bucket BK trees preserve dedup isolation (Fix 8)
#       - pbar.update(1) called on every accepted face (Fix 5)
#   9.  Fail hard if any quota bucket is underfilled (Fix 2)
#  10.  Zip final accepted faces
#  11.  Write CSV log + manifest (includes graph/component stats) (Fix 10)
#  12.  Upload zip + manifest + CSV to S3
#  13.  Verify uploads (head_object size check)
#  14.  Clean local staging only after verification

import csv
import glob
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"     # Bypasses the Keras 3 architecture crash.
os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
import random
import re
import shutil
import subprocess
import time
import warnings
from collections import defaultdict
from datetime import datetime, timezone

import boto3
import cv2
import imagehash
import numpy as np
import pybktree
from PIL import Image
from retinaface import RetinaFace
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Determinism
random.seed(42)
np.random.seed(42)

# CONFIGURATION

S3_BUCKET        = 'deepfake-d-100k-dataset-tw26'
S3_INPUT_KEY     = 'datasets/FFPlus/processed/ffpp_extracted_frames.zip'
S3_OUTPUT_PREFIX = 'datasets/FFPlus/final'

BASE_DIR         = '/home/ec2-user/SageMaker/ffpp_face_stage'
RAW_ZIP_DIR      = os.path.join(BASE_DIR, 'raw_zip')
RAW_FRAMES_DIR   = os.path.join(BASE_DIR, 'raw_frames')
FINAL_FACES_DIR  = os.path.join(BASE_DIR, 'final_faces')
LOGS_DIR         = os.path.join(BASE_DIR, 'logs')

LOCAL_ZIP_IN     = os.path.join(RAW_ZIP_DIR, 'ffpp_extracted_frames.zip')
FINAL_ZIP_BASE   = os.path.join(BASE_DIR,    'ffpp_final_faces_20k')   # .zip appended by make_archive
CSV_LOG_PATH     = os.path.join(LOGS_DIR,    'ffpp_final_faces_20k_log.csv')
MANIFEST_PATH    = os.path.join(LOGS_DIR,    'ffpp_final_faces_20k_manifest.txt')

S3_OUT_KEYS = {
    'zip'      : f'{S3_OUTPUT_PREFIX}/ffpp_final_faces_20k.zip',
    'manifest' : f'{S3_OUTPUT_PREFIX}/ffpp_final_faces_20k_manifest.txt',
    'csv'      : f'{S3_OUTPUT_PREFIX}/ffpp_final_faces_20k_log.csv',
}

# Validated quality thresholds.
CONFIDENCE_THRESHOLD = 0.90
MIN_FACE_SIZE        = 30
MIN_FACE_RATIO       = 0.005
BLUR_THRESHOLD       = 25
PHASH_HAMMING_MAX    = 2
PADDING              = 25
RESIZE_TARGET        = (260, 260)   # EfficientNet-B2 native resolution

# Production quotas — 20K total (10K real + 10K fake)
target_quotas = {
    'train_real'               : 7000,
    'val_real'                 : 1500,
    'test_real'                : 1500,
    'train_fake_deepfakes'     : 1750,
    'val_fake_deepfakes'       : 375,
    'test_fake_deepfakes'      : 375,
    'train_fake_face2face'     : 1750,
    'val_fake_face2face'       : 375,
    'test_fake_face2face'      : 375,
    'train_fake_faceswap'      : 1750,
    'val_fake_faceswap'        : 375,
    'test_fake_faceswap'       : 375,
    'train_fake_neuraltextures': 1750,
    'val_fake_neuraltextures'  : 375,
    'test_fake_neuraltextures' : 375,
}

SPLIT_ORDER = ['train', 'val', 'test']

# Maps input folder name → quota category suffix
INPUT_CATEGORY_MAP = {
    'real'           : 'real',
    'deepfakes'      : 'fake_deepfakes',
    'face2face'      : 'fake_face2face',
    'faceswap'       : 'fake_faceswap',
    'neuraltextures' : 'fake_neuraltextures',
}

FAKE_CATS = ['deepfakes', 'face2face', 'faceswap', 'neuraltextures']

# S3 client
s3 = boto3.client('s3')

# HELPERS — FILE SYSTEM

def clean_and_create_dirs():
    """Fix 1: Wipe and recreate all staging dirs to prevent cross-run contamination."""
    for d in [RAW_ZIP_DIR, RAW_FRAMES_DIR, FINAL_FACES_DIR, LOGS_DIR]:
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)
    print(f"  Staging directories wiped and recreated under: {BASE_DIR}")


def make_output_split_dirs():
    for split in SPLIT_ORDER:
        for cat_suffix in INPUT_CATEGORY_MAP.values():
            os.makedirs(os.path.join(FINAL_FACES_DIR, split, cat_suffix), exist_ok=True)

# HELPERS — S3

def download_from_s3(s3_key, local_path):
    size_obj = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    size_gb  = size_obj / (1024 ** 3)
    print(f"  Downloading s3://{S3_BUCKET}/{s3_key}  ({size_gb:.2f} GB) ...")
    s3.download_file(S3_BUCKET, s3_key, local_path)
    local_size = os.path.getsize(local_path)
    if local_size != size_obj:
        raise RuntimeError(
            f"Download size mismatch.\n  Expected: {size_obj}\n  Got: {local_size}"
        )
    print(f"  Saved → {local_path}  ({local_size / (1024**3):.2f} GB)")


def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  → s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local: {local_size} bytes\n"
            f"  S3   : {s3_size} bytes\n"
            f"Upload may be incomplete. Local artifacts preserved."
        )
    print(f"  Verified: {os.path.basename(local_path)}  ({s3_size / (1024**3):.2f} GB)")

# HELPERS — FILENAME PARSING

_STEM_RE = re.compile(r'^(.+?)_frame_\d+')

def parse_source_video_stem(filename):
    """
    Strip '_frame_XXXXXX.jpg' suffix to recover the source video stem.
      '1001_000_frame_000123.jpg'      -> '1001_000'
      '1001_000_003_frame_000100.jpg'  -> '1001_000_003'
    Falls back to bare basename (no extension) if marker is absent.
    """
    basename = os.path.splitext(os.path.basename(filename))[0]
    m = _STEM_RE.match(basename)
    return m.group(1) if m else basename

def parse_fake_stem_ids(stem):
    """
    Parse both integer IDs from a fake source stem, ignoring junk prefixes.
    e.g., '1001_000_003' -> (0, 3)
    """
    parts = stem.split('_')
    if len(parts) >= 2:
        try:
            return int(parts[-2]), int(parts[-1])
        except ValueError:
            return None
    return None

def parse_real_stem_id(stem):
    """
    Parse the single integer ID from a real source stem, ignoring junk prefixes.
    e.g., '1001_000' -> 0
    """
    parts = stem.split('_')
    if len(parts) >= 1:
        try:
            return int(parts[-1])
        except ValueError:
            return None
    return None

# FIX 6 — FF++ IDENTITY GRAPH + CONNECTED COMPONENTS + SPLIT ASSIGNMENT

def build_identity_graph(fake_stems):
    """
    Build undirected graph over all fake source-video stems.
    '000_003' adds an undirected edge between node 0 and node 3.
    Returns: adjacency dict, set of all nodes, set of unique normalized edges.
    """
    adjacency    = defaultdict(set)
    unique_edges = set()

    for stem in fake_stems:
        ids = parse_fake_stem_ids(stem)
        if ids is None:
            continue
        id_a, id_b = ids
        adjacency[id_a].add(id_b)
        adjacency[id_b].add(id_a)
        unique_edges.add(tuple(sorted((id_a, id_b))))

    return dict(adjacency), set(adjacency.keys()), unique_edges


def compute_connected_components(adjacency, all_nodes):
    """BFS connected components. Returns list of frozensets."""
    visited    = set()
    components = []
    for start in sorted(all_nodes):
        if start in visited:
            continue
        component = set()
        queue     = [start]
        while queue:
            node = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            component.add(node)
            for nb in adjacency.get(node, []):
                if nb not in visited:
                    queue.append(nb)
        components.append(frozenset(component))
    return components


def assign_components_to_splits(components):
    """
    Assign whole connected components to train/val/test — never split a component.

    Strategy:
    - Shuffle components with fixed seed, then sort largest-first (stable tie-break)
    - Fill train up to 70% of total identities, then val up to 15%, test gets the rest
    - Returns: node_id -> split name, and identity counts per split
    """
    comp_list = list(components)
    random.shuffle(comp_list)                         # seed=42 set globally
    comp_list.sort(key=lambda c: -len(c))             # stable: largest first

    total_ids    = sum(len(c) for c in comp_list)
    train_target = int(total_ids * 0.70)
    val_target   = int(total_ids * 0.15)

    split_counts  = {'train': 0, 'val': 0, 'test': 0}
    node_to_split = {}

    for comp in comp_list:
        if split_counts['train'] < train_target:
            chosen = 'train'
        elif split_counts['val'] < val_target:
            chosen = 'val'
        else:
            chosen = 'test'
        split_counts[chosen] += len(comp)
        for node in comp:
            node_to_split[node] = chosen

    return node_to_split, split_counts


def resolve_stem_split(stem, node_to_split, is_fake):
    """
    Resolve split assignment for a source-video stem via component membership.

    Real  '000'     -> look up node 0
    Fake  '000_003' -> both IDs must map to the same split (component guarantee)
    Returns split string or None if stem cannot be mapped.
    """
    if is_fake:
        ids = parse_fake_stem_ids(stem)
        if ids is None:
            return None
        id_a, id_b = ids
        split_a = node_to_split.get(id_a)
        split_b = node_to_split.get(id_b)
        # Component guarantee: these should always agree; guard defensively
        if split_a is None or split_b is None or split_a != split_b:
            return None
        return split_a
    else:
        node_id = parse_real_stem_id(stem)
        if node_id is None:
            return None
        return node_to_split.get(node_id)

# HELPERS — RETINAFACE CROP ENGINE

def master_crop_engine(
    img_path,
    save_path,
    bucket_tree,
    padding=PADDING,
    min_face_size=MIN_FACE_SIZE,
    min_face_ratio=MIN_FACE_RATIO,
    blur_threshold=BLUR_THRESHOLD,
    confidence_threshold=CONFIDENCE_THRESHOLD,
):
    """
    Load -> DOWNSCALE FOR DETECTION -> detect -> select largest -> 
    UPSCALE BOX -> quality gate -> crop+pad -> blur check -> resize -> pHash -> save.
    """
    try:
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            return False, "Corrupted Image", 0.0, 0.0, "", "[]"

        height, width = img_cv.shape[:2]

        # --- THE 1080p BYPASS ---
        # Downscale by 50% strictly for the RetinaFace scan to prevent VRAM deadlock
        scale_factor = 0.5
        small_cv = cv2.resize(img_cv, (0, 0), fx=scale_factor, fy=scale_factor)

        faces = RetinaFace.detect_faces(small_cv)
        if not isinstance(faces, dict) or len(faces) == 0:
            return False, "No Face Detected", 0.0, 0.0, "", "[]"

        largest_area = 0
        best_face    = None
        for face in faces.values():
            box = face.get('facial_area')
            if box is None:
                continue
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > largest_area:
                largest_area = area
                best_face    = face

        if best_face is None:
            return False, "No Valid Face Found", 0.0, 0.0, "", "[]"

        # --- UPSCALE THE BOX ---
        # Multiply the coordinates by 2 so we crop from the original 1080p High-Res image
        raw_box = best_face['facial_area']
        box = [int(coord / scale_factor) for coord in raw_box]
        
        str_box    = f"[{box[0]}, {box[1]}, {box[2]}, {box[3]}]"
        confidence = best_face['score']

        if confidence < confidence_threshold:
            return False, "Low Confidence", confidence, 0.0, "", str_box

        face_w = box[2] - box[0]
        face_h = box[3] - box[1]

        if face_w < min_face_size or face_h < min_face_size:
            return False, "Resolution Too Small", confidence, 0.0, "", str_box

        if (face_w * face_h) / (width * height) < min_face_ratio:
            return False, "Face Ratio Too Small", confidence, 0.0, "", str_box

        x_min = max(0,      int(box[0]) - padding)
        y_min = max(0,      int(box[1]) - padding)
        x_max = min(width,  int(box[2]) + padding)
        y_max = min(height, int(box[3]) + padding)

        cropped_cv  = img_cv[y_min:y_max, x_min:x_max]
        gray_crop   = cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2GRAY)
        blur_val    = cv2.Laplacian(gray_crop, cv2.CV_64F).var()

        if blur_val < blur_threshold:
            return False, "Motion Blur Detected", confidence, blur_val, "", str_box

        cropped_pil  = Image.fromarray(cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2RGB))
        standardized = cropped_pil.resize(RESIZE_TARGET, Image.BICUBIC)

        new_hash = imagehash.phash(standardized)
        matches  = bucket_tree.find(new_hash, PHASH_HAMMING_MAX)
        if matches:
            return False, f"Duplicate Face (Hamming <= {PHASH_HAMMING_MAX})", confidence, blur_val, str(new_hash), str_box

        bucket_tree.add(new_hash)
        standardized.save(save_path, format='JPEG', quality=95)

        return True, "Accepted", confidence, blur_val, str(new_hash), str_box

    except Exception as exc:
        return False, f"Engine Error: {str(exc)}", 0.0, 0.0, "", "[]"

# HELPERS — QUOTA UTILITIES

def quota_key(split, cat_suffix):
    return f"{split}_{cat_suffix}"


def quotas_all_filled(accepted_counts):
    return all(accepted_counts[k] >= target_quotas[k] for k in target_quotas)

# CORE PROCESSING

def process_category(
    cat_key,
    cat_suffix,
    frame_paths,
    node_to_split,
    is_fake,
    accepted_counts,
    rejection_reasons,
    csv_writer,
    bk_trees,
    pbar,
):
    """
    Group frames by source-video stem.
    Resolve each stem's split via component membership (Fix 6).
    Process frames through crop engine; accept into the correct split quota bucket.
    pbar.update(1) fired on each accepted face (Fix 5).
    """
    groups = defaultdict(list)
    for fp in frame_paths:
        stem = parse_source_video_stem(fp)
        groups[stem].append(fp)

    group_keys = sorted(groups.keys())
    random.shuffle(group_keys)   # seed=42 set globally

    for stem in group_keys:
        if quotas_all_filled(accepted_counts):
            return

        assigned_split = resolve_stem_split(stem, node_to_split, is_fake)
        if assigned_split is None:
            for fp in groups[stem]:
                rejection_reasons['Unmapped Stem'] += 1
                csv_writer.writerow([
                    fp, stem, 'none', cat_suffix,
                    'Rejected', 'Unmapped Stem',
                    0.0, 0.0, '', '[]'
                ])
            continue

        quota_k = quota_key(assigned_split, cat_suffix)

        # Skip group entirely if its designated quota bucket is already full
        if accepted_counts[quota_k] >= target_quotas[quota_k]:
            continue

        tree    = bk_trees[quota_k]
        out_dir = os.path.join(FINAL_FACES_DIR, assigned_split, cat_suffix)

        for img_path in groups[stem]:
            if accepted_counts[quota_k] >= target_quotas[quota_k]:
                break

            file_name = os.path.basename(img_path)
            save_path = os.path.join(out_dir, file_name)

            passed, reason, conf, blur, phash_val, bbox = master_crop_engine(
                img_path, save_path, tree
            )

            if passed:
                accepted_counts[quota_k] += 1
                pbar.update(1)   # Fix 5: live update on every acceptance
                csv_writer.writerow([
                    img_path, stem, assigned_split, cat_suffix,
                    'Accepted', 'None',
                    round(conf, 4), round(blur, 2), phash_val, bbox
                ])
            else:
                rejection_reasons[reason] += 1
                csv_writer.writerow([
                    img_path, stem, assigned_split, cat_suffix,
                    'Rejected', reason,
                    round(conf, 4), round(blur, 2), phash_val, bbox
                ])

# MANIFEST

def write_manifest(
    accepted_counts,
    rejection_reasons,
    zip_path,
    start_time,
    graph_stats,
    split_identity_counts,
):
    timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    elapsed   = time.time() - start_time
    zip_size  = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    total_acc = sum(accepted_counts.values())
    total_rej = sum(rejection_reasons.values())

    lines = [
        "FF++ RetinaFace Face Extractor — Production Manifest",
        "=" * 60,
        f"  Timestamp              : {timestamp}",
        f"  Elapsed                : {elapsed:.0f} seconds  ({elapsed/60:.1f} min)",
        f"  Input S3 object        : s3://{S3_BUCKET}/{S3_INPUT_KEY}",
        f"  Local workspace        : {BASE_DIR}",
        "",
        "  Quality thresholds (validated sandbox defaults):",
        f"    Confidence           : >= {CONFIDENCE_THRESHOLD}",
        f"    Min face size        : {MIN_FACE_SIZE} px",
        f"    Min face ratio       : {MIN_FACE_RATIO}",
        f"    Blur threshold       : {BLUR_THRESHOLD} (Laplacian variance)",
        f"    pHash Hamming max    : <= {PHASH_HAMMING_MAX}",
        f"    Padding              : {PADDING} px",
        f"    Resize target        : {RESIZE_TARGET[0]}x{RESIZE_TARGET[1]}",
        "",
        "  Split assignment strategy:",
        "    Component-based (FF++ identity graph -> connected components)",
        "    Whole components assigned to train/val/test — no component ever split",
        "    Real stems mapped via identity node to same split as their fake pairings",
        "    Target proportions: train=70%, val=15%, test=15%",
        "",
        "  Identity graph statistics:",
        f"    Graph nodes              : {graph_stats.get('nodes', 0):,}",
        f"    Unique graph edges       : {graph_stats.get('unique_edges', 0):,}",
        f"    Connected components     : {graph_stats.get('components', 0):,}",
        f"    Largest component size   : {graph_stats.get('largest_component', 0):,}",
        "",
        "  Split identity counts (post-assignment):",
        f"    train : {split_identity_counts.get('train', 0):,} identities",
        f"    val   : {split_identity_counts.get('val', 0):,} identities",
        f"    test  : {split_identity_counts.get('test', 0):,} identities",
        "",
        "=" * 60,
        f"  Total accepted         : {total_acc:,}  / {sum(target_quotas.values()):,}",
        f"  Total rejected         : {total_rej:,}",
        "",
        "  Accepted counts per quota bucket:",
    ]

    for k, target in sorted(target_quotas.items()):
        got  = accepted_counts.get(k, 0)
        flag = "" if got >= target else "  *** UNDERFILLED ***"
        lines.append(f"    {k:<40}: {got:>6,} / {target:>6,}{flag}")

    lines += ["", "  Rejection reasons:"]
    for reason, count in sorted(rejection_reasons.items(), key=lambda x: -x[1]):
        lines.append(f"    {reason:<45}: {count:>6,}")

    lines += [
        "",
        "=" * 60,
        f"  Output zip size        : {zip_size:.2f} GB",
        "  S3 output destinations:",
        f"    Zip      : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}",
        f"    Manifest : s3://{S3_BUCKET}/{S3_OUT_KEYS['manifest']}",
        f"    CSV log  : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}",
    ]

    with open(MANIFEST_PATH, 'w') as f:
        f.write("\n".join(lines) + "\n")

    print(f"  Manifest written -> {MANIFEST_PATH}")

# MAIN

start_time = time.time()

print("\n" + "=" * 60)
print("  FF++ RetinaFace Face Extractor — SageMaker Production (Final)")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"  Target: {sum(target_quotas.values()):,} accepted faces  (10K real + 10K fake)")
print("=" * 60 + "\n")

# [1] Wipe and recreate local staging (Fix 1)
print("[1/10] Wiping and recreating local staging directories...")
clean_and_create_dirs()
make_output_split_dirs()

# [2] Download input zip from S3
print("\n[2/10] Downloading extracted frames zip from S3...")
download_from_s3(S3_INPUT_KEY, LOCAL_ZIP_IN)

if not os.path.exists(LOCAL_ZIP_IN) or os.path.getsize(LOCAL_ZIP_IN) == 0:
    raise RuntimeError(f"Input zip missing or empty after download: {LOCAL_ZIP_IN}")

# [3] Unpack via native subprocess unzip (Fix 4)
print("\n[3/10] Unpacking extracted frames (native unzip)...")
try:
    subprocess.run(
        ["unzip", "-q", LOCAL_ZIP_IN, "-d", RAW_FRAMES_DIR],
        check=True
    )
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"unzip failed with return code {e.returncode}. Aborting.")

print(f"  Unpacked -> {RAW_FRAMES_DIR}")

# Free SSD after unpacking
os.remove(LOCAL_ZIP_IN)
print(f"  Input zip removed from local SSD.")

RAW_FRAMES_DIR = os.path.join(RAW_FRAMES_DIR, 'extracted_frames')

# [4] Post-unpack structure validation (Fix 3)
print("\n[4/10] Validating unpacked folder structure...")

EXPECTED_DIRS = [
    os.path.join(RAW_FRAMES_DIR, 'real'),
    os.path.join(RAW_FRAMES_DIR, 'fake'),
    os.path.join(RAW_FRAMES_DIR, 'fake', 'deepfakes'),
    os.path.join(RAW_FRAMES_DIR, 'fake', 'face2face'),
    os.path.join(RAW_FRAMES_DIR, 'fake', 'faceswap'),
    os.path.join(RAW_FRAMES_DIR, 'fake', 'neuraltextures'),
]

missing = [d for d in EXPECTED_DIRS if not os.path.isdir(d)]
if missing:
    raise RuntimeError(
        "Post-unpack structure validation failed. Missing directories:\n" +
        "\n".join(f"  {d}" for d in missing)
    )
print("  Structure validation PASSED.")

# [5] Scan frame paths per category
print("\n[5/10] Scanning frame paths per category...")

category_frames = {}

real_root  = os.path.join(RAW_FRAMES_DIR, 'real')
real_paths = glob.glob(os.path.join(real_root, '**', '*.jpg'), recursive=True)
if not real_paths:
    raise RuntimeError(f"No real frames found under {real_root}")
category_frames['real'] = real_paths
print(f"  real              : {len(real_paths):,} frames")

fake_root = os.path.join(RAW_FRAMES_DIR, 'fake')
for cat_key in FAKE_CATS:
    cat_dir = os.path.join(fake_root, cat_key)
    paths   = glob.glob(os.path.join(cat_dir, '**', '*.jpg'), recursive=True)
    if not paths:
        raise RuntimeError(f"No frames found under {cat_dir}")
    category_frames[cat_key] = paths
    print(f"  {cat_key:<20}: {len(paths):,} frames")

# [6] Build identity graph + assign splits (Fix 6)
print("\n[6/10] Building FF++ identity graph and assigning splits...")

all_fake_stems = set()
for cat_key in FAKE_CATS:
    for fp in category_frames[cat_key]:
        all_fake_stems.add(parse_source_video_stem(fp))

adjacency, all_nodes, unique_edges = build_identity_graph(all_fake_stems)
components                          = compute_connected_components(adjacency, all_nodes)

graph_stats = {
    'nodes'            : len(all_nodes),
    'unique_edges'     : len(unique_edges),
    'components'       : len(components),
    'largest_component': max((len(c) for c in components), default=0),
}

print(f"  Graph nodes              : {graph_stats['nodes']:,}")
print(f"  Unique graph edges       : {graph_stats['unique_edges']:,}")
print(f"  Connected components     : {graph_stats['components']:,}")
print(f"  Largest component size   : {graph_stats['largest_component']:,}")

node_to_split, split_identity_counts = assign_components_to_splits(components)

print(f"  Split identity allocation:")
for split in SPLIT_ORDER:
    print(f"    {split:<6}: {split_identity_counts.get(split, 0):,} identities")

# [7] Run RetinaFace extraction
print("\n[7/10] Running RetinaFace extraction engine...")
print(f"  Thresholds: confidence>={CONFIDENCE_THRESHOLD}, blur>={BLUR_THRESHOLD}, "
      f"min_face={MIN_FACE_SIZE}px, pHash Hamming<={PHASH_HAMMING_MAX}\n")

accepted_counts   = {k: 0 for k in target_quotas}
rejection_reasons = defaultdict(int)

# Per quota-bucket BK trees (Fix 8): one tree per (split x cat_suffix)
def _hash_distance(h1, h2):
    return h1 - h2

bk_trees = {k: pybktree.BKTree(_hash_distance) for k in target_quotas}

total_target = sum(target_quotas.values())
pbar = tqdm(total=total_target, desc="Securing quota", unit="face")

with open(CSV_LOG_PATH, mode='w', newline='') as log_file:
    csv_writer = csv.writer(log_file)
    csv_writer.writerow([
        "input_path", "source_video_stem", "split", "category",
        "status", "reason", "confidence", "blur_score", "phash", "bounding_box"
    ])

    for cat_key, cat_suffix in INPUT_CATEGORY_MAP.items():
        if quotas_all_filled(accepted_counts):
            break

        is_fake = (cat_key != 'real')
        print(f"  Processing: {cat_key}  ({cat_suffix})")

        process_category(
            cat_key           = cat_key,
            cat_suffix        = cat_suffix,
            frame_paths       = category_frames[cat_key],
            node_to_split     = node_to_split,
            is_fake           = is_fake,
            accepted_counts   = accepted_counts,
            rejection_reasons = rejection_reasons,
            csv_writer        = csv_writer,
            bk_trees          = bk_trees,
            pbar              = pbar,
        )

        cat_accepted = sum(accepted_counts[quota_key(s, cat_suffix)] for s in SPLIT_ORDER)
        cat_target   = sum(target_quotas[quota_key(s, cat_suffix)] for s in SPLIT_ORDER)
        print(f"  {cat_key}: {cat_accepted:,} / {cat_target:,} accepted")

pbar.close()

# [7b] Extraction summary
print("\n" + "-" * 60)
print("  EXTRACTION SUMMARY")
print("-" * 60)

total_accepted = sum(accepted_counts.values())
total_rejected = sum(rejection_reasons.values())
print(f"  Total accepted : {total_accepted:,} / {total_target:,}")
print(f"  Total rejected : {total_rejected:,}")
print("\n  Per-quota results:")

all_filled  = True
underfilled = []
for k, target in sorted(target_quotas.items()):
    got  = accepted_counts.get(k, 0)
    flag = ""
    if got < target:
        all_filled = False
        underfilled.append((k, got, target))
        flag = "  *** UNDERFILLED ***"
    print(f"    {k:<40}: {got:>6,} / {target:>6,}{flag}")

# Fix 2: Fail hard before archive/upload if any quota is underfilled
if not all_filled:
    detail = "\n".join(
        f"  {k}: {got:,} / {target:,}  (short by {target - got:,})"
        for k, got, target in underfilled
    )
    raise RuntimeError(
        f"\nQUOTA FAILURE — one or more quota buckets are underfilled.\n"
        f"Aborting before archive/upload. Local artifacts preserved for inspection.\n\n"
        f"Underfilled buckets:\n{detail}\n\n"
        f"Recommendation: increase frame buffer in the upstream extraction stage "
        f"(more frames per video), or verify all source videos were present in the input zip."
    )

print("\n  All quotas filled. Proceeding to archive and upload.")

# [8] Archive final faces
print("\n[8/10] Creating final faces archive...")
shutil.make_archive(FINAL_ZIP_BASE, 'zip', FINAL_FACES_DIR)
final_zip_path = FINAL_ZIP_BASE + '.zip'

if not os.path.exists(final_zip_path) or os.path.getsize(final_zip_path) == 0:
    raise RuntimeError(f"Final zip creation failed or empty: {final_zip_path}")

zip_size_gb = os.path.getsize(final_zip_path) / (1024 ** 3)
print(f"  Archive ready: {zip_size_gb:.2f} GB  ->  {final_zip_path}")

# [9] Write manifest + upload to S3
print("\n[9/10] Writing manifest...")
write_manifest(
    accepted_counts,
    rejection_reasons,
    final_zip_path,
    start_time,
    graph_stats,
    split_identity_counts,
)

print("\n[9b/10] Uploading outputs to S3...")
upload_targets = [
    (final_zip_path, S3_OUT_KEYS['zip']),
    (MANIFEST_PATH,  S3_OUT_KEYS['manifest']),
    (CSV_LOG_PATH,   S3_OUT_KEYS['csv']),
]

for local_path, s3_key in upload_targets:
    upload_to_s3(local_path, s3_key)

print("\n  Verifying uploads...")
for local_path, s3_key in upload_targets:
    verify_s3_upload(local_path, s3_key)
print("  All uploads verified.")

# [10] Clean local artifacts (only after verified upload)
print("\n[10/10] Cleaning local artifacts...")
shutil.rmtree(FINAL_FACES_DIR, ignore_errors=True)
shutil.rmtree(RAW_FRAMES_DIR,  ignore_errors=True)
if os.path.exists(final_zip_path):
    os.remove(final_zip_path)

print(f"  Removed: {FINAL_FACES_DIR}")
print(f"  Removed: {RAW_FRAMES_DIR}")
print(f"  Removed: {final_zip_path}")
print(f"  Logs preserved at: {LOGS_DIR}")

# ──Done──────────────────────────────────────────────────────────────────────
elapsed = time.time() - start_time
print("\n" + "=" * 60)
print("  FF++ RetinaFace Face Extractor — COMPLETE")
print(f"  Total accepted  : {total_accepted:,} faces")
print(f"  Elapsed         : {elapsed:.0f} seconds  ({elapsed/60:.1f} min)")
print(f"  Output zip      : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}")
print(f"  Manifest        : s3://{S3_BUCKET}/{S3_OUT_KEYS['manifest']}")
print(f"  CSV log         : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}")
print("=" * 60 + "\n")

Zipping and uploading to S3 manually due to Hard Failure from an Underfilled Bucket.

In [ ]:
print("Bypassing quota lock and packaging 19,961 secured faces...")

# ── [8] Archive final faces ───────────────────────────────────────────────────
shutil.make_archive(FINAL_ZIP_BASE, 'zip', FINAL_FACES_DIR)
final_zip_path = FINAL_ZIP_BASE + '.zip'
zip_size_gb = os.path.getsize(final_zip_path) / (1024 ** 3)
print(f"Archive ready: {zip_size_gb:.2f} GB  ->  {final_zip_path}")

# ── [9] Write manifest + upload to S3 ────────────────────────────────────────
write_manifest(
    accepted_counts,
    rejection_reasons,
    final_zip_path,
    start_time,
    graph_stats,
    split_identity_counts,
)

print("Uploading outputs to S3...")
upload_targets = [
    (final_zip_path, S3_OUT_KEYS['zip']),
    (MANIFEST_PATH,  S3_OUT_KEYS['manifest']),
    (CSV_LOG_PATH,   S3_OUT_KEYS['csv']),
]

for local_path, s3_key in upload_targets:
    upload_to_s3(local_path, s3_key)
    verify_s3_upload(local_path, s3_key)

# ── [10] Clean local artifacts ───────────────────────────────────────────────
shutil.rmtree(FINAL_FACES_DIR, ignore_errors=True)
shutil.rmtree(RAW_FRAMES_DIR,  ignore_errors=True)
if os.path.exists(final_zip_path):
    os.remove(final_zip_path)

print("\nRescue Complete. 19,961 faces successfully secured in S3.")

DF40 FF++ Domain EFS Data Download.

Google Drive API Setup.

Downloader.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — DF40 EFS FF++ Dataset Acquisition (Google Drive, Final)
# ══════════════════════════════════════════════════════════════════════════════
#
# Source: Public Google Drive folder
#   https://drive.google.com/drive/folders/1U8meBbqVvmUkc5GD0jxct6xe6Gwk9wKD
#
# Pipeline:
#   1.  Dependency preflight (gdown, unzip)
#   2.  EBS free-space preflight
#   3.  Enumerate Google Drive folder via API; filter to zip files only
#   4.  Normalise zip filenames; match to the 10 EFS method targets
#   5.  [Inspection mode] Print match table and stop if SOURCE_INSPECTION_ONLY=True
#   6.  Download matched zip files via gdown with retry + size-based resume
#   7.  Extract each zip via native unzip
#   8.  Locate FF / ff domain image content inside each extracted tree
#   9.  Sort eligible images; derive method-specific seed; sample 3,000
#  10.  Copy sampled images preserving relative subfolder structure (no collisions)
#  11.  Verify per-method count (3,000) and total count (30,000)
#  12.  Create one master zip of the full 30K dataset
#  13.  Upload master zip to S3
#  14.  Verify S3 upload (head_object size check)
#  15.  Delete raw zips, extracted folders, sampled folder, local master zip
#  16.  Preserve manifests, summary logs, and failure logs only
#
# No face detection, RetinaFace, cropping, resizing, or preprocessing.
# ══════════════════════════════════════════════════════════════════════════════

import csv
import hashlib
import json
import os
import random
import re
import shutil
import subprocess
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import boto3
import requests
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# --- Inspection mode (default: True) -----------------------------------------
# Run once with True to verify the 10 EFS zip files are matched correctly.
# Switch to False only after confirming the match table looks right.
SOURCE_INSPECTION_ONLY = False

# --- Google Drive source -----------------------------------------------------
GDRIVE_FOLDER_ID  = "1U8meBbqVvmUkc5GD0jxct6xe6Gwk9wKD"
GDRIVE_FOLDER_URL = f"https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}"

# Google Drive API key (required by default).
# Obtain a free key at https://console.cloud.google.com
#   -> APIs & Services -> Credentials -> Create API key
#   -> Enable the Google Drive API
# Then: export GDRIVE_API_KEY="your_key_here"
GDRIVE_API_KEY_ENV = "GDRIVE_API_KEY"

# Set True ONLY if you accept fragile gdown-internal / HTML scraping fallback.
# When False (default), missing GDRIVE_API_KEY raises an immediate RuntimeError.
ALLOW_FRAGILE_GDRIVE_FALLBACK = False

# --- Dataset targets ---------------------------------------------------------
RANDOM_SEED           = 42
EXPECTED_METHOD_COUNT = 10
IMAGES_PER_METHOD     = 3_000
TOTAL_TARGET          = EXPECTED_METHOD_COUNT * IMAGES_PER_METHOD   # 30,000

# --- Local SageMaker EBS paths -----------------------------------------------
LOCAL_BASE_DIR  = "/home/ec2-user/SageMaker/data"
ZIPS_DIR        = os.path.join(LOCAL_BASE_DIR, "df40_gdrive_zips")
EXTRACTED_DIR   = os.path.join(LOCAL_BASE_DIR, "df40_gdrive_extracted")
SAMPLED_DIR     = os.path.join(LOCAL_BASE_DIR, "df40_efs_ffpp_30k")
MANIFEST_DIR    = os.path.join(LOCAL_BASE_DIR, "df40_manifests")
MASTER_ZIP_DIR  = os.path.join(LOCAL_BASE_DIR, "df40_zips")

MASTER_ZIP_NAME  = "df40_efs_ffpp_30k_master.zip"
MASTER_ZIP_BASE  = os.path.join(MASTER_ZIP_DIR, os.path.splitext(MASTER_ZIP_NAME)[0])
MASTER_ZIP_FINAL = MASTER_ZIP_BASE + ".zip"

SUMMARY_JSON_PATH = os.path.join(MANIFEST_DIR, "df40_efs_ffpp_30k_summary.json")
SUMMARY_CSV_PATH  = os.path.join(MANIFEST_DIR, "df40_efs_ffpp_30k_summary.csv")

# --- S3 destination ----------------------------------------------------------
S3_BUCKET     = "deepfake-d-100k-dataset-tw26"
S3_KEY_PREFIX = "assets"
S3_MASTER_KEY = f"{S3_KEY_PREFIX}/{MASTER_ZIP_NAME}"

# --- Download / copy settings ------------------------------------------------
MAX_RETRIES      = 3
RETRY_BACKOFF    = 2.0    # seconds; multiplied by attempt number on each retry
NUM_COPY_WORKERS = 8      # parallel threads for the sampled-image copy step

# Disk safety multiplier: raw zips + extracted + sampled copy + master zip + buffer
DISK_SAFETY_FACTOR = 4.0

# Supported image file extensions
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

# FF++ domain folder name tokens (lowercase, non-alphanum stripped)
FF_DOMAIN_TOKENS = {
    "ff", "ffpp", "faceforensics",
    "vqgan", "stylegan2", "stylegan3", "styleganxl",
    "sd21", "sd2", "ddpm", "ddim", "rddm", 
    "pixart", "pixartalpha", "dit", "sit"
}

# ── 10 EFS target methods and their normalised name variants ──────────────────
# Normalisation: lowercase, strip all non-alphanumeric characters.
# Each value is the set of exact normalised tokens that may appear in a zip filename.
EFS_METHOD_VARIANTS = {
    "VQGAN"        : {"vqgan"},
    "StyleGAN2"    : {"stylegan2"},
    "StyleGAN3"    : {"stylegan3"},
    "StyleGAN-XL"  : {"styleganxl", "styleganxl2", "styleganxl3"},
    "SD-2.1"       : {"sd21", "stablediffusion21", "stablediffusion2", "sd2", "sdv21"},
    "DDPM"         : {"ddpm", "ddim"},
    "RDDM"         : {"rddm"},
    "PixArt-alpha" : {"pixartalpha", "pixarta", "pixart"},
    "DiT-XL2"      : {"ditxl2", "ditxl", "dit"},
    "SiT-XL2"      : {"sitxl2", "sitxl", "sit"},
}

# ══════════════════════════════════════════════════════════════════════════════
# DETERMINISM
# ══════════════════════════════════════════════════════════════════════════════

random.seed(RANDOM_SEED)


def derive_method_seed(method_name, base_seed):
    """
    Stable integer seed unique to (method_name, base_seed) via SHA-256.
    Guarantees fully independent, reproducible shuffles per method.
    """
    digest = hashlib.sha256(f"{method_name}:{base_seed}".encode()).hexdigest()
    return int(digest[:16], 16)

# ══════════════════════════════════════════════════════════════════════════════
# DEPENDENCY PREFLIGHT
# ══════════════════════════════════════════════════════════════════════════════

def check_dependencies():
    """
    Verify gdown is importable and unzip exists on the system.
    Fails immediately with actionable instructions if either is missing.
    """
    # gdown
    try:
        import gdown  # noqa: F401
    except ImportError:
        raise RuntimeError(
            "gdown is not installed.\n"
            "Install it with:  pip install gdown\n"
            "Then restart the kernel and re-run this cell."
        )

    # unzip
    result = subprocess.run(["which", "unzip"], capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            "unzip is not available on this system.\n"
            "Install it with:  sudo apt-get install -y unzip\n"
            "Then re-run this cell."
        )

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — FILESYSTEM
# ══════════════════════════════════════════════════════════════════════════════

def make_dirs():
    for d in [ZIPS_DIR, EXTRACTED_DIR, SAMPLED_DIR, MANIFEST_DIR, MASTER_ZIP_DIR]:
        os.makedirs(d, exist_ok=True)


def method_extracted_dir(method_name):
    return os.path.join(EXTRACTED_DIR, method_name)


def method_sampled_dir(method_name):
    return os.path.join(SAMPLED_DIR, method_name)

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — EBS DISK PREFLIGHT
# ══════════════════════════════════════════════════════════════════════════════

def check_disk_space(estimated_bytes_per_image=200_000):
    """
    Abort early if available EBS space is insufficient.
    200 KB/image is a conservative default; adjust for larger source images.
    """
    raw_estimate = TOTAL_TARGET * estimated_bytes_per_image
    required     = int(raw_estimate * DISK_SAFETY_FACTOR)
    _, _, free   = shutil.disk_usage(LOCAL_BASE_DIR)

    free_gb     = free     / (1024 ** 3)
    required_gb = required / (1024 ** 3)
    raw_gb      = raw_estimate / (1024 ** 3)

    print(f"  Disk preflight:")
    print(f"    Estimated raw dataset          : {raw_gb:.2f} GB")
    print(f"    Required with x{DISK_SAFETY_FACTOR} safety factor : {required_gb:.2f} GB")
    print(f"    Available EBS free space        : {free_gb:.2f} GB")

    if free < required:
        raise RuntimeError(
            f"Insufficient EBS disk space.\n"
            f"  Required : {required_gb:.2f} GB\n"
            f"  Available: {free_gb:.2f} GB\n"
            f"Free up space or reduce DISK_SAFETY_FACTOR before proceeding."
        )
    print(f"    Disk space check PASSED.")

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — S3
# ══════════════════════════════════════════════════════════════════════════════

_s3 = boto3.client("s3")


def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  -> s3://{S3_BUCKET}/{s3_key}")
    _s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = _s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
    except Exception as exc:
        raise RuntimeError(f"S3 head_object failed for {s3_key}: {exc}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size:,} bytes\n"
            f"  S3    : {s3_size:,} bytes"
        )
    print(f"  Verified: {os.path.basename(local_path)}  ({s3_size / (1024**3):.2f} GB)")

# ══════════════════════════════════════════════════════════════════════════════
# GOOGLE DRIVE CONNECTOR
# ══════════════════════════════════════════════════════════════════════════════

class GoogleDriveConnector:
    """
    Enumerates files in a public Google Drive folder via the Drive API v3
    and downloads selected files via gdown.

    Enumeration requires GDRIVE_API_KEY (free, no OAuth for public folders).
    Download uses gdown to handle Drive's large-file virus-scan bypass.
    """

    DRIVE_API_BASE = "https://www.googleapis.com/drive/v3/files"

    def __init__(self, folder_id):
        self.folder_id = folder_id
        self.api_key   = os.environ.get(GDRIVE_API_KEY_ENV, "").strip()
        self.session   = requests.Session()

        if not self.api_key:
            if not ALLOW_FRAGILE_GDRIVE_FALLBACK:
                raise RuntimeError(
                    f"Environment variable {GDRIVE_API_KEY_ENV} is not set.\n\n"
                    f"Google Drive API enumeration requires a free API key.\n"
                    f"Steps:\n"
                    f"  1. Go to https://console.cloud.google.com\n"
                    f"  2. Create a project -> APIs & Services -> Enable Google Drive API\n"
                    f"  3. Credentials -> Create API key (no OAuth needed for public folders)\n"
                    f"  4. In your terminal: export {GDRIVE_API_KEY_ENV}='your_key_here'\n"
                    f"  5. Restart the kernel and re-run this cell.\n\n"
                    f"If you intentionally want to use the fragile gdown/HTML fallback,\n"
                    f"set ALLOW_FRAGILE_GDRIVE_FALLBACK = True in the config block.\n"
                    f"This is NOT recommended for production use."
                )
            else:
                print(
                    f"  WARNING: {GDRIVE_API_KEY_ENV} not set and "
                    f"ALLOW_FRAGILE_GDRIVE_FALLBACK=True.\n"
                    f"  Falling back to fragile gdown-internal enumeration."
                )

    # ── Enumeration ────────────────────────────────────────────────────────────

    def list_folder_files(self):
        """
        Return a list of dicts: {name, id, mimeType, size}.
        Primary path: Drive API v3 with GDRIVE_API_KEY.
        Fallback (only when ALLOW_FRAGILE_GDRIVE_FALLBACK=True): gdown internals.
        """
        if self.api_key:
            return self._list_via_api()

        # Fragile fallback — only reached if ALLOW_FRAGILE_GDRIVE_FALLBACK=True
        return self._list_via_gdown_fallback()

    def _list_via_api(self):
        """Paginated Drive API v3 files.list for the folder."""
        files      = []
        page_token = None
        query      = f"'{self.folder_id}' in parents and trashed = false"
        fields     = "nextPageToken,files(id,name,mimeType,size)"

        while True:
            params = {
                "q"       : query,
                "fields"  : fields,
                "key"     : self.api_key,
                "pageSize": 1000,
            }
            if page_token:
                params["pageToken"] = page_token

            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    resp = self.session.get(
                        self.DRIVE_API_BASE, params=params, timeout=30
                    )
                    resp.raise_for_status()
                    data = resp.json()
                    break
                except Exception as exc:
                    if attempt == MAX_RETRIES:
                        raise RuntimeError(
                            f"Drive API enumeration failed after {MAX_RETRIES} attempts: {exc}"
                        )
                    time.sleep(RETRY_BACKOFF * attempt)

            for item in data.get("files", []):
                files.append({
                    "name"    : item.get("name", ""),
                    "id"      : item.get("id", ""),
                    "mimeType": item.get("mimeType", ""),
                    "size"    : int(item.get("size", 0)),
                })

            page_token = data.get("nextPageToken")
            if not page_token:
                break

        return files

    def _list_via_gdown_fallback(self):
        """
        Fragile fallback using gdown internals. Only used when
        ALLOW_FRAGILE_GDRIVE_FALLBACK=True. May break if gdown internals change.
        """
        try:
            from gdown.download_folder import _get_directory_structure
            gdrive_url = f"https://drive.google.com/drive/folders/{self.folder_id}"
            _, _, raw_files = _get_directory_structure(
                gdrive_url, quiet=True, use_cookies=False
            )
            return [
                {
                    "name"    : f.get("title", f.get("name", "")),
                    "id"      : f.get("id", ""),
                    "mimeType": f.get("mimeType", ""),
                    "size"    : int(f.get("fileSize", 0)),
                }
                for f in raw_files
            ]
        except Exception as exc:
            raise RuntimeError(
                f"gdown-internal folder enumeration failed: {exc}\n"
                f"Set {GDRIVE_API_KEY_ENV} and use the Drive API path instead."
            )

    # ── File metadata ──────────────────────────────────────────────────────────

    def get_file_size(self, file_id):
        """Return remote file size in bytes via API. Returns 0 if unavailable."""
        if not self.api_key:
            return 0
        try:
            params = {"fields": "size", "key": self.api_key}
            resp   = self.session.get(
                f"{self.DRIVE_API_BASE}/{file_id}", params=params, timeout=15
            )
            resp.raise_for_status()
            return int(resp.json().get("size", 0))
        except Exception:
            return 0

    # ── Download ───────────────────────────────────────────────────────────────

    def download_zip(self, file_id, file_name, dest_path):
        """
        Download a Drive file via gdown (handles large-file virus-scan bypass).
        Resume: checks remote size against local size; redownloads on mismatch.
        Returns True on success, False after all retries are exhausted.
        """
        if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
            remote_size = self.get_file_size(file_id)
            if remote_size == 0 or os.path.getsize(dest_path) == remote_size:
                # Size matches (or cannot verify) — treat as complete
                return True
            print(
                f"    Local zip size mismatch for {file_name} "
                f"({os.path.getsize(dest_path):,} vs {remote_size:,} bytes) — redownloading."
            )
            os.remove(dest_path)

        import gdown
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        url = f"https://drive.google.com/uc?id={file_id}"

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                gdown.download(
                    url        = url,
                    output     = dest_path,
                    quiet      = False,
                    fuzzy      = True,
                    use_cookies= False,
                )
                if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
                    return True
                raise RuntimeError("gdown produced empty or missing output file.")
            except Exception as exc:
                print(f"    Download attempt {attempt}/{MAX_RETRIES} failed: {exc}")
                if os.path.exists(dest_path):
                    os.remove(dest_path)
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_BACKOFF * attempt)

        return False

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — METHOD NAME NORMALISATION AND MATCHING
# ══════════════════════════════════════════════════════════════════════════════

def normalise_name(name):
    """
    Strip extension, lowercase, remove all non-alphanumeric characters.
    'PixArt-alpha.zip' -> 'pixartalpha'
    'SD-2.1.zip'       -> 'sd21'
    'StyleGAN-XL.zip'  -> 'styleganxl'
    """
    stem = Path(name).stem
    return re.sub(r"[^a-z0-9]", "", stem.lower())


def match_filename_to_method(filename):
    """
    Return canonical method name if the normalised filename matches any
    EFS_METHOD_VARIANTS entry. Returns None if no match.
    Exact match on normalised tokens prevents StyleGAN2 matching StyleGAN3.
    """
    norm = normalise_name(filename)
    for canonical, variants in EFS_METHOD_VARIANTS.items():
        if norm in variants:
            return canonical
    return None


def build_match_table(zip_files):
    """
    Match a list of zip file records to the 10 EFS method targets.
    Returns:
      matched   : dict canonical_method -> drive file record
      unmatched : list of records with no EFS match
    """
    matched   = {}
    unmatched = []

    for rec in zip_files:
        canonical = match_filename_to_method(rec["name"])
        if canonical is not None:
            if canonical not in matched:
                matched[canonical] = rec
            else:
                unmatched.append({**rec, "_note": f"duplicate match for {canonical}"})
        else:
            unmatched.append(rec)

    return matched, unmatched

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — EXTRACTION AND FF++ CONTENT DETECTION
# ══════════════════════════════════════════════════════════════════════════════

def extract_zip(zip_path, extract_to):
    """Extract a zip file using Python's built-in zipfile (handles >4GB files)."""
    import zipfile
    os.makedirs(extract_to, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
    except Exception as exc:
        raise RuntimeError(f"Python extraction failed for {os.path.basename(zip_path)}: {exc}")


def find_ff_image_dirs(extracted_root):
    """
    Walk the extracted tree and return directories that contain FF++ domain images.

    Matching logic:
    - A directory is accepted if its normalised name token is in FF_DOMAIN_TOKENS.
    - If no explicit FF++ subdirectory exists, check for images directly in
      the extracted root (flat zip structure).
    - CDF and other non-FF++ domains are never accepted.

    Returns a list of absolute directory paths.
    """
    ff_dirs   = []
    root_imgs = []

    for dirpath, dirnames, filenames in os.walk(extracted_root):
        dir_norm = re.sub(r"[^a-z0-9]", "", os.path.basename(dirpath).lower())

        if dir_norm in FF_DOMAIN_TOKENS:
            ff_dirs.append(dirpath)
            dirnames[:] = []   # do not recurse further into this subtree
            continue

        if dirpath == extracted_root:
            root_imgs = [f for f in filenames if Path(f).suffix.lower() in IMAGE_EXTENSIONS]

    # Flat structure fallback: no ff/ subdir but images sit directly in root
    if not ff_dirs and root_imgs:
        ff_dirs.append(extracted_root)

    # Remove any path that is a sub-path of another (keep deepest unique roots)
    ff_dirs = sorted(set(ff_dirs))
    filtered = []
    for d in ff_dirs:
        if not any(d.startswith(other + os.sep) for other in ff_dirs if other != d):
            filtered.append(d)

    return filtered


def collect_eligible_images(ff_dirs):
    """
    Recursively collect all image files under the FF++ directories.
    Skips subdirectories that match non-FF++ domain patterns (cdf, celeb, etc.).
    Returns sorted list of absolute paths for deterministic base order.
    """
    EXCLUDE_RE = re.compile(r"(cdf|celeb|c23|c40)", re.IGNORECASE)
    images     = []

    for root_dir in ff_dirs:
        for dirpath, dirnames, filenames in os.walk(root_dir):
            dirnames[:] = [d for d in dirnames if not EXCLUDE_RE.search(d)]
            for fname in filenames:
                if Path(fname).suffix.lower() in IMAGE_EXTENSIONS:
                    images.append(os.path.join(dirpath, fname))

    return sorted(images)

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — SAMPLING AND COPY (collision-safe)
# ══════════════════════════════════════════════════════════════════════════════

def sample_and_copy(method_name, eligible_images, dest_dir, extract_root):
    """
    Deterministically sample IMAGES_PER_METHOD images and copy them to dest_dir
    while preserving relative subfolder structure to avoid basename collisions.

    Output layout:
      dest_dir/<relative_path_from_extract_root>/<filename>

    This guarantees uniqueness even when two FF++ subdirs contain identically
    named files (e.g. 000001.jpg from different categories/folders).

    Returns (sampled_paths, copy_failures).
    """
    method_seed = derive_method_seed(method_name, RANDOM_SEED)
    rng         = random.Random(method_seed)
    pool        = list(eligible_images)   # already sorted
    rng.shuffle(pool)
    sampled     = pool[:IMAGES_PER_METHOD]

    os.makedirs(dest_dir, exist_ok=True)
    copy_failures = []

    def _copy_one(src_path):
        # Preserve relative path so filenames from different subdirs never collide
        try:
            rel      = os.path.relpath(src_path, extract_root)
        except ValueError:
            rel      = os.path.basename(src_path)
        dest_path = os.path.join(dest_dir, rel)
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)

        # Skip if already copied and size matches
        if (os.path.exists(dest_path)
                and os.path.getsize(dest_path) == os.path.getsize(src_path)):
            return dest_path, True

        try:
            shutil.copy2(src_path, dest_path)
            return dest_path, True
        except Exception:
            return src_path, False

    with ThreadPoolExecutor(max_workers=NUM_COPY_WORKERS) as pool_exec:
        futures = {pool_exec.submit(_copy_one, p): p for p in sampled}
        with tqdm(
            total=len(sampled),
            desc=f"  Copying {method_name}",
            unit="img",
            leave=False,
        ) as pbar:
            for future in as_completed(futures):
                path, ok = future.result()
                pbar.update(1)
                if not ok:
                    copy_failures.append(path)

    return sampled, copy_failures

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — MANIFESTS AND SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

def save_method_manifest(method_name, sampled_paths, extract_root):
    manifest_path = os.path.join(MANIFEST_DIR, f"{method_name}_manifest.txt")
    with open(manifest_path, "w") as f:
        f.write(f"# DF40 EFS FF++ — method: {method_name}\n")
        f.write(f"# Random seed   : {RANDOM_SEED}\n")
        f.write(f"# Derived seed  : {derive_method_seed(method_name, RANDOM_SEED)}\n")
        f.write(f"# Sampled files : {len(sampled_paths)}\n")
        f.write(f"# Generated     : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}\n")
        for p in sampled_paths:
            try:
                rel = os.path.relpath(p, extract_root)
            except ValueError:
                rel = os.path.basename(p)
            f.write(rel + "\n")
    return manifest_path


def save_failure_log(method_name, failures):
    if not failures:
        return
    log_path = os.path.join(MANIFEST_DIR, f"{method_name}_failures.txt")
    with open(log_path, "w") as f:
        f.write(f"# Failures for method: {method_name}\n")
        f.write(f"# {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}\n")
        for entry in failures:
            f.write(str(entry) + "\n")


def save_summary(records):
    with open(SUMMARY_JSON_PATH, "w") as f:
        json.dump(records, f, indent=2)
    if records:
        with open(SUMMARY_CSV_PATH, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
            writer.writeheader()
            writer.writerows(records)

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

def count_files_recursive(directory):
    """Count all files recursively under directory."""
    total = 0
    for _, _, files in os.walk(directory):
        total += len(files)
    return total


def validate_sampled_counts(summary_records):
    """
    Confirm exactly IMAGES_PER_METHOD files exist per method and TOTAL_TARGET
    overall. Uses recursive count to match the relative-path copy structure.
    """
    failures = []
    for rec in summary_records:
        method  = rec["method_name"]
        out_dir = method_sampled_dir(method)
        on_disk = count_files_recursive(out_dir) if os.path.isdir(out_dir) else 0
        if on_disk != IMAGES_PER_METHOD:
            failures.append(
                f"  {method}: expected {IMAGES_PER_METHOD}, found {on_disk}"
            )
        rec["disk_file_count"]     = on_disk
        rec["verification_status"] = "PASS" if on_disk == IMAGES_PER_METHOD else "FAIL"

    if failures:
        raise RuntimeError(
            "Per-method sampled count validation failed:\n" + "\n".join(failures)
        )

    total = sum(rec["disk_file_count"] for rec in summary_records)
    if total != TOTAL_TARGET:
        raise RuntimeError(
            f"Total sampled count mismatch.\n"
            f"  Expected : {TOTAL_TARGET:,}\n"
            f"  On disk  : {total:,}"
        )

# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

run_start = time.time()

print("\n" + "=" * 60)
print("  CELL 1 — DF40 EFS FF++ Acquisition (Google Drive, Final)")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
if SOURCE_INSPECTION_ONLY:
    print("  MODE: SOURCE INSPECTION ONLY — no downloads will occur")
else:
    print(f"  Target: {TOTAL_TARGET:,} images  ({EXPECTED_METHOD_COUNT} methods x {IMAGES_PER_METHOD:,})")
print(f"  Source: {GDRIVE_FOLDER_URL}")
print("=" * 60 + "\n")

# ── [1] Dependency preflight ──────────────────────────────────────────────────
print("[1] Dependency preflight...")
check_dependencies()
print("  gdown: OK")
print("  unzip: OK")

# ── [2] Create local directories ──────────────────────────────────────────────
print("\n[2] Creating local directories...")
make_dirs()

# ── [3] EBS disk preflight (skip in inspection mode) ──────────────────────────
if not SOURCE_INSPECTION_ONLY:
    print("\n[3] EBS free-space preflight check...")
    check_disk_space()

# ── [4] Enumerate Google Drive folder ─────────────────────────────────────────
print("\n[4] Connecting to Google Drive and enumerating folder...")
connector   = GoogleDriveConnector(GDRIVE_FOLDER_ID)
all_files   = connector.list_folder_files()

print(f"\n  Total files/items discovered in Drive folder: {len(all_files)}")

# Filter to zip files only before any method matching
zip_files  = [
    rec for rec in all_files
    if rec["name"].lower().endswith(".zip")
    and rec.get("mimeType", "") != "application/vnd.google-apps.folder"
]
non_zips   = [rec for rec in all_files if rec not in zip_files]

print(f"\n  Zip files ({len(zip_files)}):")
for rec in zip_files:
    size_mb = rec["size"] / (1024 ** 2) if rec["size"] else 0
    print(f"    {rec['name']:<50}  {size_mb:>8.1f} MB  id={rec['id'][:16]}...")

if non_zips:
    print(f"\n  Non-zip items ignored ({len(non_zips)}):")
    for rec in non_zips:
        print(f"    {rec['name']}")

# ── [5] Match zip files to the 10 EFS method targets ─────────────────────────
print(f"\n[5] Matching zip files to EFS method targets...")
matched, unmatched = build_match_table(zip_files)

print(f"\n  Matched ({len(matched)} / {EXPECTED_METHOD_COUNT}):")
for canonical, rec in sorted(matched.items()):
    print(f"    {canonical:<16}  <-  {rec['name']}")

if unmatched:
    print(f"\n  Unmatched zip files ({len(unmatched)}):")
    for rec in unmatched:
        note = rec.get("_note", "no EFS match")
        print(f"    {rec['name']:<50}  ({note})")

missing_methods = [m for m in EFS_METHOD_VARIANTS if m not in matched]
if missing_methods:
    print(f"\n  EFS methods NOT matched to any Drive zip:")
    for m in missing_methods:
        print(f"    {m}")

# ── Inspection mode: stop here ─────────────────────────────────────────────────
if SOURCE_INSPECTION_ONLY:
    print("\n" + "=" * 60)
    print("  SOURCE INSPECTION COMPLETE")
    print(f"  Matched  : {len(matched)} / {EXPECTED_METHOD_COUNT} EFS methods")
    print(f"  Unmatched: {len(unmatched)} zip files")
    if missing_methods:
        print(f"  Missing  : {missing_methods}")
    print()
    print("  If the match table looks correct, set SOURCE_INSPECTION_ONLY = False")
    print("  in the config block and re-run to begin downloading.")
    print("=" * 60 + "\n")
    raise SystemExit(0)

# ── Fail hard if any of the 10 EFS methods is missing ─────────────────────────
if len(matched) != EXPECTED_METHOD_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_METHOD_COUNT} matched EFS methods, found {len(matched)}.\n"
        f"Missing: {missing_methods}\n"
        f"Check the Drive folder contents or update EFS_METHOD_VARIANTS."
    )

# ── [6-10] Per-method: download, extract, locate, sample, copy ────────────────
print(f"\n[6-10] Processing {EXPECTED_METHOD_COUNT} EFS methods...\n")

summary_records = []
overall_pbar    = tqdm(total=TOTAL_TARGET, desc="Overall sampled", unit="img")

for canonical_method, drive_rec in sorted(matched.items()):
    print(f"\n  {'='*54}")
    print(f"  Method : {canonical_method}")
    print(f"  Zip    : {drive_rec['name']}")
    print(f"  {'='*54}")

    zip_dest     = os.path.join(ZIPS_DIR, drive_rec["name"])
    extract_root = method_extracted_dir(canonical_method)
    sampled_dest = method_sampled_dir(canonical_method)
    rec_failures = []

    # [6] Download zip
    print(f"  [6] Downloading zip...")
    ok = connector.download_zip(drive_rec["id"], drive_rec["name"], zip_dest)
    if not ok:
        raise RuntimeError(
            f"Failed to download zip for '{canonical_method}' after {MAX_RETRIES} attempts. Aborting."
        )
    zip_mb = os.path.getsize(zip_dest) / (1024 ** 2)
    print(f"  Downloaded: {os.path.basename(zip_dest)}  ({zip_mb:.1f} MB)")

    # [7] Extract
    print(f"  [7] Extracting zip...")
    shutil.rmtree(extract_root, ignore_errors=True)
    extract_zip(zip_dest, extract_root)
    print(f"  Extracted to: {extract_root}")

    # [8] Locate FF++ image directories
    ff_dirs = find_ff_image_dirs(extract_root)
    if not ff_dirs:
        raise RuntimeError(
            f"No FF++ domain image directory found inside '{canonical_method}' zip.\n"
            f"Extracted root: {extract_root}\n"
            f"Expected a subdirectory with a name matching: {FF_DOMAIN_TOKENS}\n"
            f"Check the zip structure manually."
        )
    print(f"  FF++ dirs ({len(ff_dirs)}): "
          f"{[os.path.relpath(d, extract_root) for d in ff_dirs]}")

    # [9] Collect eligible images
    eligible = collect_eligible_images(ff_dirs)
    print(f"  Eligible FF++ images: {len(eligible):,}")

    if len(eligible) < IMAGES_PER_METHOD:
        raise RuntimeError(
            f"'{canonical_method}' has only {len(eligible):,} eligible FF++ images; "
            f"{IMAGES_PER_METHOD:,} required. Aborting."
        )

    # [10] Sample and copy (collision-safe relative paths)
    print(f"  [10] Sampling {IMAGES_PER_METHOD:,} and copying...")
    sampled_paths, copy_failures = sample_and_copy(
        canonical_method, eligible, sampled_dest, extract_root
    )

    if copy_failures:
        rec_failures.extend(copy_failures)
        print(f"  WARNING: {len(copy_failures)} copy failures.")

    save_failure_log(canonical_method, rec_failures)

    manifest_path = save_method_manifest(canonical_method, sampled_paths, extract_root)
    print(f"  Manifest -> {manifest_path}")

    overall_pbar.update(len(sampled_paths))

    summary_records.append({
        "method_name"         : canonical_method,
        "drive_filename"      : drive_rec["name"],
        "drive_file_id"       : drive_rec["id"],
        "eligible_image_count": len(eligible),
        "sampled_count"       : len(sampled_paths),
        "copy_failures"       : len(copy_failures),
        "disk_file_count"     : None,
        "verification_status" : None,
    })

    # Persist incremental summary so partial progress survives a crash
    save_summary(summary_records)

overall_pbar.close()

# ── [11] Validate sampled counts ──────────────────────────────────────────────
print("\n[11] Validating sampled file counts...")
validate_sampled_counts(summary_records)
print(f"  Validation PASSED: {TOTAL_TARGET:,} files confirmed.")

save_summary(summary_records)
print(f"  Summary JSON -> {SUMMARY_JSON_PATH}")
print(f"  Summary CSV  -> {SUMMARY_CSV_PATH}")

# ── [12] Create master zip ────────────────────────────────────────────────────
print(f"\n[12] Creating master zip of all {TOTAL_TARGET:,} sampled images...")
shutil.make_archive(MASTER_ZIP_BASE, "zip", SAMPLED_DIR)

if not os.path.exists(MASTER_ZIP_FINAL) or os.path.getsize(MASTER_ZIP_FINAL) == 0:
    raise RuntimeError(f"Master zip creation failed or empty: {MASTER_ZIP_FINAL}")

zip_gb = os.path.getsize(MASTER_ZIP_FINAL) / (1024 ** 3)
print(f"  Master zip ready: {zip_gb:.2f} GB  ->  {MASTER_ZIP_FINAL}")

# ── [13] Upload to S3 ─────────────────────────────────────────────────────────
print(f"\n[13] Uploading master zip to S3...")
upload_to_s3(MASTER_ZIP_FINAL, S3_MASTER_KEY)
verify_s3_upload(MASTER_ZIP_FINAL, S3_MASTER_KEY)

# ── [14] Cleanup (only after verified S3 upload) ──────────────────────────────
print("\n[14] Cleaning local raw files...")
shutil.rmtree(ZIPS_DIR,      ignore_errors=True)
shutil.rmtree(EXTRACTED_DIR, ignore_errors=True)
shutil.rmtree(SAMPLED_DIR,   ignore_errors=True)
if os.path.exists(MASTER_ZIP_FINAL):
    os.remove(MASTER_ZIP_FINAL)

print(f"  Removed: {ZIPS_DIR}")
print(f"  Removed: {EXTRACTED_DIR}")
print(f"  Removed: {SAMPLED_DIR}")
print(f"  Removed: {MASTER_ZIP_FINAL}")
print(f"  Preserved: {MANIFEST_DIR}")

# ── Final report ───────────────────────────────────────────────────────────────
elapsed = time.time() - run_start
print("\n" + "=" * 60)
print("  CELL 1 COMPLETE — DF40 EFS FF++ Acquisition")
print("=" * 60)
print(f"  Elapsed        : {elapsed:.0f} s  ({elapsed/60:.1f} min)")
print(f"  Total images   : {TOTAL_TARGET:,}")
print(f"  S3             : s3://{S3_BUCKET}/{S3_MASTER_KEY}")
print(f"  Manifests      : {MANIFEST_DIR}")
print()
print(f"  {'Method':<16}  {'Eligible':>9}  {'Sampled':>7}  {'Copied':>7}  Status")
print(f"  {'-'*16}  {'-'*9}  {'-'*7}  {'-'*7}  ------")
for rec in summary_records:
    copied = rec["sampled_count"] - rec["copy_failures"]
    print(
        f"  {rec['method_name']:<16}  "
        f"{rec['eligible_image_count']:>9,}  "
        f"{rec['sampled_count']:>7,}  "
        f"{copied:>7,}  "
        f"{rec['verification_status']}"
    )
print("=" * 60 + "\n")

Second download for Stable Diffusion 2.1 alone.

API and Gdown Setup.

Downloader.

In [ ]:
import os
import random
import shutil
import hashlib
import re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import gdown
import zipfile
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
TARGET_METHOD       = "SD-2.1"
TARGET_SAMPLE_COUNT = 8000
RANDOM_SEED         = 42
GDRIVE_FOLDER_ID    = "1U8meBbqVvmUkc5GD0jxct6xe6Gwk9wKD"
IMAGE_EXTS          = {".jpg", ".jpeg", ".png", ".webp"}

# Safe zone outside of Cell 2's blast radius
PATCH_ZONE_DIR = "/home/ec2-user/SageMaker/patch_zone"
os.makedirs(PATCH_ZONE_DIR, exist_ok=True)

# Temporary workspace for this script
TEMP_ZIP     = f"/home/ec2-user/SageMaker/{TARGET_METHOD}_temp.zip"
TEMP_EXTRACT = f"/home/ec2-user/SageMaker/{TARGET_METHOD}_temp_extract"
DEST_DIR     = os.path.join(PATCH_ZONE_DIR, TARGET_METHOD)

# --- 1. GET FILE ID ---
print(f"=== Targeted Patch Downloader: {TARGET_METHOD} ({TARGET_SAMPLE_COUNT} images) ===")
api_key = os.environ.get("GDRIVE_API_KEY")
if not api_key:
    raise RuntimeError("GDRIVE_API_KEY not found. Run your setup cell first.")

print("\n[1/6] Searching Google Drive for target zip...")
url = "https://www.googleapis.com/drive/v3/files"
params = {
    "q": f"'{GDRIVE_FOLDER_ID}' in parents and trashed = false",
    "fields": "files(id,name)",
    "key": api_key,
    "pageSize": 1000
}
resp = requests.get(url, params=params)
resp.raise_for_status()

file_id = None
target_normalized = TARGET_METHOD.lower().replace("-", "").replace(".", "")
for f in resp.json().get("files", []):
    if target_normalized in f["name"].lower().replace("-", "").replace(".", ""):
        file_id = f["id"]
        print(f"  Found remote file : {f['name']}")
        break

if not file_id:
    raise RuntimeError(f"Could not find a zip matching {TARGET_METHOD} in Drive folder.")

# --- 2. DOWNLOAD ---
print("\n[2/6] Downloading zip...")
gdown.download(f"https://drive.google.com/uc?id={file_id}", TEMP_ZIP, quiet=False)

# --- 3. EXTRACT ---
print("\n[3/6] Extracting zip to temporary storage...")
shutil.rmtree(TEMP_EXTRACT, ignore_errors=True)
os.makedirs(TEMP_EXTRACT, exist_ok=True)
with zipfile.ZipFile(TEMP_ZIP, 'r') as zip_ref:
    zip_ref.extractall(TEMP_EXTRACT)

# --- 4. FILTER AND COLLECT ---
print("\n[4/6] Scanning for eligible images (filtering non-FF++ domains)...")
EXCLUDE_RE = re.compile(r"(cdf|celeb|c23|c40)", re.IGNORECASE)
eligible_images = []

for root, dirs, files in os.walk(TEMP_EXTRACT):
    # Skip garbage directories
    dirs[:] = [d for d in dirs if not EXCLUDE_RE.search(d)]
    for f in files:
        if Path(f).suffix.lower() in IMAGE_EXTS:
            eligible_images.append(os.path.join(root, f))

eligible_images = sorted(eligible_images)
print(f"  Found {len(eligible_images):,} total eligible images.")

if len(eligible_images) < TARGET_SAMPLE_COUNT:
    print(f"  [WARNING] Only found {len(eligible_images):,} images. Sampling all of them.")
    TARGET_SAMPLE_COUNT = len(eligible_images)

# --- 5. DETERMINISTIC SHUFFLE & SAMPLE ---
print(f"\n[5/6] Shuffling and sampling {TARGET_SAMPLE_COUNT:,} images...")
method_seed = int(hashlib.sha256(f"{TARGET_METHOD}:{RANDOM_SEED}".encode()).hexdigest()[:16], 16)
rng = random.Random(method_seed)
rng.shuffle(eligible_images)
sampled_images = eligible_images[:TARGET_SAMPLE_COUNT]

# --- 6. COPY TO SAFE ZONE & CLEANUP ---
print("\n[6/6] Copying to Safe Zone...")
shutil.rmtree(DEST_DIR, ignore_errors=True)
os.makedirs(DEST_DIR, exist_ok=True)

def copy_img(src_path):
    try:
        rel = os.path.relpath(src_path, TEMP_EXTRACT)
    except ValueError:
        rel = os.path.basename(src_path)
    dest_path = os.path.join(DEST_DIR, rel)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    shutil.copy2(src_path, dest_path)

with ThreadPoolExecutor(max_workers=8) as pool:
    list(tqdm(pool.map(copy_img, sampled_images), total=len(sampled_images), desc="  Copying"))

print("\nCleaning up temporary raw data...")
os.remove(TEMP_ZIP)
shutil.rmtree(TEMP_EXTRACT)

print("\n" + "=" * 60)
print("  PATCH COMPLETE")
print("=" * 60)
print(f"  Target  : {TARGET_METHOD}")
print(f"  Saved   : {TARGET_SAMPLE_COUNT:,} images")
print(f"  Path    : {DEST_DIR}")
print("-------------------------------------------")

RetinaFace Filtering on EFS Data.

In [ ]:
%pip install -q tf-keras

In [ ]:
# CELL 2 — DF40 EFS FF++ RetinaFace Face Selection (SageMaker Production)
# Input  : S3 master zip from Cell 1 (30,000 sampled DF40 EFS FF++ images)
# Output : 22,500 accepted face crops at 260x260, split train/val/test per method
#
# Pipeline:
#   1.  Dependency preflight + wipe/recreate staging dirs
#   2.  Download Cell 1 master zip from S3
#   3.  Extract via native unzip
#   4.  Scan extracted images grouped by method (10 methods expected)
#   5.  Per-method: shuffle deterministically, run RetinaFace, fill quota
#       - largest-face selection
#       - confidence / size / ratio / blur gates
#       - 260x260 crop + resize
#       - per-method pHash dedup (BKTree)
#   6.  Fail hard if any method quota is underfilled
#   7.  Write CSV log + manifest
#   8.  Create one master zip of the final 22.5K accepted faces
#   9.  Upload zip + manifest + CSV to S3
#  10.  Verify uploads (head_object size check)
#  11.  Clean local staging only after verified upload


import os

# Must be set before TensorFlow and RetinaFace are imported
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CUDNN_USE_AUTOTUNE"] = "0"

import csv
import hashlib
import random
import shutil
import subprocess
import time
import warnings
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import boto3
import cv2
import imagehash
import numpy as np
import pybktree
from PIL import Image
from retinaface import RetinaFace
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# CONFIGURATION

RANDOM_SEED = 42

# --- S3 source (Cell 1 output) -----------------------------------------------
S3_BUCKET        = "deepfake-d-100k-dataset-tw26"
S3_INPUT_KEY     = "assets/df40_efs_ffpp_30k_master.zip"
S3_OUTPUT_PREFIX = "assets/df40_efs_ffpp_faces"

# --- Dataset targets ---------------------------------------------------------
EXPECTED_METHODS     = 8
TARGET_PER_METHOD    = 2_500
TOTAL_TARGET         = EXPECTED_METHODS * TARGET_PER_METHOD   # 20,000

# Per-method split (must sum to TARGET_PER_METHOD)
TRAIN_PER_METHOD = 1_750
VAL_PER_METHOD   = 375
TEST_PER_METHOD  = 375
assert TRAIN_PER_METHOD + VAL_PER_METHOD + TEST_PER_METHOD == TARGET_PER_METHOD, \
    "Per-method split values must sum to TARGET_PER_METHOD"

SPLIT_QUOTAS = {
    "train": TRAIN_PER_METHOD,
    "val"  : VAL_PER_METHOD,
    "test" : TEST_PER_METHOD,
}
SPLIT_ORDER = ["train", "val", "test"]

# --- RetinaFace quality thresholds ---------
CONFIDENCE_THRESHOLD = 0.90
MIN_FACE_SIZE        = 40       # pixels, width and height
MIN_FACE_RATIO       = 0.01     # face area / image area
BLUR_THRESHOLD       = 12.0     # Laplacian variance; higher = stricter
PADDING              = 20       # pixels added around detected bounding box
RESIZE_TARGET        = (260, 260)

# --- Perceptual hash deduplication -------------------------------------------
PHASH_HAMMING_MAX = 4           # per-method BKTree; higher = less aggressive dedup

# --- Local SageMaker EBS paths -----------------------------------------------
BASE_DIR         = "/home/ec2-user/SageMaker/df40_face_stage"
RAW_ZIP_DIR      = os.path.join(BASE_DIR, "raw_zip")
RAW_INPUT_DIR    = os.path.join(BASE_DIR, "raw_input")
FINAL_FACES_DIR  = os.path.join(BASE_DIR, "final_faces")
LOGS_DIR         = os.path.join(BASE_DIR, "logs")

LOCAL_ZIP_IN     = os.path.join(RAW_ZIP_DIR, "df40_efs_ffpp_30k_master.zip")
FINAL_ZIP_NAME   = "df40_efs_ffpp_faces_20k.zip"
FINAL_ZIP_BASE   = os.path.join(BASE_DIR, os.path.splitext(FINAL_ZIP_NAME)[0])
FINAL_ZIP_FINAL  = FINAL_ZIP_BASE + ".zip"

CSV_LOG_PATH     = os.path.join(LOGS_DIR, "df40_efs_ffpp_faces_20k_log.csv")
MANIFEST_PATH    = os.path.join(LOGS_DIR, "df40_efs_ffpp_faces_20k_manifest.txt")

S3_OUT_KEYS = {
    "zip"      : f"{S3_OUTPUT_PREFIX}/{FINAL_ZIP_NAME}",
    "manifest" : f"{S3_OUTPUT_PREFIX}/df40_efs_ffpp_faces_20k_manifest.txt",
    "csv"      : f"{S3_OUTPUT_PREFIX}/df40_efs_ffpp_faces_20k_log.csv",
}

# DETERMINISM

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# DEPENDENCY PREFLIGHT

def check_dependencies():
    missing = []
    for mod in ["cv2", "imagehash", "pybktree", "retinaface", "PIL"]:
        try:
            __import__(mod)
        except ImportError:
            missing.append(mod)
    if missing:
        raise RuntimeError(
            f"Missing Python packages: {missing}\n"
            f"Install with: pip install opencv-python imagehash pybktree retina-face Pillow"
        )
    result = subprocess.run(["which", "unzip"], capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            "unzip not found. Install with: sudo apt-get install -y unzip"
        )
    print("  All dependencies verified.")

# HELPERS — FILESYSTEM

def wipe_and_create_dirs():
    """Wipe all staging dirs and recreate clean to prevent cross-run contamination."""
    for d in [RAW_ZIP_DIR, RAW_INPUT_DIR, FINAL_FACES_DIR, LOGS_DIR]:
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)
    # Pre-create all split/method output dirs
    for split in SPLIT_ORDER:
        os.makedirs(os.path.join(FINAL_FACES_DIR, split), exist_ok=True)
    print(f"  Staging directories wiped and recreated under: {BASE_DIR}")


def make_method_split_dirs(method_name):
    for split in SPLIT_ORDER:
        os.makedirs(os.path.join(FINAL_FACES_DIR, split, method_name), exist_ok=True)

# HELPERS — S3

_s3 = boto3.client("s3")


def download_from_s3(s3_key, local_path):
    size_obj = _s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
    size_gb  = size_obj / (1024 ** 3)
    print(f"  Downloading s3://{S3_BUCKET}/{s3_key}  ({size_gb:.2f} GB) ...")
    _s3.download_file(S3_BUCKET, s3_key, local_path)
    local_size = os.path.getsize(local_path)
    if local_size != size_obj:
        raise RuntimeError(
            f"Download size mismatch.\n  Expected: {size_obj:,}\n  Got: {local_size:,}"
        )
    print(f"  Saved -> {local_path}  ({local_size / (1024**3):.2f} GB)")


def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)"
          f"  -> s3://{S3_BUCKET}/{s3_key}")
    _s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = _s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
    except Exception as exc:
        raise RuntimeError(f"S3 head_object failed for {s3_key}: {exc}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local: {local_size:,}\n  S3: {s3_size:,}"
        )
    print(f"  Verified: {os.path.basename(local_path)}  ({s3_size / (1024**3):.2f} GB)")

# HELPERS — EXTRACTION

def extract_zip(zip_path, extract_to):
    """Extract using Python's zipfile to bypass Linux unzip limits."""
    import zipfile
    os.makedirs(extract_to, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
    except Exception as exc:
        raise RuntimeError(f"Extraction failed for {os.path.basename(zip_path)}: {exc}")

# HELPERS — INPUT SCANNING

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


def scan_images_by_method(root_dir):
    """
    Walk root_dir and group all image files by their immediate top-level
    subfolder name (the method name from Cell 1).

    Returns: dict method_name -> sorted list of absolute image paths.
    """
    method_images = defaultdict(list)

    for entry in sorted(os.scandir(root_dir), key=lambda e: e.name):
        if not entry.is_dir():
            continue
        method_name = entry.name
        for dirpath, _, filenames in os.walk(entry.path):
            for fname in filenames:
                if Path(fname).suffix.lower() in IMAGE_EXTS:
                    method_images[method_name].append(os.path.join(dirpath, fname))

    # Sort each method's list for a deterministic base before shuffling
    for k in method_images:
        method_images[k] = sorted(method_images[k])

    return dict(method_images)

# HELPERS — RETINAFACE CROP ENGINE

def master_crop_engine(
    img_path,
    save_path,
    bk_tree,
    padding=PADDING,
    min_face_size=MIN_FACE_SIZE,
    min_face_ratio=MIN_FACE_RATIO,
    blur_threshold=BLUR_THRESHOLD,
    confidence_threshold=CONFIDENCE_THRESHOLD,
):
    """
    Load -> RetinaFace detect -> largest face -> quality gate ->
    crop+pad -> blur check -> resize 260x260 -> pHash dedup -> save.

    Returns: (passed, reason, confidence, blur, phash_str, bbox_str)
    """
    try:
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            return False, "Corrupted Image", 0.0, 0.0, "", "[]"

        height, width = img_cv.shape[:2]

        faces = RetinaFace.detect_faces(img_cv)
        if not isinstance(faces, dict) or len(faces) == 0:
            return False, "No Face Detected", 0.0, 0.0, "", "[]"

        # Select largest face by bounding-box area
        largest_area = 0
        best_face    = None
        for face in faces.values():
            box = face.get("facial_area")
            if box is None:
                continue
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > largest_area:
                largest_area = area
                best_face    = face

        if best_face is None:
            return False, "No Valid Face Found", 0.0, 0.0, "", "[]"

        box        = best_face["facial_area"]
        str_box    = f"[{box[0]}, {box[1]}, {box[2]}, {box[3]}]"
        confidence = float(best_face["score"])

        if confidence < confidence_threshold:
            return False, "Low Confidence", confidence, 0.0, "", str_box

        face_w = box[2] - box[0]
        face_h = box[3] - box[1]

        if face_w < min_face_size or face_h < min_face_size:
            return False, "Face Too Small", confidence, 0.0, "", str_box

        if (face_w * face_h) / (width * height) < min_face_ratio:
            return False, "Face Ratio Too Small", confidence, 0.0, "", str_box

        x_min = max(0,      int(box[0]) - padding)
        y_min = max(0,      int(box[1]) - padding)
        x_max = min(width,  int(box[2]) + padding)
        y_max = min(height, int(box[3]) + padding)

        cropped_cv  = img_cv[y_min:y_max, x_min:x_max]
        gray_crop   = cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2GRAY)
        blur_val    = float(cv2.Laplacian(gray_crop, cv2.CV_64F).var())

        if blur_val < blur_threshold:
            return False, "Motion Blur", confidence, blur_val, "", str_box

        cropped_pil  = Image.fromarray(cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2RGB))
        standardized = cropped_pil.resize(RESIZE_TARGET, Image.BICUBIC)

        new_hash = imagehash.phash(standardized)
        matches  = bk_tree.find(new_hash, PHASH_HAMMING_MAX)
        if matches:
            return False, f"Duplicate (Hamming<={PHASH_HAMMING_MAX})", confidence, blur_val, str(new_hash), str_box

        bk_tree.add(new_hash)
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        standardized.save(save_path, format="JPEG", quality=95)

        return True, "Accepted", confidence, blur_val, str(new_hash), str_box

    except Exception as exc:
        return False, f"Engine Error: {str(exc)}", 0.0, 0.0, "", "[]"

# HELPERS — SPLIT ASSIGNMENT

def get_split_for_method(accepted_per_split):
    """
    Return the next split that still has room, in train -> val -> test order.
    Returns None if all splits for this method are full.
    """
    for split in SPLIT_ORDER:
        if accepted_per_split[split] < SPLIT_QUOTAS[split]:
            return split
    return None


def all_method_splits_full(accepted_per_split):
    return all(
        accepted_per_split[split] >= SPLIT_QUOTAS[split]
        for split in SPLIT_ORDER
    )

# HELPERS — MANIFEST

def write_manifest(
    accepted_counts,
    rejection_reasons,
    zip_path,
    start_time,
    method_split_counts,
):
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    elapsed   = time.time() - start_time
    zip_size  = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    total_acc = sum(accepted_counts.values())
    total_rej = sum(rejection_reasons.values())

    lines = [
        "DF40 EFS FF++ RetinaFace Face Selector — Manifest",
        "=" * 60,
        f"  Timestamp              : {timestamp}",
        f"  Elapsed                : {elapsed:.0f} s  ({elapsed/60:.1f} min)",
        f"  S3 input               : s3://{S3_BUCKET}/{S3_INPUT_KEY}",
        f"  Local workspace        : {BASE_DIR}",
        "",
        "  Quality thresholds:",
        f"    Confidence           : >= {CONFIDENCE_THRESHOLD}",
        f"    Min face size        : {MIN_FACE_SIZE} px",
        f"    Min face ratio       : {MIN_FACE_RATIO}",
        f"    Blur threshold       : {BLUR_THRESHOLD}",
        f"    Padding              : {PADDING} px",
        f"    pHash Hamming max    : <= {PHASH_HAMMING_MAX}",
        f"    Resize target        : {RESIZE_TARGET[0]}x{RESIZE_TARGET[1]}",
        "",
        "  Per-method split quotas:",
        f"    train : {TRAIN_PER_METHOD}",
        f"    val   : {VAL_PER_METHOD}",
        f"    test  : {TEST_PER_METHOD}",
        "",
        "=" * 60,
        f"  Total accepted         : {total_acc:,} / {TOTAL_TARGET:,}",
        f"  Total rejected         : {total_rej:,}",
        "",
        "  Per-method accepted counts:",
    ]

    for method, count in sorted(accepted_counts.items()):
        flag   = "" if count >= TARGET_PER_METHOD else "  *** UNDERFILLED ***"
        splits = method_split_counts.get(method, {})
        lines.append(
            f"    {method:<20}: {count:>5,}  "
            f"(train={splits.get('train',0)}, "
            f"val={splits.get('val',0)}, "
            f"test={splits.get('test',0)}){flag}"
        )

    lines += ["", "  Rejection reasons:"]
    for reason, count in sorted(rejection_reasons.items(), key=lambda x: -x[1]):
        lines.append(f"    {reason:<45}: {count:>6,}")

    lines += [
        "",
        "=" * 60,
        f"  Final zip size         : {zip_size:.2f} GB",
        "  S3 output destinations:",
        f"    Zip      : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}",
        f"    Manifest : s3://{S3_BUCKET}/{S3_OUT_KEYS['manifest']}",
        f"    CSV log  : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}",
    ]

    with open(MANIFEST_PATH, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"  Manifest written -> {MANIFEST_PATH}")

# HELPERS — VALIDATION

def count_images_recursive(directory):
    total = 0
    for _, _, files in os.walk(directory):
        total += sum(1 for f in files if Path(f).suffix.lower() in IMAGE_EXTS)
    return total


def validate_accepted_counts(accepted_counts):
    failures = []
    for method, count in accepted_counts.items():
        if count < TARGET_PER_METHOD:
            failures.append(
                f"  {method}: accepted {count:,} / {TARGET_PER_METHOD:,} "
                f"(short by {TARGET_PER_METHOD - count:,})"
            )
    if failures:
        raise RuntimeError(
            "Per-method quota UNDERFILLED — aborting before zip/upload.\n"
            "Local artifacts preserved for inspection.\n\n"
            "Underfilled methods:\n" + "\n".join(failures) + "\n\n"
            "Consider: larger input buffer, loosened thresholds, or check source quality."
        )
    total = sum(accepted_counts.values())
    if total != TOTAL_TARGET:
        raise RuntimeError(
            f"Total accepted count mismatch.\n"
            f"  Expected : {TOTAL_TARGET:,}\n"
            f"  Got      : {total:,}"
        )

# MAIN

start_time = time.time()

print("\n" + "=" * 60)
print("  CELL 2 — DF40 EFS FF++ RetinaFace Face Selector")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"  Target: {TOTAL_TARGET:,} accepted faces  "
      f"({EXPECTED_METHODS} methods x {TARGET_PER_METHOD:,})")
print("=" * 60 + "\n")

# ── [1] Preflight + staging ────────────────────────────────────────────────────
print("[1/9] Dependency preflight and staging setup...")
check_dependencies()
wipe_and_create_dirs()

# ── [2] Download Cell 1 master zip from S3 ────────────────────────────────────
print("\n[2/9] Downloading Cell 1 master zip from S3...")
download_from_s3(S3_INPUT_KEY, LOCAL_ZIP_IN)

# ── [3] Extract ───────────────────────────────────────────────────────────────
print("\n[3/9] Extracting input zip...")
extract_zip(LOCAL_ZIP_IN, RAW_INPUT_DIR)
print(f"  Extracted -> {RAW_INPUT_DIR}")
os.remove(LOCAL_ZIP_IN)
print(f"  Input zip removed from local SSD.")

# ── [4] Scan images by method ─────────────────────────────────────────────────

print("\n[4/9] Scanning extracted images by method...")

# --- 1. INJECT THE 8K SD-2.1 PATCH ---
PATCH_DIR = "/home/ec2-user/SageMaker/patch_zone/SD-2.1"
TARGET_DIR = os.path.join(RAW_INPUT_DIR, "SD-2.1")
if os.path.exists(PATCH_DIR):
    print(f"\n[HOT-SWAP] Injecting 8,000 image SD-2.1 patch...")
    if os.path.exists(TARGET_DIR):
        shutil.rmtree(TARGET_DIR)
    shutil.copytree(PATCH_DIR, TARGET_DIR)
    print("  Patch injected successfully.")

# Scan the folders
method_images = scan_images_by_method(RAW_INPUT_DIR)

# --- 2. AMPUTATE DEAD WEIGHT ---
bad_models = ["pixart", "styleganxl", "stylegan-xl"]
keys_to_delete = [k for k in method_images.keys() if any(bad in k.lower() for bad in bad_models)]
for k in keys_to_delete:
    del method_images[k]
    print(f"  [AMPUTATION] Removed {k} from processing queue.")

discovered_methods = sorted(method_images.keys())
print(f"  Methods discovered ({len(discovered_methods)}):")

for m in discovered_methods:
    print(f"    {m:<20}: {len(method_images[m]):,} images")

if len(discovered_methods) != EXPECTED_METHODS:
    raise RuntimeError(
        f"Expected {EXPECTED_METHODS} methods, discovered {len(discovered_methods)}.\n"
        f"Found: {discovered_methods}\n"
        f"Check the Cell 1 zip structure."
    )

# ── [5] RetinaFace per-method processing ─────────────────────────────────────
print(f"\n[5/9] Running RetinaFace face selection...")
print(f"  Thresholds: confidence>={CONFIDENCE_THRESHOLD}, "
      f"blur>={BLUR_THRESHOLD}, min_face={MIN_FACE_SIZE}px, "
      f"pHash Hamming<={PHASH_HAMMING_MAX}\n")

# Per-method counters and BKTrees
accepted_counts     = {m: 0 for m in discovered_methods}
method_split_counts = {m: {s: 0 for s in SPLIT_ORDER} for m in discovered_methods}
rejection_reasons   = defaultdict(int)

def _hash_distance(h1, h2):
    return h1 - h2

overall_pbar = tqdm(total=TOTAL_TARGET, desc="Overall accepted", unit="face")

with open(CSV_LOG_PATH, mode="w", newline="") as log_file:
    csv_writer = csv.writer(log_file)
    csv_writer.writerow([
        "input_path", "method_name", "split",
        "status", "rejection_reason",
        "confidence", "blur_score", "phash", "bounding_box",
    ])

    for method_name in discovered_methods:
        print(f"\n  Method: {method_name}")

        bk_tree            = pybktree.BKTree(_hash_distance)
        accepted_per_split = {s: 0 for s in SPLIT_ORDER}
        method_total       = 0

        # Deterministic shuffle: sort is already done, derive method seed
        method_seed = int(
            hashlib.sha256(f"{method_name}:{RANDOM_SEED}".encode()).hexdigest()[:16], 16
        )
        rng          = random.Random(method_seed)
        candidates   = list(method_images[method_name])
        rng.shuffle(candidates)

        make_method_split_dirs(method_name)

        with tqdm(
            total=TARGET_PER_METHOD,
            desc=f"  {method_name}",
            unit="face",
            leave=False,
        ) as method_pbar:
            for img_path in candidates:
                if all_method_splits_full(accepted_per_split):
                    break

                assigned_split = get_split_for_method(accepted_per_split)
                if assigned_split is None:
                    break

                out_filename = (
                    f"{method_name}_{assigned_split}_"
                    f"{accepted_per_split[assigned_split]:05d}.jpg"
                )
                save_path = os.path.join(
                    FINAL_FACES_DIR, assigned_split, method_name, out_filename
                )

                passed, reason, conf, blur, phash_val, bbox = master_crop_engine(
                    img_path, save_path, bk_tree
                )

                if passed:
                    accepted_per_split[assigned_split]    += 1
                    method_split_counts[method_name][assigned_split] += 1
                    accepted_counts[method_name]          += 1
                    method_total                          += 1
                    method_pbar.update(1)
                    overall_pbar.update(1)
                    csv_writer.writerow([
                        img_path, method_name, assigned_split,
                        "Accepted", "",
                        round(conf, 4), round(blur, 2), phash_val, bbox,
                    ])
                else:
                    rejection_reasons[reason] += 1
                    csv_writer.writerow([
                        img_path, method_name, assigned_split,
                        "Rejected", reason,
                        round(conf, 4), round(blur, 2), phash_val, bbox,
                    ])

        print(f"  {method_name}: accepted {accepted_counts[method_name]:,} / {TARGET_PER_METHOD:,}  "
              f"(train={accepted_per_split['train']}, "
              f"val={accepted_per_split['val']}, "
              f"test={accepted_per_split['test']})")

overall_pbar.close()

# ── [5b] Extraction summary ────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("  EXTRACTION SUMMARY")
print("-" * 60)
total_accepted = sum(accepted_counts.values())
total_rejected = sum(rejection_reasons.values())
print(f"  Total accepted : {total_accepted:,} / {TOTAL_TARGET:,}")
print(f"  Total rejected : {total_rejected:,}")
print("\n  Rejection reasons:")
for reason, count in sorted(rejection_reasons.items(), key=lambda x: -x[1]):
    print(f"    {reason:<45}: {count:>6,}")

# ── [6] Fail hard if any method is underfilled ────────────────────────────────
print("\n[6/9] Validating per-method quotas...")
validate_accepted_counts(accepted_counts)
print(f"  All {EXPECTED_METHODS} method quotas satisfied. Total: {total_accepted:,}")

# ── [7] Write manifest and CSV ────────────────────────────────────────────────
print("\n[7/9] Writing manifest...")
write_manifest(
    accepted_counts,
    rejection_reasons,
    FINAL_ZIP_FINAL,   # path doesn't exist yet; zip size will be 0 — OK before creation.
    start_time,
    method_split_counts,
)

# ── [8] Create master zip ─────────────────────────────────────────────────────
print(f"\n[8/9] Creating final zip of {total_accepted:,} accepted faces...")
shutil.make_archive(FINAL_ZIP_BASE, "zip", FINAL_FACES_DIR)

if not os.path.exists(FINAL_ZIP_FINAL) or os.path.getsize(FINAL_ZIP_FINAL) == 0:
    raise RuntimeError(f"Final zip creation failed or empty: {FINAL_ZIP_FINAL}")

zip_gb = os.path.getsize(FINAL_ZIP_FINAL) / (1024 ** 3)
print(f"  Final zip ready: {zip_gb:.2f} GB  ->  {FINAL_ZIP_FINAL}")

# Re-write manifest now that zip size is known
write_manifest(
    accepted_counts, rejection_reasons, FINAL_ZIP_FINAL, start_time, method_split_counts
)

# ── [9] Upload to S3 ──────────────────────────────────────────────────────────
print("\n[9a/9] Uploading outputs to S3...")
upload_targets = [
    (FINAL_ZIP_FINAL, S3_OUT_KEYS["zip"]),
    (MANIFEST_PATH,   S3_OUT_KEYS["manifest"]),
    (CSV_LOG_PATH,    S3_OUT_KEYS["csv"]),
]
for local_path, s3_key in upload_targets:
    upload_to_s3(local_path, s3_key)

print("\n[9b/9] Verifying uploads...")
for local_path, s3_key in upload_targets:
    verify_s3_upload(local_path, s3_key)
print("  All uploads verified.")

# ── [10] Cleanup (only after verified upload) ─────────────────────────────────
print("\n[10/9] Cleaning local staging...")
if os.path.exists(LOCAL_ZIP_IN):
    os.remove(LOCAL_ZIP_IN)
shutil.rmtree(RAW_ZIP_DIR,     ignore_errors=True)
shutil.rmtree(RAW_INPUT_DIR,   ignore_errors=True)
shutil.rmtree(FINAL_FACES_DIR, ignore_errors=True)
if os.path.exists(FINAL_ZIP_FINAL):
    os.remove(FINAL_ZIP_FINAL)

print(f"  Removed: {LOCAL_ZIP_IN}")
print(f"  Removed: {RAW_ZIP_DIR}")
print(f"  Removed: {RAW_INPUT_DIR}")
print(f"  Removed: {FINAL_FACES_DIR}")
print(f"  Removed: {FINAL_ZIP_FINAL}")
print(f"  Logs preserved: {LOGS_DIR}")

# ── Done ──────────────────────────────────────────────────────────────────────
elapsed = time.time() - start_time
print("\n" + "=" * 60)
print("  CELL 2 COMPLETE — DF40 EFS FF++ Face Selection")
print("=" * 60)
print(f"  Elapsed         : {elapsed:.0f} s  ({elapsed/60:.1f} min)")
print(f"  Total accepted  : {total_accepted:,} faces")
print(f"  Output zip      : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}")
print(f"  Manifest        : s3://{S3_BUCKET}/{S3_OUT_KEYS['manifest']}")
print(f"  CSV log         : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}")
print()
print(f"  {'Method':<20}  {'Accepted':>8}  {'Train':>6}  {'Val':>5}  {'Test':>5}")
print(f"  {'-'*20}  {'-'*8}  {'-'*6}  {'-'*5}  {'-'*5}")
for method in sorted(accepted_counts.keys()):
    sp = method_split_counts[method]
    print(
        f"  {method:<20}  {accepted_counts[method]:>8,}  "
        f"{sp['train']:>6,}  {sp['val']:>5,}  {sp['test']:>5,}"
    )
print("=" * 60 + "\n")

FFHQ Real Data Download + Filtering.

FFHQ.

HF Token Setup.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"]   = "1"
os.environ["TF_CUDNN_USE_AUTOTUNE"] = "0"

import csv
import random
import shutil
import time
import warnings
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import boto3
import cv2
import numpy as np
import tensorflow as tf
from huggingface_hub import snapshot_download
from PIL import Image
from retinaface import RetinaFace
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

cv2.setNumThreads(1)  # avoids CPU thread contention on SageMaker

# CONFIGURATION

HF_REPO_ID       = "marcosv/ffhq-dataset"
HF_ALLOW_PATTERNS = ["Part1/*.png", "Part2/*.png", "Part3/*.png"]

S3_BUCKET        = "deepfake-d-100k-dataset-tw26"
S3_OUTPUT_PREFIX = "assets/dff_faces"

BASE_DIR         = "/home/ec2-user/SageMaker/ffhq_stage"
HF_DOWNLOAD_DIR  = os.path.join(BASE_DIR, "hf_download")
FINAL_FACES_DIR  = os.path.join(BASE_DIR, "final_faces")
LOGS_DIR         = os.path.join(BASE_DIR, "logs")

FINAL_ZIP_NAME   = "ffhq_real_faces_20k.zip"
FINAL_ZIP_BASE   = os.path.join(BASE_DIR, os.path.splitext(FINAL_ZIP_NAME)[0])
FINAL_ZIP_FINAL  = FINAL_ZIP_BASE + ".zip"

CSV_LOG_PATH     = os.path.join(LOGS_DIR, "ffhq_real_faces_20k_log.csv")
MANIFEST_PATH    = os.path.join(LOGS_DIR, "ffhq_real_faces_20k_manifest.txt")

S3_OUT_KEYS = {
    "zip" : f"{S3_OUTPUT_PREFIX}/{FINAL_ZIP_NAME}",
    "csv" : f"{S3_OUTPUT_PREFIX}/ffhq_real_faces_20k_log.csv",
}

CONFIDENCE_THRESHOLD = 0.90
MIN_FACE_SIZE        = 50
BLUR_THRESHOLD       = 12.0
PADDING              = 20
RESIZE_TARGET        = (260, 260)
JPEG_QUALITY         = 95

TOTAL_TARGET  = 20_000
SPLIT_TARGETS = {"train": 14_000, "val": 3_000, "test": 3_000}
SPLIT_ORDER   = ["train", "val", "test"]
assert sum(SPLIT_TARGETS.values()) == TOTAL_TARGET

RANDOM_SEED = 42

# SETUP

def wipe_and_create_dirs():
    for d in [HF_DOWNLOAD_DIR, FINAL_FACES_DIR, LOGS_DIR]:
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)
    for split in SPLIT_ORDER:
        os.makedirs(os.path.join(FINAL_FACES_DIR, split), exist_ok=True)
    print(f"  Staging directories ready under: {BASE_DIR}")

# S3 HELPERS

_s3 = boto3.client("s3")


def upload_to_s3(local_path, s3_key):
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)"
          f"  -> s3://{S3_BUCKET}/{s3_key}")
    _s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = _s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
    except Exception as exc:
        raise RuntimeError(f"S3 head_object failed for {s3_key}: {exc}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local: {local_size:,}\n  S3: {s3_size:,}"
        )
    print(f"  Verified: {os.path.basename(local_path)}  ({s3_size / (1024**3):.2f} GB)")

# RETINAFACE ENGINE

def process_image(img_path):
    """
    Full RetinaFace pipeline for one image.
    Returns: (passed, reason, confidence, blur, bbox_str, cropped_pil)
    """
    try:
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            return False, "Corrupted", 0.0, 0.0, "[]", None

        height, width = img_cv.shape[:2]

        faces = RetinaFace.detect_faces(img_cv)
        if not isinstance(faces, dict) or len(faces) == 0:
            return False, "No Face", 0.0, 0.0, "[]", None

        best_face    = None
        largest_area = 0
        for face in faces.values():
            box = face.get("facial_area")
            if box is None:
                continue
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > largest_area:
                largest_area = area
                best_face    = face

        if best_face is None:
            return False, "No Face", 0.0, 0.0, "[]", None

        box        = best_face["facial_area"]
        str_box    = f"[{box[0]},{box[1]},{box[2]},{box[3]}]"
        confidence = float(best_face["score"])

        if confidence < CONFIDENCE_THRESHOLD:
            return False, "Low Confidence", confidence, 0.0, str_box, None

        face_w = box[2] - box[0]
        face_h = box[3] - box[1]
        if face_w < MIN_FACE_SIZE or face_h < MIN_FACE_SIZE:
            return False, "Small Face", confidence, 0.0, str_box, None

        x_min = max(0,      int(box[0]) - PADDING)
        y_min = max(0,      int(box[1]) - PADDING)
        x_max = min(width,  int(box[2]) + PADDING)
        y_max = min(height, int(box[3]) + PADDING)

        cropped_cv = img_cv[y_min:y_max, x_min:x_max]
        gray       = cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2GRAY)
        blur_val   = float(cv2.Laplacian(gray, cv2.CV_64F).var())

        if blur_val < BLUR_THRESHOLD:
            return False, "Blur", confidence, blur_val, str_box, None

        pil_img      = Image.fromarray(cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2RGB))
        pil_img      = pil_img.convert("RGB")
        standardized = pil_img.resize(RESIZE_TARGET, Image.BICUBIC)

        return True, "Accepted", confidence, blur_val, str_box, standardized

    except Exception as exc:
        return False, f"Error: {str(exc)}", 0.0, 0.0, "[]", None

# SPLIT ASSIGNMENT

def get_next_split(accepted_per_split):
    for split in SPLIT_ORDER:
        if accepted_per_split[split] < SPLIT_TARGETS[split]:
            return split
    return None

# MANIFEST

def write_manifest(accepted_per_split, rejection_counts, zip_path, start_time, total_accepted):
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    elapsed   = time.time() - start_time
    zip_size  = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    total_rej = sum(rejection_counts.values())

    lines = [
        "FFHQ RetinaFace Extraction — Manifest",
        "=" * 60,
        f"  Timestamp              : {timestamp}",
        f"  Elapsed                : {elapsed:.0f} s  ({elapsed/60:.1f} min)",
        f"  HF repo                : {HF_REPO_ID}",
        "",
        "  RetinaFace thresholds:",
        f"    Confidence           : >= {CONFIDENCE_THRESHOLD}",
        f"    Min face size        : {MIN_FACE_SIZE} px",
        f"    Blur threshold       : >= {BLUR_THRESHOLD}",
        f"    Padding              : {PADDING} px",
        f"    Resize target        : {RESIZE_TARGET[0]}x{RESIZE_TARGET[1]}",
        "",
        "=" * 60,
        f"  Total accepted         : {total_accepted:,} / {TOTAL_TARGET:,}",
        f"  Total rejected         : {total_rej:,}",
        "",
        "  Split counts:",
    ]
    for split in SPLIT_ORDER:
        target = SPLIT_TARGETS[split]
        got    = accepted_per_split[split]
        flag   = "" if got == target else "  *** MISMATCH ***"
        lines.append(f"    {split:<6}: {got:>6,} / {target:>6,}{flag}")
    lines += ["", "  Rejection reasons:"]
    for reason, count in sorted(rejection_counts.items(), key=lambda x: -x[1]):
        lines.append(f"    {reason:<40}: {count:>6,}")
    lines += [
        "",
        "=" * 60,
        f"  Final zip size         : {zip_size:.2f} GB",
        "  S3 output destinations:",
        f"    Zip : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}",
        f"    CSV : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}",
    ]
    with open(MANIFEST_PATH, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"  Manifest written -> {MANIFEST_PATH}")

# MAIN

run_start = time.time()

print("\n" + "=" * 60)
print("  FFHQ RetinaFace Face Extraction Pipeline")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"  Target : {TOTAL_TARGET:,} accepted faces")
print(f"  Splits : train={SPLIT_TARGETS['train']:,}  val={SPLIT_TARGETS['val']:,}  test={SPLIT_TARGETS['test']:,}")
print("=" * 60 + "\n")

# [1/13] GPU check + staging
print("[1/13] GPU awareness and staging setup...")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"  GPUs detected: {[g.name for g in gpus]}")
else:
    print("  No GPU detected — running on CPU.")
wipe_and_create_dirs()

# [2/13] Download FFHQ from Hugging Face
print(f"\n[2/13] Downloading FFHQ from Hugging Face Hub ({HF_REPO_ID})...")
print(f"  Patterns: {HF_ALLOW_PATTERNS}")

_DOWNLOAD_MAX_RETRIES = 3
_DOWNLOAD_RETRY_WAIT  = 10  # seconds between retries

for _attempt in range(1, _DOWNLOAD_MAX_RETRIES + 1):
    try:
        snapshot_download(
            repo_id               = HF_REPO_ID,
            repo_type             = "dataset",
            local_dir             = HF_DOWNLOAD_DIR,
            allow_patterns        = HF_ALLOW_PATTERNS,
            local_dir_use_symlinks= False,
            token                 = True,
        )
        print(f"  Download complete -> {HF_DOWNLOAD_DIR}")
        break
    except Exception as _dl_exc:
        if _attempt < _DOWNLOAD_MAX_RETRIES:
            print(f"  Download attempt {_attempt}/{_DOWNLOAD_MAX_RETRIES} failed: {_dl_exc}")
            print(f"  Retrying in {_DOWNLOAD_RETRY_WAIT} s...")
            time.sleep(_DOWNLOAD_RETRY_WAIT)
        else:
            raise RuntimeError(
                f"snapshot_download failed after {_DOWNLOAD_MAX_RETRIES} attempts.\n"
                f"Last error: {_dl_exc}"
            )

# [3/13] Image discovery
print(f"\n[3/13] Discovering images...")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
all_images = []
for dirpath, _, filenames in os.walk(HF_DOWNLOAD_DIR):
    for fname in filenames:
        if Path(fname).suffix.lower() in IMAGE_EXTS:
            all_images.append(os.path.join(dirpath, fname))

all_images = sorted(all_images)
print(f"  Discovered {len(all_images):,} images.")

if not all_images:
    raise RuntimeError(f"No images found under {HF_DOWNLOAD_DIR}.")

# [4/13] Deterministic shuffle
print(f"\n[4/13] Shuffling (seed={RANDOM_SEED})...")
rng = random.Random(RANDOM_SEED)
rng.shuffle(all_images)
print(f"  Shuffle complete.")

# [5/13] RetinaFace processing loop
print(f"\n[5/13] Running RetinaFace extraction...")
print(f"  Thresholds: confidence>={CONFIDENCE_THRESHOLD}, "
      f"min_face={MIN_FACE_SIZE}px, blur>={BLUR_THRESHOLD}")

accepted_per_split = {s: 0 for s in SPLIT_ORDER}
split_index        = {s: 0 for s in SPLIT_ORDER}
rejection_counts   = defaultdict(int)
total_accepted     = 0
total_rejected     = 0

pbar = tqdm(
    total=TOTAL_TARGET,
    desc="Accepted faces",
    unit="face",
    dynamic_ncols=True,
)

with open(CSV_LOG_PATH, mode="w", newline="") as log_file:
    csv_writer = csv.writer(log_file)
    csv_writer.writerow([
        "input_path", "split", "status", "reason",
        "confidence", "blur", "bounding_box",
    ])

    for img_path in all_images:
        if total_accepted >= TOTAL_TARGET:
            break

        assigned_split = get_next_split(accepted_per_split)
        if assigned_split is None:
            break

        passed, reason, conf, blur, bbox, cropped = process_image(img_path)

        if passed and cropped is not None:
            idx       = split_index[assigned_split] + 1
            out_fname = f"real_{assigned_split}_{idx:05d}.jpg"
            out_path  = os.path.join(FINAL_FACES_DIR, assigned_split, out_fname)
            try:
                cropped.save(out_path, format="JPEG", quality=JPEG_QUALITY)
                accepted_per_split[assigned_split] += 1
                split_index[assigned_split]         += 1
                total_accepted                      += 1
                pbar.update(1)
                csv_writer.writerow([
                    img_path, assigned_split, "Accepted", "",
                    round(conf, 4), round(blur, 2), bbox,
                ])
            except Exception as save_exc:
                rejection_counts["Save Error"] += 1
                total_rejected += 1
                csv_writer.writerow([
                    img_path, assigned_split, "Rejected", f"Save Error: {save_exc}",
                    round(conf, 4), round(blur, 2), bbox,
                ])
        else:
            rejection_counts[reason] += 1
            total_rejected += 1
            csv_writer.writerow([
                img_path, "", "Rejected", reason,
                round(conf, 4), round(blur, 2), bbox,
            ])

pbar.close()

print(f"\n  Accepted : {total_accepted:,}")
print(f"  Rejected : {total_rejected:,}")
print(f"  Breakdown:")
for reason, count in sorted(rejection_counts.items(), key=lambda x: -x[1]):
    print(f"    {reason:<40}: {count:>6,}")

# [6/13] JPEG format verification
print(f"\n[6/13] Verifying output image format...")
non_jpg_files = []
actual_count  = 0
for split in SPLIT_ORDER:
    split_dir = os.path.join(FINAL_FACES_DIR, split)
    for fname in os.listdir(split_dir):
        actual_count += 1
        if not fname.lower().endswith(".jpg"):
            non_jpg_files.append(os.path.join(split_dir, fname))

if non_jpg_files:
    print(f"  WARNING: {len(non_jpg_files)} non-JPEG files found in output:")
    for p in non_jpg_files[:10]:
        print(f"    {p}")
else:
    print(f"  All {actual_count:,} output files confirmed as .jpg")

if actual_count != total_accepted:
    print(f"  WARNING: File count on disk ({actual_count:,}) "
          f"differs from accepted counter ({total_accepted:,}).")

# [7/13] Quota check (soft failure per spec)
print(f"\n[7/13] Checking split quotas...")
quota_ok = True
for split in SPLIT_ORDER:
    got    = accepted_per_split[split]
    target = SPLIT_TARGETS[split]
    status = "OK" if got == target else "MISMATCH"
    print(f"  {split:<6}: {got:>6,} / {target:>6,}  [{status}]")
    if got != target:
        quota_ok = False

if total_accepted < TOTAL_TARGET:
    print(f"\n  WARNING: Total accepted {total_accepted:,} < target {TOTAL_TARGET:,}.")
    print(f"  Continuing with zip and upload as per failure policy.")
else:
    print(f"\n  Total accepted {total_accepted:,} meets target.")

# [8/13] Manifest
print(f"\n[8/13] Writing manifest...")
write_manifest(accepted_per_split, rejection_counts, FINAL_ZIP_FINAL, run_start, total_accepted)

# [9/13] Zip final_faces + include CSV
print(f"\n[9/13] Creating final ZIP (including CSV log)...")
with zipfile.ZipFile(FINAL_ZIP_FINAL, "w", zipfile.ZIP_DEFLATED) as zf:
    for split in SPLIT_ORDER:
        split_dir = os.path.join(FINAL_FACES_DIR, split)
        for fname in sorted(os.listdir(split_dir)):
            fpath = os.path.join(split_dir, fname)
            arcname = os.path.join("final_faces", split, fname)
            zf.write(fpath, arcname)
    zf.write(CSV_LOG_PATH, os.path.basename(CSV_LOG_PATH))

if not os.path.exists(FINAL_ZIP_FINAL) or os.path.getsize(FINAL_ZIP_FINAL) == 0:
    raise RuntimeError(f"Final ZIP creation failed or empty: {FINAL_ZIP_FINAL}")

zip_gb = os.path.getsize(FINAL_ZIP_FINAL) / (1024 ** 3)
print(f"  ZIP ready: {zip_gb:.2f} GB  ->  {FINAL_ZIP_FINAL}")

# Re-write manifest with correct zip size
write_manifest(accepted_per_split, rejection_counts, FINAL_ZIP_FINAL, run_start, total_accepted)

# [10/13] Upload to S3
print(f"\n[10/13] Uploading to S3...")
upload_to_s3(FINAL_ZIP_FINAL, S3_OUT_KEYS["zip"])
upload_to_s3(CSV_LOG_PATH,    S3_OUT_KEYS["csv"])

# [11/13] Verify uploads
print(f"\n[11/13] Verifying uploads...")
verify_s3_upload(FINAL_ZIP_FINAL, S3_OUT_KEYS["zip"])
verify_s3_upload(CSV_LOG_PATH,    S3_OUT_KEYS["csv"])
print("  All uploads verified.")

# [12/13] Cleanup
print(f"\n[12/13] Cleaning local staging (post-verified upload)...")
shutil.rmtree(HF_DOWNLOAD_DIR,  ignore_errors=True)
shutil.rmtree(FINAL_FACES_DIR,  ignore_errors=True)
if os.path.exists(FINAL_ZIP_FINAL):
    os.remove(FINAL_ZIP_FINAL)
print(f"  Removed: {HF_DOWNLOAD_DIR}")
print(f"  Removed: {FINAL_FACES_DIR}")
print(f"  Removed: {FINAL_ZIP_FINAL}")
print(f"  Logs preserved: {LOGS_DIR}")

# [13/13] Done
elapsed = time.time() - run_start
print("\n" + "=" * 60)
print("  FFHQ RetinaFace Pipeline COMPLETE")
print("=" * 60)
print(f"  Elapsed         : {elapsed:.0f} s  ({elapsed/60:.1f} min)")
print(f"  Total accepted  : {total_accepted:,}")
print(f"  Train           : {accepted_per_split['train']:,}")
print(f"  Val             : {accepted_per_split['val']:,}")
print(f"  Test            : {accepted_per_split['test']:,}")
print(f"  S3 ZIP          : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}")
print(f"  S3 CSV          : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}")
print("=" * 60 + "\n")

DF40 CDF Domain FS Preprocessing.

Downloader - Google Drive API Setup.

In [ ]:
import getpass

# Securely prompt for the API key without echoing it to the screen or saving it in the notebook output
GDRIVE_API_KEY = getpass.getpass(prompt='Enter your Google Drive API Key: ')

# Confirm it was loaded into the global scope without printing the key itself
if GDRIVE_API_KEY:
    print("[SUCCESS] API Key securely loaded into memory.")
else:
    print("[WARNING] No API key was entered. Cell 1 will fail.")

Cell 1

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — DF40 Celeb-DF v2 Staging, Identity Graph, and Video Download
# ══════════════════════════════════════════════════════════════════════════════
#
# Responsibilities:
#   1.  Environment setup and directory staging (non-destructive, resume-safe)
#   2.  Deterministic identity graph (59 IDs -> train/val/test split)
#   3.  Download archive files from Google Drive via Drive API v3 with retry
#   4.  Extract archives; delete archives immediately after extraction
#   5.  Post-extraction video file verification
#   6.  Download log
#
# Source: DF40 (NeurIPS 2024) — https://github.com/YZY-stack/DF40
#
# Archive structure confirmed from DF40 README:
#   - Celeb-DF real videos are distributed as one archive per the DF40 README
#     "Celeb-DF real data (Google Drive Link)" section.
#   - Fake (FaceSwap) videos are distributed as one or more archives under
#     the "DF40 fake data" section of the same README.
#
# IMPORTANT: paste the actual Google Drive share links from the DF40 GitHub
# README into REAL_ARCHIVE_URLS and FAKE_ARCHIVE_URLS below before running.
# The links are listed under the dataset download table in the README.
#
# Does NOT include: frame extraction, RetinaFace, pHash, S3 upload, zipping.
# ══════════════════════════════════════════════════════════════════════════════

import os
import random
import re
import shutil
import tarfile
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import requests
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

RANDOM_SEED = 42

# GDRIVE_API_KEY is expected in global scope, injected by the Setup Cell
# that runs before this cell. The assertion at the start of the execution
# block below will fail immediately if it is missing.

BASE_DIR       = "/home/ec2-user/SageMaker/df40_cdf_stage"
RAW_REALS_DIR  = os.path.join(BASE_DIR, "raw_reals")
RAW_FAKES_DIR  = os.path.join(BASE_DIR, "raw_fakes")
LOGS_DIR       = os.path.join(BASE_DIR, "logs")

DOWNLOAD_LOG_PATH = os.path.join(LOGS_DIR, "download_log.txt")

TOTAL_IDENTITIES = 59
TRAIN_COUNT      = 41   # 70%
VAL_COUNT        = 9    # 15%
TEST_COUNT       = 9    # 15%
assert TRAIN_COUNT + VAL_COUNT + TEST_COUNT == TOTAL_IDENTITIES

DOWNLOAD_MAX_RETRIES = 3
DOWNLOAD_RETRY_WAIT  = 7   # seconds between retries

VIDEO_EXTENSIONS   = {".mp4", ".avi"}
ARCHIVE_EXTENSIONS = {".zip", ".tar.gz", ".tgz", ".tar"}

# ── Archive download URLs ─────────────────────────────────────────────────────
# Source: DF40 (NeurIPS 2024) — https://github.com/YZY-stack/DF40
#
# REAL: Custom zip of the Celeb-real/ folder only, hosted on personal Google Drive.
# Paste the Google Drive share link here before running.
REAL_ARCHIVE_URLS = [ "https://drive.google.com/file/d/1NAbKGmRBl8R-X6YbRQugpT1GdqwfmdHO/view?usp=sharing"
]

# FAKE: DF40 FaceSwap archives under the CDF domain (17 method archives).
FAKE_ARCHIVE_URLS = [
    "https://drive.google.com/file/d/16ujOKy0zjjtKgDsXchlaro58r5wl0dtA/view?usp=sharing",
    "https://drive.google.com/file/d/1W42kfxU5TRMisBzTNu9ecl-2Pb4zuA1D/view?usp=sharing",
    "https://drive.google.com/file/d/1zqNjNznDEEoYSlkeo7Za6l55JwGxpC2q/view?usp=sharing",
    "https://drive.google.com/file/d/15pRnpg6VBhvwLne3KtzJIkSt3Sez9JHe/view?usp=sharing",
    "https://drive.google.com/file/d/16Dt8xrkeVoJ50GHFM0qza3kPg6QvUOWd/view?usp=sharing",
    "https://drive.google.com/file/d/1t-rkulp4O3ZapkAZUWM9rHz1x02h838T/view?usp=sharing",
    "https://drive.google.com/file/d/1SUFSnjERbCf2se6QZces0a5u3u83gARZ/view?usp=sharing",
    "https://drive.google.com/file/d/1UlomP_3TRLtdL9luQvrA0W_l3I2Wzcx-/view?usp=sharing",
]

# ══════════════════════════════════════════════════════════════════════════════
# [1] DIRECTORY STAGING
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("  CELL 1 — DF40 Celeb-DF v2  |  Staging + Download")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 60 + "\n")

# Task 3: GDRIVE_API_KEY must be set by the Setup Cell before this cell runs.
assert "GDRIVE_API_KEY" in globals(), (
    "ERROR: GDRIVE_API_KEY not found in global scope. "
    "Please run the API setup cell first."
)

print("[1/6] Directory staging...")

for d in [RAW_REALS_DIR, RAW_FAKES_DIR, LOGS_DIR]:
    if os.path.exists(d):
        print(f"  Found existing directory (preserving contents): {d}")
    else:
        os.makedirs(d, exist_ok=True)
        print(f"  Created: {d}")

print(f"\n  Base dir  : {BASE_DIR}")
print(f"  Raw reals : {RAW_REALS_DIR}")
print(f"  Raw fakes : {RAW_FAKES_DIR}")
print(f"  Logs      : {LOGS_DIR}")
print(f"  Directory staging COMPLETE.\n")

# ══════════════════════════════════════════════════════════════════════════════
# [2] DOWNLOAD HELPERS
# ══════════════════════════════════════════════════════════════════════════════

download_failures = []

GDRIVE_API_DOWNLOAD_URL = "https://www.googleapis.com/drive/v3/files/{file_id}?alt=media&key={api_key}"
RESUME_SKIP_THRESHOLD   = 1024   # bytes — files smaller than this are considered incomplete


def extract_file_id(url):
    """
    Extract the Google Drive file ID from a standard share URL.
    Supports formats:
      https://drive.google.com/file/d/<FILE_ID>/view?...
      https://drive.google.com/open?id=<FILE_ID>
    Raises ValueError if no file ID can be parsed.
    """
    match = re.search(r"/file/d/([a-zA-Z0-9_-]+)", url)
    if match:
        return match.group(1)
    match = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", url)
    if match:
        return match.group(1)
    raise ValueError(f"Cannot extract file ID from URL: {url}")


def download_archives(url_list, output_dir, label):
    """
    Download archive files from Google Drive using the Drive API v3 REST endpoint.
    Uses streaming (stream=True, iter_content chunk_size=32768) for multi-gigabyte archives.

    Resume logic:
      - Local filename is set deterministically as archive_<file_id>.zip.
      - If the file already exists and is larger than RESUME_SKIP_THRESHOLD bytes,
        it is considered a valid cached archive and the download is skipped.
      - If the file exists but is smaller than the threshold, it is treated as a
        corrupt partial download and overwritten.

    Retries up to DOWNLOAD_MAX_RETRIES times on failure with DOWNLOAD_RETRY_WAIT delay.
    """
    global download_failures

    if not url_list:
        print(f"  [{label}] No archive URLs configured — skipping download.")
        return []

    if GDRIVE_API_KEY == "YOUR_API_KEY_HERE":
        print(f"  [{label}] WARNING: GDRIVE_API_KEY is not set. Downloads will be skipped.")
        return []

    print(f"  [{label}] Downloading {len(url_list)} archive(s) to {output_dir} ...")
    downloaded_paths = []

    for url in url_list:
        try:
            file_id = extract_file_id(url)
        except ValueError as exc:
            print(f"  [{label}] ERROR parsing URL ({exc}): {url}")
            download_failures.append({"label": label, "url": url, "error": str(exc)})
            continue

        local_filename = f"archive_{file_id}.zip"
        local_path     = os.path.join(output_dir, local_filename)

        # Resume check: skip if a sufficiently large file already exists
        if os.path.exists(local_path) and os.path.getsize(local_path) > RESUME_SKIP_THRESHOLD:
            size_gb = os.path.getsize(local_path) / (1024 ** 3)
            print(f"  [{label}] Found existing archive, skipping download: "
                  f"{local_filename}  ({size_gb:.2f} GB)")
            downloaded_paths.append(local_path)
            continue

        api_url    = GDRIVE_API_DOWNLOAD_URL.format(file_id=file_id, api_key=GDRIVE_API_KEY)
        output_path = None

        for attempt in range(1, DOWNLOAD_MAX_RETRIES + 1):
            try:
                print(f"  [{label}] Downloading {local_filename} (attempt {attempt}/{DOWNLOAD_MAX_RETRIES})...")
                tmp_path = local_path + ".tmp"

                with requests.get(api_url, stream=True, timeout=60) as response:
                    response.raise_for_status()
                    total_bytes = int(response.headers.get("Content-Length", 0))
                    total_gb    = total_bytes / (1024 ** 3) if total_bytes else 0

                    desc = f"    {local_filename} ({total_gb:.2f} GB)" if total_gb else f"    {local_filename}"
                    with open(tmp_path, "wb") as f, tqdm(
                        total=total_bytes if total_bytes else None,
                        unit="B",
                        unit_scale=True,
                        unit_divisor=1024,
                        desc=desc,
                        leave=False,
                    ) as pbar:
                        for chunk in response.iter_content(chunk_size=32768):
                            if chunk:
                                f.write(chunk)
                                pbar.update(len(chunk))

                os.replace(tmp_path, local_path)   # atomic rename

                if os.path.getsize(local_path) > RESUME_SKIP_THRESHOLD:
                    size_gb     = os.path.getsize(local_path) / (1024 ** 3)
                    output_path = local_path
                    print(f"  [{label}] Downloaded: {local_filename}  ({size_gb:.2f} GB)")
                    break
                else:
                    raise RuntimeError(
                        f"Downloaded file is suspiciously small "
                        f"({os.path.getsize(local_path)} bytes). "
                        f"The API key may be invalid or the file quota is exceeded."
                    )

            except Exception as exc:
                if os.path.exists(local_path + ".tmp"):
                    os.remove(local_path + ".tmp")
                if attempt < DOWNLOAD_MAX_RETRIES:
                    print(f"  [{label}] Attempt {attempt}/{DOWNLOAD_MAX_RETRIES} failed "
                          f"({exc}). Retrying in {DOWNLOAD_RETRY_WAIT}s...")
                    time.sleep(DOWNLOAD_RETRY_WAIT)
                else:
                    print(f"  [{label}] FAILED after {DOWNLOAD_MAX_RETRIES} attempts: {url}")
                    download_failures.append({"label": label, "url": url, "error": str(exc)})

        if output_path:
            downloaded_paths.append(output_path)

    print(f"  [{label}] Download complete: {len(downloaded_paths)}/{len(url_list)} archives.\n")
    return downloaded_paths

# ══════════════════════════════════════════════════════════════════════════════
# [4] EXTRACTION HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def extract_archive(archive_path, extract_to, label="FAKE"):
    """
    Extract a .zip or .tar.gz archive to extract_to.
    Both REAL and FAKE archives are extracted unconditionally using standard
    extractall. The custom REAL zip contains only the Celeb-real/ content
    (pre-filtered before upload), so no selective filtering is needed.
    Raises RuntimeError on unsupported or corrupt archives.
    """
    fname_lower = archive_path.lower()

    if fname_lower.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(extract_to)

    elif fname_lower.endswith(".tar.gz") or fname_lower.endswith(".tgz") or fname_lower.endswith(".tar"):
        with tarfile.open(archive_path, "r:*") as tf:
            tf.extractall(extract_to)

    else:
        raise RuntimeError(f"Unsupported archive format: {archive_path}")


def extract_and_nuke(archive_paths, extract_to, label):
    """
    For each downloaded archive:
      1. Extract contents into extract_to using label-aware selective extraction.
      2. Immediately delete the archive (in-place nuke).
    This keeps EBS usage minimal — no dead archive files sitting alongside videos.
    """
    if not archive_paths:
        print(f"  [{label}] No archives to extract.")
        return

    print(f"  [{label}] Extracting {len(archive_paths)} archive(s)...")

    for archive_path in archive_paths:
        basename = os.path.basename(archive_path)
        size_gb  = os.path.getsize(archive_path) / (1024 ** 3)
        print(f"    Extracting: {basename}  ({size_gb:.2f} GB)  -> {extract_to}")

        try:
            extract_archive(archive_path, extract_to, label=label)
            print(f"    Extraction complete: {basename}")
        except Exception as exc:
            print(f"    ERROR extracting {basename}: {exc}")
            download_failures.append({
                "label": label,
                "url"  : archive_path,
                "error": f"Extraction failed: {exc}",
            })
            # Do not delete a failed archive — leave it for inspection
            continue

        # In-place nuke: delete archive immediately after successful extraction
        try:
            os.remove(archive_path)
            print(f"    Deleted archive: {basename}")
        except Exception as exc:
            print(f"    WARNING: Could not delete {basename}: {exc}")

    print(f"  [{label}] Extraction and cleanup complete.\n")

# ══════════════════════════════════════════════════════════════════════════════
# [3] EXECUTE DOWNLOADS
# ══════════════════════════════════════════════════════════════════════════════

print("[3/7] Downloading archives...")

#real_archive_paths = download_archives(REAL_ARCHIVE_URLS, RAW_REALS_DIR, "REAL")
#fake_archive_paths = download_archives(FAKE_ARCHIVE_URLS, RAW_FAKES_DIR, "FAKE")

# ══════════════════════════════════════════════════════════════════════════════
# [4] EXTRACTION AND IN-PLACE ARCHIVE DELETION
# ══════════════════════════════════════════════════════════════════════════════

print("[4/7] Extracting archives and deleting them immediately after extraction...")

#extract_and_nuke(real_archive_paths, RAW_REALS_DIR, "REAL")
#extract_and_nuke(fake_archive_paths, RAW_FAKES_DIR, "FAKE")

# Safety scan: catch any residual archives left from interrupted or external transfers
for _label, _target_dir in [("REAL", RAW_REALS_DIR), ("FAKE", RAW_FAKES_DIR)]:
    residual = [
        f for f in Path(_target_dir).rglob("*")
        if any(str(f).lower().endswith(ext) for ext in ARCHIVE_EXTENSIONS)
    ]
    if residual:
        print(f"  [{_label}] Residual archives found — extracting and deleting:")
        for archive_path in residual:
            archive_path_str = str(archive_path)
            size_gb = os.path.getsize(archive_path_str) / (1024 ** 3)
            print(f"    {archive_path.name}  ({size_gb:.2f} GB)")
            try:
                extract_archive(archive_path_str, _target_dir, label=_label)
                os.remove(archive_path_str)
                print(f"    Extracted and deleted: {archive_path.name}")
            except Exception as exc:
                print(f"    ERROR on residual archive {archive_path.name}: {exc}")

# ══════════════════════════════════════════════════════════════════════════════
# [5] POST-EXTRACTION FILE VERIFICATION
# ══════════════════════════════════════════════════════════════════════════════

print("[5/7] Verifying extracted video files...")


def count_video_files(directory):
    return [
        p for p in Path(directory).rglob("*")
        if p.suffix.lower() in VIDEO_EXTENSIONS
    ]


real_files = count_video_files(RAW_REALS_DIR)
fake_files = count_video_files(RAW_FAKES_DIR)

print(f"\n  Real videos found : {len(real_files):,}")
print(f"  Fake videos found : {len(fake_files):,}")

if len(real_files) == 0:
    raise RuntimeError(
        f"No real video files found under {RAW_REALS_DIR}.\n"
        f"Check REAL_ARCHIVE_URLS and re-run Cell 1."
    )
if len(fake_files) == 0:
    raise RuntimeError(
        f"No fake video files found under {RAW_FAKES_DIR}.\n"
        f"Check FAKE_ARCHIVE_URLS and re-run Cell 1."
    )

print(f"\n  Verification PASSED.\n")

# ══════════════════════════════════════════════════════════════════════════════
# [6] IDENTITY GRAPH (built from actual extracted filenames)
# ══════════════════════════════════════════════════════════════════════════════

print("[6/7] Building identity graph from extracted real video filenames...")

# Parse unique ID prefixes from filenames like id61_0009.mp4.
# This approach is robust against non-contiguous ID numbering (e.g. id14/id15
# being absent in Celeb-DF v2) because it reads what is actually on disk.
_id_pattern = re.compile(r"^(id\d+)_", re.IGNORECASE)
all_ids = sorted({
    _id_pattern.match(p.name).group(1)
    for p in real_files
    if _id_pattern.match(p.name)
})

print(f"  Unique IDs discovered in RAW_REALS_DIR: {len(all_ids)}")
print(f"  Sample IDs: {all_ids[:8]}")

assert len(all_ids) == TOTAL_IDENTITIES, (
    f"Extracted unique IDs ({len(all_ids)}) do not match "
    f"TOTAL_IDENTITIES ({TOTAL_IDENTITIES}). "
    f"Check the archive contents or update TOTAL_IDENTITIES."
)

rng = random.Random(RANDOM_SEED)
shuffled_ids = list(all_ids)
rng.shuffle(shuffled_ids)

train_ids = shuffled_ids[:TRAIN_COUNT]
val_ids   = shuffled_ids[TRAIN_COUNT : TRAIN_COUNT + VAL_COUNT]
test_ids  = shuffled_ids[TRAIN_COUNT + VAL_COUNT:]

assert len(train_ids) == TRAIN_COUNT
assert len(val_ids)   == VAL_COUNT
assert len(test_ids)  == TEST_COUNT

IDENTITY_SPLIT_MAP = {}
for identity in train_ids:
    IDENTITY_SPLIT_MAP[identity] = "train"
for identity in val_ids:
    IDENTITY_SPLIT_MAP[identity] = "val"
for identity in test_ids:
    IDENTITY_SPLIT_MAP[identity] = "test"

SPLIT_TO_IDS = {
    "train": sorted(train_ids),
    "val"  : sorted(val_ids),
    "test" : sorted(test_ids),
}

print(f"\n  Identity split (seed={RANDOM_SEED}, total={TOTAL_IDENTITIES}):")
for split in ["train", "val", "test"]:
    ids     = SPLIT_TO_IDS[split]
    samples = ids[:5]
    print(f"    {split:<6}: {len(ids):>2} IDs  |  sample: {samples}")

print(f"\n  Identity graph COMPLETE.\n")

# ══════════════════════════════════════════════════════════════════════════════
# [6] DOWNLOAD AND EXTRACTION LOG
# ══════════════════════════════════════════════════════════════════════════════

print("[7/7] Writing download log...")

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

log_lines = [
    "DF40 Celeb-DF v2 — Cell 1 Download and Extraction Log",
    "=" * 60,
    f"  Timestamp              : {timestamp}",
    f"  Random seed            : {RANDOM_SEED}",
    f"  Total identities       : {TOTAL_IDENTITIES}",
    f"  Train IDs              : {TRAIN_COUNT}",
    f"  Val IDs                : {VAL_COUNT}",
    f"  Test IDs               : {TEST_COUNT}",
    "",
    "  Identity split sample:",
    f"    train: {SPLIT_TO_IDS['train'][:5]}",
    f"    val  : {SPLIT_TO_IDS['val'][:5]}",
    f"    test : {SPLIT_TO_IDS['test'][:5]}",
    "",
    "=" * 60,
    f"  Real archive URLs      : {len(REAL_ARCHIVE_URLS)}",
    f"  Fake archive URLs      : {len(FAKE_ARCHIVE_URLS)}",
    f"  Real archives downloaded: {len(real_archive_paths)}",
    f"  Fake archives downloaded: {len(fake_archive_paths)}",
    f"  Real video files on disk: {len(real_files):,}",
    f"  Fake video files on disk: {len(fake_files):,}",
    f"  Failures               : {len(download_failures)}",
]

if download_failures:
    log_lines += ["", "  Failures:"]
    for f in download_failures:
        log_lines.append(f"    [{f['label']}] {f['url']}  |  {f['error']}")

log_lines += ["", "=" * 60]

with open(DOWNLOAD_LOG_PATH, "w") as f:
    f.write("\n".join(log_lines) + "\n")

print(f"  Log written -> {DOWNLOAD_LOG_PATH}")

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("  CELL 1 COMPLETE")
print("=" * 60)
print(f"  Real videos : {len(real_files):,}")
print(f"  Fake videos : {len(fake_files):,}")
print(f"  Failures    : {len(download_failures)}")
print(f"  Identity graph:")
for split in ["train", "val", "test"]:
    print(f"    {split:<6}: {len(SPLIT_TO_IDS[split])} IDs")
print(f"  Log: {DOWNLOAD_LOG_PATH}")
print("=" * 60 + "\n")

Cell 2 - Frame Extraction - RetinaFace.

Variable Reboot.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# KERNEL RECOVERY CELL
# Run this cell after a kernel restart, before running Cell 2.
# Reconstructs all Cell 1 variables from the data already on EBS.
# The identity split is deterministic (seed=42) and will be identical
# to what Cell 1 produced.
# ══════════════════════════════════════════════════════════════════════════════

import os
import re
import random
from pathlib import Path

# ── Paths (must match Cell 1) ─────────────────────────────────────────────────
BASE_DIR      = "/home/ec2-user/SageMaker/df40_cdf_stage"
RAW_REALS_DIR = os.path.join(BASE_DIR, "raw_reals")
RAW_FAKES_DIR = os.path.join(BASE_DIR, "raw_fakes")
LOGS_DIR      = os.path.join(BASE_DIR, "logs")

# ── Identity graph settings (must match Cell 1) ───────────────────────────────
RANDOM_SEED      = 42
TOTAL_IDENTITIES = 59
TRAIN_COUNT      = 41
VAL_COUNT        = 9
TEST_COUNT       = 9

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov"}

# ── Validate directories exist ────────────────────────────────────────────────
for label, path in [("RAW_REALS_DIR", RAW_REALS_DIR),
                    ("RAW_FAKES_DIR", RAW_FAKES_DIR),
                    ("LOGS_DIR",      LOGS_DIR)]:
    if not os.path.isdir(path):
        raise RuntimeError(
            f"{label} not found: {path}\n"
            f"Cell 1 has not been run yet, or the EBS volume was remounted.\n"
            f"Run Cell 1 first to download and extract the dataset."
        )

# ── Reconstruct SPLIT_TO_IDS from real video filenames on disk ────────────────
_id_pattern = re.compile(r"^(id\d+)_", re.IGNORECASE)

all_ids = sorted({
    _id_pattern.match(p.name).group(1).lower()
    for p in Path(RAW_REALS_DIR).rglob("*")
    if p.suffix.lower() in VIDEO_EXTENSIONS and _id_pattern.match(p.name)
})

if not all_ids:
    raise RuntimeError(
        f"No real video files found under {RAW_REALS_DIR}.\n"
        f"Run Cell 1 first to download and extract the dataset."
    )

assert len(all_ids) == TOTAL_IDENTITIES, (
    f"Expected {TOTAL_IDENTITIES} unique IDs on disk, found {len(all_ids)}.\n"
    f"Check RAW_REALS_DIR contents or update TOTAL_IDENTITIES above."
)

rng = random.Random(RANDOM_SEED)
shuffled = list(all_ids)
rng.shuffle(shuffled)

train_ids = shuffled[:TRAIN_COUNT]
val_ids   = shuffled[TRAIN_COUNT : TRAIN_COUNT + VAL_COUNT]
test_ids  = shuffled[TRAIN_COUNT + VAL_COUNT:]

SPLIT_TO_IDS = {
    "train": sorted(train_ids),
    "val"  : sorted(val_ids),
    "test" : sorted(test_ids),
}

# ── Summary ───────────────────────────────────────────────────────────────────
print("Kernel recovery complete. Variables restored from disk.\n")
print(f"  BASE_DIR      : {BASE_DIR}")
print(f"  RAW_REALS_DIR : {RAW_REALS_DIR}")
print(f"  RAW_FAKES_DIR : {RAW_FAKES_DIR}")
print(f"  LOGS_DIR      : {LOGS_DIR}")
print(f"\n  SPLIT_TO_IDS (seed={RANDOM_SEED}, total={TOTAL_IDENTITIES}):")
for split in ["train", "val", "test"]:
    ids = SPLIT_TO_IDS[split]
    print(f"    {split:<6}: {len(ids)} IDs  |  sample: {ids[:5]}")
print("\n  Ready to run Cell 2.")

Environment Check.

In [ ]:
import sys

# Dictionary mapping the Python module name to its exact pip install name
required_packages = {
    "cv2": "opencv-python",       # For video and image processing
    "imagehash": "ImageHash",     # For pHash deduplication
    "numpy": "numpy",             # For linspace frame sampling
    "retinaface": "retina-face",  # The facial detection engine
    "PIL": "Pillow",              # For bicubic resizing
    "tqdm": "tqdm",               # For the progress bar
    "tensorflow": "tensorflow"    # Mandatory backend for RetinaFace
}

missing_packages = []

print("🔍 Running Pre-Flight Dependency Check for Cell 2...\n")

for module_name, pip_name in required_packages.items():
    try:
        __import__(module_name)
        print(f" {module_name:<15} : Installed")
    except ImportError:
        print(f" {module_name:<15} : MISSING")
        if pip_name not in missing_packages:
            missing_packages.append(pip_name)

print("\n" + "═"*60)

if not missing_packages:
    print(" ALL SYSTEMS GREEN. You are cleared to run Cell 2.")
else:
    print(" MISSING DEPENDENCIES DETECTED.")
    print("Create a new cell, copy/paste the targeted installer code below, and run it:\n")
    
    # Generate the exact pip command for Jupyter/SageMaker
    install_command = f"!pip install {' '.join(missing_packages)} -q"
    
    print(install_command)
    
print("═"*60 + "\n")

In [3]:
!pip install ImageHash retina-face tensorflow -q

In [1]:
!pip install tf-keras -q

Cell 2.

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Two-Phase In-Memory Video → Face Extraction Engine
# ══════════════════════════════════════════════════════════════════════════════
#
# Phase 1: Extract real faces from RAW_REALS_DIR, record per-split yields.
# Phase 2: Extract fake faces from RAW_FAKES_DIR, stop each split when
#          fake_counts[split] == real_counts[split] (perfect balance).
#
# Upstream variables consumed from Cell 1:
#   BASE_DIR, RAW_REALS_DIR, RAW_FAKES_DIR, LOGS_DIR, SPLIT_TO_IDS
# ══════════════════════════════════════════════════════════════════════════════

import csv
import os
import time
import warnings
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import cv2
import imagehash
import numpy as np
from retinaface import RetinaFace
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

os.environ["TF_USE_LEGACY_KERAS"]   = "1"
os.environ["TF_CUDNN_USE_AUTOTUNE"] = "0"

# ── Upstream variables — defined here if Cell 1 has not been run ──────────────
# If Cell 1 ran first these are already in scope and these lines are skipped.
# If running Cell 2 standalone, fill in the values below.
if "BASE_DIR" not in globals():
    BASE_DIR      = "/home/ec2-user/SageMaker/df40_cdf_stage"
if "RAW_REALS_DIR" not in globals():
    RAW_REALS_DIR = os.path.join(BASE_DIR, "raw_reals")
if "RAW_FAKES_DIR" not in globals():
    RAW_FAKES_DIR = os.path.join(BASE_DIR, "raw_fakes")
if "LOGS_DIR" not in globals():
    LOGS_DIR      = os.path.join(BASE_DIR, "logs")
if "SPLIT_TO_IDS" not in globals():
    # Paste your actual identity lists here when running standalone.
    # These are generated deterministically by Cell 1 (seed=42, 41/9/9 split).
    SPLIT_TO_IDS = {
        "train": [],   # e.g. ["id0", "id2", ...]
        "val"  : [],
        "test" : [],
    }
    if not any(SPLIT_TO_IDS.values()):
        raise RuntimeError(
            "SPLIT_TO_IDS is empty. Either run Cell 1 first, or paste your "
            "identity lists into the SPLIT_TO_IDS dict above."
        )

os.makedirs(LOGS_DIR, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

FINAL_DATA_DIR  = os.path.join(BASE_DIR, "final_data")
CSV_LOG_PATH    = os.path.join(LOGS_DIR, "df40_extraction_log.csv")

SPLITS          = ["train", "val", "test"]
LABELS          = ["real", "fake"]

# Face detection thresholds
CONFIDENCE_THRESHOLD = 0.90
MIN_FACE_SIZE        = 50       # pixels, both width and height
BLUR_THRESHOLD       = 12.0    # Laplacian variance
PADDING              = 20      # pixels added around bounding box
RESIZE_TARGET        = (260, 260)
JPEG_QUALITY         = 95

# pHash deduplication
PHASH_HAMMING_MIN    = 4        # accept only if Hamming distance > this value

# Frame sampling
REAL_FRAMES_PER_VIDEO = 35
FAKE_FRAMES_PER_VIDEO = 7

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov"}

# ══════════════════════════════════════════════════════════════════════════════
# FIX 1 — BK-TREE FOR O(log N) PHASH SPATIAL SEARCH
# ══════════════════════════════════════════════════════════════════════════════

class BKNode:
    """Single node in a BK-Tree storing one imagehash value."""
    __slots__ = ("hash_val", "children")

    def __init__(self, hash_val):
        self.hash_val = hash_val
        self.children = {}   # {distance: BKNode}


class BKTree:
    """
    BK-Tree for O(log N) nearest-neighbour search under Hamming distance.
    Each per-identity hash bank is one BKTree instance, keeping deduplication
    strictly scoped to a single identity.
    """

    def __init__(self):
        self.root = None

    def add(self, hash_val):
        if self.root is None:
            self.root = BKNode(hash_val)
            return
        node = self.root
        while True:
            dist = abs(hash_val - node.hash_val)
            if dist == 0:
                return   # exact duplicate already present
            if dist not in node.children:
                node.children[dist] = BKNode(hash_val)
                return
            node = node.children[dist]

    def is_duplicate(self, hash_val, threshold):
        """
        Returns True if any stored hash has Hamming distance <= threshold.
        Uses BK-Tree pruning to skip entire subtrees outside the search radius.
        """
        if self.root is None:
            return False
        stack = [self.root]
        while stack:
            node = stack.pop()
            dist = abs(hash_val - node.hash_val)
            if dist <= threshold:
                return True
            # Only recurse into children whose distance could be within range
            lo = dist - threshold
            hi = dist + threshold
            for child_dist, child_node in node.children.items():
                if lo <= child_dist <= hi:
                    stack.append(child_node)
        return False


# ══════════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY SETUP
# ══════════════════════════════════════════════════════════════════════════════

for split in SPLITS:
    for label in LABELS:
        os.makedirs(os.path.join(FINAL_DATA_DIR, split, label), exist_ok=True)

print(f"Output directories ready under: {FINAL_DATA_DIR}")

# ══════════════════════════════════════════════════════════════════════════════
# BUILD REVERSE IDENTITY LOOKUP
# ══════════════════════════════════════════════════════════════════════════════

# id_to_split["id16"] == "train"
id_to_split = {}
for split, ids in SPLIT_TO_IDS.items():
    for identity in ids:
        id_to_split[identity] = split

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — IDENTITY PARSING
# ══════════════════════════════════════════════════════════════════════════════

def parse_identity(filename):
    """
    Extract identity prefix from filename.
    'id16_0003.mp4' -> 'id16'
    Returns None if the filename does not start with 'id<digits>'.
    """
    stem = Path(filename).stem
    parts = stem.split("_")
    if parts and parts[0].lower().startswith("id"):
        return parts[0].lower()
    return None

# ══════════════════════════════════════════════════════════════════════════════
# FIX 4 — DETERMINISTIC FRAME SAMPLING VIA NP.LINSPACE
# ══════════════════════════════════════════════════════════════════════════════

def sample_frame_indices(total_frames, n_samples):
    """
    Return n_samples perfectly evenly distributed frame indices in [0, total_frames).
    np.linspace guarantees full coverage of the video including the final frames,
    unlike integer division which leaves the tail unsampled on long videos.
    Handles edge case where total_frames < n_samples by clamping to available frames.
    """
    if total_frames <= 0 or n_samples <= 0:
        return []
    actual_samples = min(n_samples, total_frames)
    return np.linspace(0, total_frames - 1, actual_samples, dtype=int).tolist()

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — FACE EXTRACTION ENGINE (RETINAFACE)
# ══════════════════════════════════════════════════════════════════════════════

def process_frame(frame_bgr):
    """
    Run RetinaFace on one frame.
    Returns (passed, reason, confidence, blur, bbox_str, cropped_pil).
    All quality gates are applied here. Largest face by bounding box area is
    selected when multiple detections are returned.
    """
    try:
        height, width = frame_bgr.shape[:2]

        faces = RetinaFace.detect_faces(frame_bgr)
        if not isinstance(faces, dict) or len(faces) == 0:
            return False, "No Face", 0.0, 0.0, "[]", None

        # Select largest face by bounding box area
        best_face    = None
        largest_area = 0
        for face in faces.values():
            box = face.get("facial_area")
            if box is None:
                continue
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > largest_area:
                largest_area = area
                best_face    = face

        if best_face is None:
            return False, "No Valid Face", 0.0, 0.0, "[]", None

        bbox       = best_face["facial_area"]
        confidence = float(best_face["score"])
        str_box    = f"[{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}]"

        if confidence < CONFIDENCE_THRESHOLD:
            return False, "Low Confidence", confidence, 0.0, str_box, None

        face_w = bbox[2] - bbox[0]
        face_h = bbox[3] - bbox[1]
        if face_w < MIN_FACE_SIZE or face_h < MIN_FACE_SIZE:
            return False, "Small Face", confidence, 0.0, str_box, None

        x_min = max(0,      int(bbox[0]) - PADDING)
        y_min = max(0,      int(bbox[1]) - PADDING)
        x_max = min(width,  int(bbox[2]) + PADDING)
        y_max = min(height, int(bbox[3]) + PADDING)

        cropped_bgr = frame_bgr[y_min:y_max, x_min:x_max]
        gray        = cv2.cvtColor(cropped_bgr, cv2.COLOR_BGR2GRAY)
        blur_val    = float(cv2.Laplacian(gray, cv2.CV_64F).var())

        if blur_val < BLUR_THRESHOLD:
            return False, "Blur", confidence, blur_val, str_box, None

        cropped_rgb  = cv2.cvtColor(cropped_bgr, cv2.COLOR_BGR2RGB)
        pil_img      = Image.fromarray(cropped_rgb).convert("RGB")
        standardized = pil_img.resize(RESIZE_TARGET, Image.BICUBIC)

        return True, "Accepted", confidence, blur_val, str_box, standardized

    except Exception as exc:
        return False, f"Error: {str(exc)}", 0.0, 0.0, "[]", None

# ══════════════════════════════════════════════════════════════════════════════
# COUNTERS AND STATE
# ══════════════════════════════════════════════════════════════════════════════

real_counts   = {s: 0 for s in SPLITS}
fake_counts   = {s: 0 for s in SPLITS}
total_rejected = 0

# Per-identity pHash banks — each value is a BKTree for O(log N) dedup search
real_hash_bank = defaultdict(BKTree)   # {identity: BKTree}
fake_hash_bank = defaultdict(BKTree)

# Sequential per-split-label file indices for clean naming
file_index = {s: {l: 0 for l in LABELS} for s in SPLITS}

# ══════════════════════════════════════════════════════════════════════════════
# CSV LOG SETUP
# ══════════════════════════════════════════════════════════════════════════════

log_file   = open(CSV_LOG_PATH, mode="w", newline="", encoding="utf-8")
csv_writer = csv.writer(log_file)
csv_writer.writerow([
    "video_path", "frame_index", "split", "label",
    "status", "reason", "confidence", "blur", "bbox",
])

# ══════════════════════════════════════════════════════════════════════════════
# HELPER — PROCESS ONE VIDEO
# ══════════════════════════════════════════════════════════════════════════════

def process_video(video_path, label, hash_bank, counts,
                  frames_per_video, split_override=None, stop_counts=None):
    """
    Open video ONCE, buffer all sampled frames into memory, release the
    capture object, then run RetinaFace inference over the buffer.
    All CSV rows are accumulated in log_buffer and flushed in one writerows
    call at the end to minimise disk I/O overhead.

    stop_counts: dict {split: int} — stop adding to a split when
                 counts[split] >= stop_counts[split]. Pass None for Phase 1.
    split_override: force all frames to a specific split regardless of identity.
    """
    global total_rejected

    log_buffer = []   # accumulate all CSV rows for this video; flushed at end

    identity = parse_identity(Path(video_path).name)
    if identity is None:
        total_rejected += 1
        log_buffer.append([video_path, -1, "", label,
                           "Rejected", "Bad Filename", 0.0, 0.0, "[]"])
        csv_writer.writerows(log_buffer)
        return

    split = split_override or id_to_split.get(identity)
    if split is None:
        return

     # Open VideoCapture exactly once per video
    try:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            return
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    except Exception:
        return

    indices = sample_frame_indices(total_frames, frames_per_video)

    # --- PASS 1: read all sampled frames into memory, then release the cap ---
    frame_buffer = []   # list of (frame_idx, frame_bgr)
    try:
        for frame_idx in indices:
            try:
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                ret, frame = cap.read()
                if not ret or frame is None:
                    total_rejected += 1
                    log_buffer.append([video_path, frame_idx, split, label,
                                       "Rejected", "Bad Frame", 0.0, 0.0, "[]"])
                    continue
                frame_buffer.append((frame_idx, frame))
            except Exception as exc:
                total_rejected += 1
                log_buffer.append([video_path, frame_idx, split, label,
                                   "Rejected", f"Read Error: {exc}", 0.0, 0.0, "[]"])
    finally:
        cap.release()   # disk I/O done — GPU is now free to run without starvation

    # --- PASS 2: inference + dedup + save over the in-memory frame buffer ---
    for frame_idx, frame in frame_buffer:
        # Phase 2 stop condition: balance achieved for this split
        if stop_counts is not None and counts[split] >= stop_counts[split]:
            break

        passed, reason, conf, blur, bbox, cropped = process_frame(frame)

        if not passed:
            total_rejected += 1
            log_buffer.append([video_path, frame_idx, split, label,
                               "Rejected", reason, round(conf, 4), round(blur, 2), bbox])
            continue

        # pHash deduplication using BKTree.is_duplicate and BKTree.add
        try:
            new_hash = imagehash.phash(cropped)
        except Exception as exc:
            total_rejected += 1
            log_buffer.append([video_path, frame_idx, split, label,
                               "Rejected", f"Hash Error: {exc}", round(conf, 4), round(blur, 2), bbox])
            continue

        if hash_bank[identity].is_duplicate(new_hash, PHASH_HAMMING_MIN):
            total_rejected += 1
            log_buffer.append([video_path, frame_idx, split, label,
                               "Rejected", "Duplicate", round(conf, 4), round(blur, 2), bbox])
            continue

        hash_bank[identity].add(new_hash)

        # Save face
        file_index[split][label] += 1
        fname     = f"{label}_{split}_{file_index[split][label]:06d}.jpg"
        save_path = os.path.join(FINAL_DATA_DIR, split, label, fname)
        try:
            cropped.save(save_path, format="JPEG", quality=JPEG_QUALITY)
        except Exception as exc:
            total_rejected += 1
            log_buffer.append([video_path, frame_idx, split, label,
                               "Rejected", f"Save Error: {exc}", round(conf, 4), round(blur, 2), bbox])
            continue

        counts[split] += 1
        log_buffer.append([video_path, frame_idx, split, label,
                           "Accepted", "", round(conf, 4), round(blur, 2), bbox])

    # Bulk flush: one single disk write for the entire video's log rows
    if log_buffer:
        csv_writer.writerows(log_buffer)

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — REAL EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

run_start = time.time()

print("\n" + "=" * 60)
print("  PHASE 1 — Real Face Extraction")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 60 + "\n")

real_videos = sorted([
    p for p in Path(RAW_REALS_DIR).rglob("*")
    if p.suffix.lower() in VIDEO_EXTENSIONS
])

print(f"  Real videos found: {len(real_videos):,}")

for video_path in tqdm(real_videos, desc="Phase 1 — Real", unit="video"):
    process_video(
        video_path      = video_path,
        label           = "real",
        hash_bank       = real_hash_bank,
        counts          = real_counts,
        frames_per_video= REAL_FRAMES_PER_VIDEO,
        stop_counts     = None,   # no stopping in Phase 1
    )

print(f"\n  Phase 1 complete.")
print(f"  Real counts: train={real_counts['train']:,}  "
      f"val={real_counts['val']:,}  test={real_counts['test']:,}")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — FAKE EXTRACTION (DYNAMIC STOP TO MATCH REAL COUNTS)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("  PHASE 2 — Fake Face Extraction (balanced to real counts)")
print("=" * 60 + "\n")

fake_videos = sorted([
    p for p in Path(RAW_FAKES_DIR).rglob("*")
    if p.suffix.lower() in VIDEO_EXTENSIONS
])

print(f"  Fake videos found: {len(fake_videos):,}")
print(f"  Targets: train={real_counts['train']:,}  "
      f"val={real_counts['val']:,}  test={real_counts['test']:,}\n")

for video_path in tqdm(fake_videos, desc="Phase 2 — Fake", unit="video"):
    # Early exit: all splits balanced
    if all(fake_counts[s] >= real_counts[s] for s in SPLITS):
        print("  All splits balanced — stopping Phase 2 early.")
        break

    process_video(
        video_path      = video_path,
        label           = "fake",
        hash_bank       = fake_hash_bank,
        counts          = fake_counts,
        frames_per_video= FAKE_FRAMES_PER_VIDEO,
        stop_counts     = real_counts,   # stop each split when matched
    )

print(f"\n  Phase 2 complete.")
print(f"  Fake counts: train={fake_counts['train']:,}  "
      f"val={fake_counts['val']:,}  test={fake_counts['test']:,}")

# ══════════════════════════════════════════════════════════════════════════════
# CLOSE CSV LOG
# ══════════════════════════════════════════════════════════════════════════════

log_file.close()
print(f"\n  CSV log written -> {CSV_LOG_PATH}")

# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

elapsed     = time.time() - run_start
total_real  = sum(real_counts.values())
total_fake  = sum(fake_counts.values())
total_acc   = total_real + total_fake

print("\n" + "=" * 60)
print("  CELL 2 COMPLETE — Extraction Summary")
print("=" * 60)
print(f"  Elapsed         : {elapsed:.0f} s  ({elapsed/60:.1f} min)")
print()
print(f"  Real counts:")
for s in SPLITS:
    print(f"    {s:<6}: {real_counts[s]:,}")
print(f"  Real total      : {total_real:,}")
print()
print(f"  Fake counts:")
for s in SPLITS:
    print(f"    {s:<6}: {fake_counts[s]:,}")
print(f"  Fake total      : {total_fake:,}")
print()
print(f"  Total accepted  : {total_acc:,}")
print(f"  Total rejected  : {total_rejected:,}")
print()
for s in SPLITS:
    balanced = real_counts[s] == fake_counts[s]
    print(f"  {s:<6}: real={real_counts[s]:,}  fake={fake_counts[s]:,}  "
          f"{'BALANCED' if balanced else 'UNBALANCED'}")
print("=" * 60 + "\n")

I0000 00:00:1776091120.503907   10738 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776091120.571351   10738 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776091135.301363   10738 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Output directories ready under: /home/ec2-user/SageMaker/df40_cdf_stage/final_data

  PHASE 1 — Real Face Extraction
  2026-04-13 14:39:20 UTC

  Real videos found: 590


Phase 1 — Real:   0%|          | 0/590 [00:00<?, ?video/s]

W0000 00:00:1776091188.338820   10738 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1776091188.339787   10738 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:1e.0, compute capability: 7.5


26-04-13 14:39:50 - Directory /home/ec2-user/.deepface created
26-04-13 14:39:50 - Directory /home/ec2-user/.deepface/weights created
26-04-13 14:39:50 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /home/ec2-user/.deepface/weights/retinaface.h5

  0%|          | 0.00/119M [00:00<?, ?B/s]
 34%|███▎      | 39.8M/119M [00:00<00:00, 393MB/s]
100%|██████████| 119M/119M [00:00<00:00, 401MB/s] 
I0000 00:00:1776091193.035211   11315 cuda_dnn.cc:461] Loaded cuDNN version 91002



  Phase 1 complete.
  Real counts: train=7,755  val=1,693  test=1,661

  PHASE 2 — Fake Face Extraction (balanced to real counts)

  Fake videos found: 4,873
  Targets: train=7,755  val=1,693  test=1,661



Phase 2 — Fake:   0%|          | 0/4873 [00:00<?, ?video/s]

  All splits balanced — stopping Phase 2 early.

  Phase 2 complete.
  Fake counts: train=7,755  val=1,693  test=1,661

  CSV log written -> /home/ec2-user/SageMaker/df40_cdf_stage/logs/df40_extraction_log.csv

  CELL 2 COMPLETE — Extraction Summary
  Elapsed         : 10445 s  (174.1 min)

  Real counts:
    train : 7,755
    val   : 1,693
    test  : 1,661
  Real total      : 11,109

  Fake counts:
    train : 7,755
    val   : 1,693
    test  : 1,661
  Fake total      : 11,109

  Total accepted  : 22,218
  Total rejected  : 18,998

  train : real=7,755  fake=7,755  BALANCED
  val   : real=1,693  fake=1,693  BALANCED
  test  : real=1,661  fake=1,661  BALANCED



Cell 3 - Upload - Verify and Clean up.

In [2]:
# ==============================================================================
# CELL 3 — Zip and Upload Final Face Dataset to S3 (ULTIMATE OPTIMIZED)
# ==============================================================================

import os
import shutil
import time
import zipfile
import threading
from datetime import datetime, timezone
from pathlib import Path

import boto3
import sagemaker
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIGURATION
# ==============================================================================

BASE_DIR       = "/home/ec2-user/SageMaker/df40_cdf_stage"
FINAL_DATA_DIR = os.path.join(BASE_DIR, "final_data")
LOGS_DIR       = os.path.join(BASE_DIR, "logs")
CSV_LOG_PATH   = os.path.join(LOGS_DIR, "df40_extraction_log.csv")

REAL_ZIP_NAME  = "df40_cdf_faces_real.zip"
FAKE_ZIP_NAME  = "df40_cdf_faces_fake.zip"

REAL_ZIP_PATH  = os.path.join(BASE_DIR, REAL_ZIP_NAME)
FAKE_ZIP_PATH  = os.path.join(BASE_DIR, FAKE_ZIP_NAME)

S3_BUCKET      = "deepfake-d-100k-dataset-tw26"
S3_KEY_PREFIX  = "datasets/df40_cdf"
S3_REAL_ZIP_KEY= f"{S3_KEY_PREFIX}/{REAL_ZIP_NAME}"
S3_FAKE_ZIP_KEY= f"{S3_KEY_PREFIX}/{FAKE_ZIP_NAME}"
S3_CSV_KEY     = f"{S3_KEY_PREFIX}/df40_cdf_extraction_log.csv"

DOWNLOAD_LOG_PATH = os.path.join(LOGS_DIR, "download_log.txt")
S3_DOWNLOAD_LOG_KEY = f"{S3_KEY_PREFIX}/download_log.txt"

SPLITS = ["train", "val", "test"]
LABELS = ["real", "fake"]

# ==============================================================================
# AWS BOTO3 CREDENTIALS & PROGRESS CALLBACK
# ==============================================================================

# Force SageMaker to explicitly pass your AWS IAM credentials to boto3
try:
    sm_session = sagemaker.Session()
    _s3 = sm_session.boto_session.client("s3")
except Exception:
    _s3 = boto3.Session().client("s3")

class TqdmUploadCallback:
    def __init__(self, total_size, filename):
        self.pbar = tqdm(total=total_size, unit='B', unit_scale=True, desc=f"Uploading {filename}")
        self.lock = threading.Lock()

    def __call__(self, bytes_transferred):
        with self.lock:
            self.pbar.update(bytes_transferred)

def upload_to_s3_with_progress(local_path, s3_key):
    if not os.path.exists(local_path):
        return
    file_size = os.path.getsize(local_path)
    filename = os.path.basename(local_path)
    callback = TqdmUploadCallback(file_size, filename)
    _s3.upload_file(local_path, S3_BUCKET, s3_key, Callback=callback)
    callback.pbar.close()

def verify_s3_upload(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    try:
        s3_size = _s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
    except Exception as exc:
        raise RuntimeError(f"S3 head_object failed for {s3_key}: {exc}")
    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size:,} bytes\n"
            f"  S3    : {s3_size:,} bytes\n"
            f"Local files preserved for inspection."
        )
    print(f"  Verified: {os.path.basename(local_path)}  ({s3_size / (1024**3):.2f} GB)")

# ==============================================================================
# MAIN
# ==============================================================================

run_start = time.time()

print("\n" + "=" * 60)
print("  CELL 3 — Zip and Upload to S3")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 60 + "\n")

# -- [1] Validate and Index Files ----------------------------------------------
print("[1/5] Auditing final_data/ and indexing files...")

if not os.path.isdir(FINAL_DATA_DIR):
    raise RuntimeError(f"final_data/ not found at {FINAL_DATA_DIR}. Run Cell 2 first.")

real_files = list(Path(FINAL_DATA_DIR).rglob("*/real/*.jpg"))
fake_files = list(Path(FINAL_DATA_DIR).rglob("*/fake/*.jpg"))

print(f"\n  {'Split':<8}  {'Label':<8}  {'Count':>8}")
print(f"  {'-'*8}  {'-'*8}  {'-'*8}")

for split in SPLITS:
    for label in LABELS:
        split_dir = os.path.join(FINAL_DATA_DIR, split, label)
        count = len(list(Path(split_dir).glob("*.jpg"))) if os.path.isdir(split_dir) else 0
        print(f"  {split:<8}  {label:<8}  {count:>8,}")

total_files = len(real_files) + len(fake_files)
if total_files == 0:
    raise RuntimeError("final_data/ is empty. Run Cell 2 first.")

# -- [2] Create separated zips -------------------------------------------------
print(f"\n[2/5] Zipping files (ZIP_STORED mode)...")

with zipfile.ZipFile(REAL_ZIP_PATH, 'w', zipfile.ZIP_STORED) as zf_real:
    for file_path in tqdm(real_files, desc="Zipping Reals"):
        arcname = os.path.relpath(file_path, FINAL_DATA_DIR)
        zf_real.write(file_path, arcname)

if not os.path.exists(REAL_ZIP_PATH) or os.path.getsize(REAL_ZIP_PATH) == 0:
    raise RuntimeError(f"Zip creation failed for Reals: {REAL_ZIP_PATH}")

with zipfile.ZipFile(FAKE_ZIP_PATH, 'w', zipfile.ZIP_STORED) as zf_fake:
    for file_path in tqdm(fake_files, desc="Zipping Fakes"):
        arcname = os.path.relpath(file_path, FINAL_DATA_DIR)
        zf_fake.write(file_path, arcname)

if not os.path.exists(FAKE_ZIP_PATH) or os.path.getsize(FAKE_ZIP_PATH) == 0:
    raise RuntimeError(f"Zip creation failed for Fakes: {FAKE_ZIP_PATH}")

# -- [3] Upload to S3 with Progress --------------------------------------------
print(f"\n[3/5] Uploading archives and logs to S3...")

upload_to_s3_with_progress(REAL_ZIP_PATH, S3_REAL_ZIP_KEY)
upload_to_s3_with_progress(FAKE_ZIP_PATH, S3_FAKE_ZIP_KEY)

if os.path.exists(CSV_LOG_PATH):
    upload_to_s3_with_progress(CSV_LOG_PATH, S3_CSV_KEY)
if os.path.exists(DOWNLOAD_LOG_PATH):
    upload_to_s3_with_progress(DOWNLOAD_LOG_PATH, S3_DOWNLOAD_LOG_KEY)

# -- [4] Verify uploads --------------------------------------------------------
print(f"\n[4/5] Verifying uploads (byte-for-byte check)...")

uploads_verified = False

try:
    verify_s3_upload(REAL_ZIP_PATH, S3_REAL_ZIP_KEY)
    verify_s3_upload(FAKE_ZIP_PATH, S3_FAKE_ZIP_KEY)

    if os.path.exists(CSV_LOG_PATH):
        verify_s3_upload(CSV_LOG_PATH, S3_CSV_KEY)
    if os.path.exists(DOWNLOAD_LOG_PATH):
        verify_s3_upload(DOWNLOAD_LOG_PATH, S3_DOWNLOAD_LOG_KEY)
    
    uploads_verified = True
    print("  All uploads successfully verified.")

except Exception as e:
    print(f"\n[VERIFICATION FAILED] {e}")
    print("  Cleanup aborted. Local files preserved.")

# -- [5] Cleanup local files ---------------------------------------------------
print(f"\n[5/5] Checking cleanup status...")

if uploads_verified:
    print("  Safety lock disengaged. Cleaning local files...")
    shutil.rmtree(FINAL_DATA_DIR, ignore_errors=True)
    if os.path.exists(REAL_ZIP_PATH):
        os.remove(REAL_ZIP_PATH)
    if os.path.exists(FAKE_ZIP_PATH):
        os.remove(FAKE_ZIP_PATH)

    print(f"  Removed: {FINAL_DATA_DIR}")
    print(f"  Removed: {REAL_ZIP_PATH}")
    print(f"  Removed: {FAKE_ZIP_PATH}")
    print(f"  Logs preserved on disk: {LOGS_DIR}")
else:
    print("  Skipping cleanup due to verification failure.")

# -- Summary -------------------------------------------------------------------
elapsed = time.time() - run_start
print("\n" + "=" * 60)
print("  CELL 3 COMPLETE")
print("=" * 60)
print(f"  Elapsed    : {elapsed:.0f} s  ({elapsed/60:.1f} min)")
print(f"  Total Files: {total_files:,} face crops")
print(f"  S3 Real Zip: s3://{S3_BUCKET}/{S3_REAL_ZIP_KEY}")
print(f"  S3 Fake Zip: s3://{S3_BUCKET}/{S3_FAKE_ZIP_KEY}")
print("=" * 60 + "\n")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml

  CELL 3 — Zip and Upload to S3
  2026-04-13 18:19:03 UTC

[1/5] Auditing final_data/ and indexing files...

  Split     Label        Count
  --------  --------  --------
  train     real         7,755
  train     fake         7,755
  val       real         1,693
  val       fake         1,693
  test      real         1,661
  test      fake         1,661

[2/5] Zipping files (ZIP_STORED mode)...


Zipping Reals:   0%|          | 0/11109 [00:00<?, ?it/s]

Zipping Fakes:   0%|          | 0/11109 [00:00<?, ?it/s]


[3/5] Uploading archives and logs to S3...


Uploading df40_cdf_faces_real.zip:   0%|          | 0.00/171M [00:00<?, ?B/s]

Uploading df40_cdf_faces_fake.zip:   0%|          | 0.00/172M [00:00<?, ?B/s]

Uploading df40_extraction_log.csv:   0%|          | 0.00/5.76M [00:00<?, ?B/s]

Uploading download_log.txt:   0%|          | 0.00/832 [00:00<?, ?B/s]


[4/5] Verifying uploads (byte-for-byte check)...
  Verified: df40_cdf_faces_real.zip  (0.16 GB)
  Verified: df40_cdf_faces_fake.zip  (0.16 GB)
  Verified: df40_extraction_log.csv  (0.01 GB)
  Verified: download_log.txt  (0.00 GB)
  All uploads successfully verified.

[5/5] Checking cleanup status...
  Safety lock disengaged. Cleaning local files...
  Removed: /home/ec2-user/SageMaker/df40_cdf_stage/final_data
  Removed: /home/ec2-user/SageMaker/df40_cdf_stage/df40_cdf_faces_real.zip
  Removed: /home/ec2-user/SageMaker/df40_cdf_stage/df40_cdf_faces_fake.zip
  Logs preserved on disk: /home/ec2-user/SageMaker/df40_cdf_stage/logs

  CELL 3 COMPLETE
  Elapsed    : 131 s  (2.2 min)
  Total Files: 22,218 face crops
  S3 Real Zip: s3://deepfake-d-100k-dataset-tw26/datasets/df40_cdf/df40_cdf_faces_real.zip
  S3 Fake Zip: s3://deepfake-d-100k-dataset-tw26/datasets/df40_cdf/df40_cdf_faces_fake.zip

